# Watermark Segmentation - Kaggle Training

This notebook trains **YOLO11n-seg** on the watermark dataset (`wm_dataset_out`), then evaluates and tests it on real scraped documents (`images_scraped`). Section 10 (below, after the YOLO sections) trains a second, independent model -- **the alpha network** (per-pixel watermark opacity). Section 11 (after section 10) improves on that model with mixed crisp/downsampled training data, scale augmentation, and both Kaggle GPUs -- see section 11's markdown for why.

### Dataset Layout in Kaggle

The Kaggle dataset is titled **"train and test2"**, mounted at `/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2` (older notebook runs may still see it at the shorter `/kaggle/input/train-and-test2` or `/kaggle/input/train-and-test` -- all three paths are checked below):

```
/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/
  images_scraped/images_scraped/*.jpg          # real test pages (no labels) -- section 8
  wm_dataset_colab/wm_dataset_out/
    annotations/  image_labels.csv, train.json, val.json   (COCO)
    images/{train,val}/wm_NNNN.jpg             # 1700 train / 300 val, all 1024x1024 -- sections 1-9
    labels/{train,val}/wm_NNNN.txt             # YOLO-seg, single class 0
    masks/{train,val}/wm_NNNN_mask.png         # class-encoded 0/1/2/3 masks
  wm_backgrounds_v2/                            # needed only by section 10 (alpha network)
  watermarks_processed/                         # needed only by section 10 (alpha network)
```

- Output directory (writable): `/kaggle/working`
- Sections 1-9 (YOLO segmentation) only need `wm_dataset_colab/wm_dataset_out` and `images_scraped/`.
- Section 10 (the alpha network) locates these same two folders itself (10.1) plus `wm_backgrounds_v2/` and `watermarks_processed/` at the top level of the dataset -- see section 10's markdown for what those two extra folders are needed for.


## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))


## 2. Install Ultralytics

In [ ]:
%pip -q install ultralytics==8.4.142
import ultralytics
ultralytics.checks()


## 3. Locate Dataset in Kaggle Input

Kaggle mounts datasets under `/kaggle/input/`. We locate `wm_dataset_out` and `images_scraped` automatically.

In [ ]:
import pathlib

# Primary path based on Kaggle dataset hierarchy
DATASET_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/wm_dataset_colab/wm_dataset_out'),
    pathlib.Path('/kaggle/input/train-and-test2/wm_dataset_colab/wm_dataset_out'),
    pathlib.Path('/kaggle/input/train-and-test/wm_dataset_colab/wm_dataset_out'),
    pathlib.Path('/kaggle/input/train-and-test/wm_dataset_out'),
]

ROOT = None
for cand in DATASET_CANDIDATES:
    if cand.exists():
        ROOT = cand
        break

if ROOT is None:
    # Recursive search if folder structure differs slightly -- prefer a
    # wm_dataset_out that actually contains images/train (an input could
    # in principle carry more than one folder with this name).
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('wm_dataset_out') if (p / 'images' / 'train').is_dir()]
    if not matches:
        matches = list(pathlib.Path('/kaggle/input').rglob('wm_dataset_out'))
    if matches:
        ROOT = matches[0]

assert ROOT and ROOT.exists(), f'Could not find wm_dataset_out in /kaggle/input. Found: {list(pathlib.Path("/kaggle/input").glob("*"))}'
print('Dataset ROOT found at:', ROOT)


## 4. Write `data.yaml` with ABSOLUTE paths

Ultralytics requires absolute paths to avoid resolving against internal default directories.

In [ ]:
import pathlib, yaml

data = {
    'path':  str(ROOT),          # ABSOLUTE path to dataset root
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': ['watermark'],
}
yaml_path = pathlib.Path('/kaggle/working/data.yaml')
yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
print(yaml_path.read_text())

from ultralytics.data.utils import check_det_dataset
resolved = check_det_dataset(str(yaml_path))
print('resolved train ->', resolved['train'])
print('resolved val   ->', resolved['val'])


## 5. Sanity-check the dataset before spending GPU time

Confirms every image has a matching label, and counts the intentional **negatives** (empty label files = clean pages with no watermark).

In [ ]:
for split in ['train', 'val']:
    imgs = sorted((ROOT/'images'/split).glob('*'))
    labs = sorted((ROOT/'labels'/split).glob('*.txt'))
    stems_i = {p.stem for p in imgs}
    stems_l = {p.stem for p in labs}
    empties = [p for p in labs if p.stat().st_size == 0]
    print(f'{split:5} images={len(imgs):4d} labels={len(labs):4d} '
          f'unmatched={len(stems_i ^ stems_l):3d} negatives={len(empties):3d}')
    assert not (stems_i ^ stems_l), f'{split}: image/label filenames do not match'


## 6. Train

1700 train / 300 val images (1024x1024, 2000 total). On a Kaggle T4 or P100 expect roughly 30-50 minutes.

`patience=30` early-stops if validation stops improving.
On CUDA OOM: drop `batch` to 8, then 4, and only reduce `imgsz` as a last resort.

In [ ]:
from ultralytics import YOLO
import pathlib
import shutil
from IPython.display import FileLink, display

# ==============================================================================
# Resume / Continue Training Configuration
# ==============================================================================
# Set RESUME_FROM_CHECKPOINT to:
#   - None: start fresh from yolo11n-seg.pt (COCO-pretrained)
#   - Path to existing best.pt or checkpoint, e.g. '/kaggle/working/runs/wm_seg/weights/best.pt',
#     an uploaded dataset weight (e.g. '/kaggle/input/.../best.pt'),
#     or a previous checkpoint like '/kaggle/working/best_epoch_30.pt'
CHECKPOINT_PATH = '/kaggle/working/runs/wm_seg/weights/best.pt'  # default check path
RESUME_FROM_CHECKPOINT = CHECKPOINT_PATH if pathlib.Path(CHECKPOINT_PATH).exists() else None

if RESUME_FROM_CHECKPOINT:
    print(f'Continuing training from checkpoint: {RESUME_FROM_CHECKPOINT}')
    model = YOLO(str(RESUME_FROM_CHECKPOINT))
else:
    print('Starting fresh training from yolo11n-seg.pt (COCO-pretrained)')
    model = YOLO('yolo11n-seg.pt')

def on_train_epoch_end_hook(trainer):
    # Every 10 epochs
    current_epoch = trainer.epoch + 1
    if current_epoch % 10 == 0:
        best_path = pathlib.Path(trainer.save_dir) / 'weights' / 'best.pt'
        if best_path.exists():
            out_file = pathlib.Path(f'/kaggle/working/best_epoch_{current_epoch}.pt')
            shutil.copy2(best_path, out_file)
            print(f'\n[Checkpoint] Saved best weights at epoch {current_epoch} to {out_file}')
            display(FileLink(f'best_epoch_{current_epoch}.pt'))

model.add_callback('on_train_epoch_end', on_train_epoch_end_hook)

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=1024,      # thin-ring shield logo needs the resolution
    batch=16,
    patience=30,
    seed=0,
    project='/kaggle/working/runs',
    name='wm_seg',
    exist_ok=True,   # allow updating existing run directory if continuing
    plots=True,
)


### 6b. Variant - freeze the first half of the network

Section 6 trains **every** layer (`freeze` is unset, which is Ultralytics' default). This cell is the alternative: freeze the first half and update only the second half.

`freeze=N` counts **top-level modules, not parameters**. `yolo11n-seg` has 23 top-level modules (0..22 are the backbone + neck, 23 is the segment head). `freeze=11` freezes modules 0..10 and leaves 11..23 trainable.

In [ ]:
from ultralytics import YOLO

FREEZE   = 11                 # freeze modules 0..11, train 12..23
RUN_NAME = 'wm_seg_frozen'     # separate run dir, so section 6's weights survive

# Print exactly what will and will not be trained BEFORE spending GPU time.
_probe = YOLO('yolo11n-seg.pt')
_mods  = _probe.model.model
print(f'Total top-level modules: {len(_mods)}')
print(f'Frozen   (0..{FREEZE-1}): {[type(m).__name__ for m in _mods[:FREEZE]]}')
print(f'Trainable ({FREEZE}..{len(_mods)-1}): {[type(m).__name__ for m in _mods[FREEZE:]]}')
del _probe

model_frozen = YOLO('yolo11n-seg.pt')
results_frozen = model_frozen.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=1024,
    batch=16,
    patience=30,
    seed=0,
    project='/kaggle/working/runs',
    name=RUN_NAME,
    freeze=FREEZE,
    plots=True,
)


## 7. Validate and look at predictions

These mAP numbers are on **held-out synthetic** data.

In [ ]:
metrics = model.val()
print('mask mAP50    :', round(float(metrics.seg.map50), 4))
print('mask mAP50-95 :', round(float(metrics.seg.map), 4))
print('mask precision:', round(float(metrics.seg.mp), 4))
print('mask recall   :', round(float(metrics.seg.mr), 4))


In [ ]:
import glob
from IPython.display import Image as IPyImage, display

for p in sorted(glob.glob('/kaggle/working/runs/wm_seg/val_batch*_pred.jpg'))[:3]:
    print(p)
    display(IPyImage(filename=p, width=900))


## 8. Inference on the REAL test set (`images_scraped`)

This section runs inference on the real documents provided in the dataset input under `images_scraped`.

In [ ]:
import pathlib

# Locate images_scraped
TEST_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/images_scraped/images_scraped'),
    pathlib.Path('/kaggle/input/train-and-test2/images_scraped/images_scraped'),
    pathlib.Path('/kaggle/input/train-and-test/images_scraped/images_scraped'),
    pathlib.Path('/kaggle/input/train-and-test/images_scraped'),
]

test_path = None
for cand in TEST_CANDIDATES:
    if cand.exists() and any(cand.iterdir()):
        test_path = cand
        break

if test_path is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('images_scraped') if p.is_dir()]
    if matches:
        # Prefer the deepest subfolder if nested
        test_path = sorted(matches, key=lambda p: len(p.parts))[-1]

assert test_path and test_path.exists(), f'Could not find images_scraped in /kaggle/input'
print('Test folder found at:', test_path)

EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
test_images = sorted(p for p in test_path.rglob('*') if p.suffix.lower() in EXTS and p.is_file())
print(f'{test_path}  ->  {len(test_images)} images')
assert test_images, f'No images found under {test_path}'


### 8b. Run inference and render overlays

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from ultralytics import YOLO

CONF = 0.25
IMGSZ = 1024          # match training
MASK_COLOR = (255, 40, 40)
MASK_ALPHA = 0.45

best_w = pathlib.Path('/kaggle/working/runs/wm_seg/weights/best.pt')
assert best_w.exists(), f'No trained weights at {best_w} - run the training cell first.'
infer_model = YOLO(str(best_w))

OUT_DIR = pathlib.Path('/kaggle/working/test_overlays'); OUT_DIR.mkdir(exist_ok=True)
rows = []

for i, p in enumerate(test_images, 1):
    img = Image.open(p).convert('RGB')
    W, H = img.size
    r = infer_model.predict(source=str(p), conf=CONF, imgsz=IMGSZ, verbose=False)[0]

    masks = None
    if r.masks is not None and r.masks.data is not None and len(r.masks.data):
        m = r.masks.data.cpu().numpy()
        resized = [(np.asarray(Image.fromarray((s * 255).astype(np.uint8)).resize((W, H), Image.NEAREST)) > 127)
                   for s in m]
        masks = np.stack(resized) if resized else None
    confs = [float(c) for c in r.boxes.conf.cpu().numpy()] if (r.boxes is not None and len(r.boxes)) else []

    arr = np.asarray(img).astype(np.float32)
    if masks is not None:
        union = masks.sum(axis=0) > 0
        arr[union] = (1 - MASK_ALPHA) * arr[union] + MASK_ALPHA * np.array(MASK_COLOR, np.float32)
    overlaid = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    n_inst = 0 if masks is None else int(masks.shape[0])
    cov = 0.0 if masks is None else float((masks.sum(axis=0) > 0).mean())

    panel = Image.new('RGB', (W * 2 + 8, H + 26), (245, 245, 245))
    panel.paste(img, (0, 26)); panel.paste(overlaid, (W + 8, 26))
    d = ImageDraw.Draw(panel)
    d.text((4, 6), f'original - {p.name}', fill=(40, 40, 40))
    d.text((W + 12, 6), f'prediction: {n_inst} inst, {cov*100:.2f}% of page', fill=(180, 30, 30))
    panel.save(OUT_DIR / f'{p.stem}_overlay.png')

    rows.append({'image': p.name, 'instances': n_inst, 'coverage': cov,
                 'conf_max': max(confs) if confs else 0.0})
    if i % 20 == 0 or i == len(test_images):
        print(f'  [{i}/{len(test_images)}]')

n_any = sum(1 for r_ in rows if r_['instances'] > 0)
covs = [r_['coverage'] for r_ in rows]
inst = [r_['instances'] for r_ in rows]
print('\n' + '=' * 64)
print('DESCRIPTIVE ONLY - no ground truth, so these are NOT accuracy metrics')
print('=' * 64)
print(f'  images                    : {len(rows)}')
print(f'  images with >=1 detection : {n_any}  ({100*n_any/len(rows):.1f}%)')
print(f'  instances/image           : mean {np.mean(inst):.1f}  median {np.median(inst):.0f}  max {max(inst)}')
print(f'  page coverage             : mean {100*np.mean(covs):.2f}%  median {100*np.median(covs):.2f}%  max {100*max(covs):.2f}%')


### 8c. Look at the overlays

Red = predicted watermark pixels. Shows the first 12; all of them are saved to `/kaggle/working/test_overlays/`.

In [ ]:
from IPython.display import Image as IPyImage, display

SHOW = 12
for r_ in rows[:SHOW]:
    stem = pathlib.Path(r_['image']).stem
    print(f"{r_['image']}  ->  {r_['instances']} inst, {r_['coverage']*100:.2f}% coverage, "
          f"max conf {r_['conf_max']:.2f}")
    display(IPyImage(filename=str(OUT_DIR / f'{stem}_overlay.png'), width=900))


## 9. Model weights and output artifacts in Kaggle

All outputs (trained weights `best.pt`, plots, logs, and `test_overlays/`) are automatically saved to `/kaggle/working/`.
They will be available in the **Output** tab of your Kaggle notebook session.

In [ ]:
import pathlib

best = pathlib.Path('/kaggle/working/runs/wm_seg/weights/best.pt')
if best.exists():
    print('Trained model weights ready at:', best)
    print('File size:', round(best.stat().st_size / 1e6, 2), 'MB')
else:
    print('best.pt not found yet. Make sure training completed successfully.')


## 10. Alpha network (per-pixel watermark opacity)

This section trains a second, independent model: a small U-Net (`AlphaUNet`) that predicts, for every pixel of a page, how much semi-transparent watermark opacity ("alpha") sits there and what colour the mark's own ink is. Because the compositing physics is known --

    observed = alpha * ink + (1 - alpha) * true_page

-- the true page is recovered by the closed-form inverse `true = (observed - alpha*ink) / (1 - alpha)`, not by classification + fill/subtract like the Document tab's other methods.

Training needs, per image, the CLEAN (unwatermarked) page and the exact per-pixel opacity map -- neither is in the reference dataset directly, so 10.3 below rebuilds them by re-running the dataset generator with the same random seed against `wm_backgrounds_v2/` + `watermarks_processed/` (one run, ~23 minutes on a Kaggle kernel, ~2.2 GB written to `/kaggle/working`) and reading off the intermediate values as it goes.

Cell order: 10.1 locates the four dataset folders this section needs; 10.2 writes the generator and the replay tool to `/kaggle/working`; 10.3 rebuilds the training targets (skipped automatically once already done); 10.4 sanity-checks them; 10.5/10.6 define the model/dataset/loss; 10.7 is the actual training cell; 10.8/10.9 are validation visuals on held-out crops and on real scraped documents; 10.10 packages the final weights for download.


### 10.1 Locate the datasets

Locates the four folders this section needs under `/kaggle/input`: `wm_dataset_colab/wm_dataset_out` (already used by the YOLO sections above), `images_scraped`, and the two extra folders needed only here, `wm_backgrounds_v2` and `watermarks_processed`.


In [ ]:
import pathlib

# --- wm_dataset_out (images/labels/annotations for the YOLO sections; also
# used here to pair replayed alpha/clean targets with the reference JPEGs) ---
WM_DATASET_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/wm_dataset_colab/wm_dataset_out'),
    pathlib.Path('/kaggle/input/train-and-test2/wm_dataset_colab/wm_dataset_out'),
    pathlib.Path('/kaggle/input/train-and-test/wm_dataset_colab/wm_dataset_out'),
]
WM_DATASET = None
for cand in WM_DATASET_CANDIDATES:
    if cand.exists():
        WM_DATASET = cand
        break
if WM_DATASET is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('wm_dataset_out') if (p / 'images' / 'train').is_dir()]
    if matches:
        WM_DATASET = matches[0]
assert WM_DATASET and WM_DATASET.exists(), f'Could not find wm_dataset_out in /kaggle/input. Found: {list(pathlib.Path("/kaggle/input").glob("*"))}'
print('WM_DATASET      ->', WM_DATASET)

# --- images_scraped (real, unlabelled test pages) ---
SCRAPED_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/images_scraped/images_scraped'),
    pathlib.Path('/kaggle/input/train-and-test2/images_scraped/images_scraped'),
    pathlib.Path('/kaggle/input/train-and-test/images_scraped/images_scraped'),
]
SCRAPED_DIR = None
for cand in SCRAPED_CANDIDATES:
    if cand.exists() and any(cand.iterdir()):
        SCRAPED_DIR = cand
        break
if SCRAPED_DIR is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('images_scraped') if p.is_dir()]
    if matches:
        SCRAPED_DIR = sorted(matches, key=lambda p: len(p.parts))[-1]  # prefer the deepest (most specific)
print('SCRAPED_DIR     ->', SCRAPED_DIR)

# --- wm_backgrounds_v2 (2000 clean background images, replay input) ---
BACKGROUNDS_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/wm_backgrounds_v2'),
    pathlib.Path('/kaggle/input/train-and-test2/wm_backgrounds_v2'),
]
BACKGROUNDS_DIR = None
for cand in BACKGROUNDS_CANDIDATES:
    if cand.exists():
        BACKGROUNDS_DIR = cand
        break
if BACKGROUNDS_DIR is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('wm_backgrounds_v2') if p.is_dir()]
    if matches:
        BACKGROUNDS_DIR = matches[0]
print('BACKGROUNDS_DIR ->', BACKGROUNDS_DIR)

# --- watermarks_processed (3 watermark PNGs, replay input) ---
WATERMARKS_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/watermarks_processed'),
    pathlib.Path('/kaggle/input/train-and-test2/watermarks_processed'),
]
WATERMARKS_DIR = None
for cand in WATERMARKS_CANDIDATES:
    if cand.exists():
        WATERMARKS_DIR = cand
        break
if WATERMARKS_DIR is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('watermarks_processed') if p.is_dir()]
    if matches:
        WATERMARKS_DIR = matches[0]
print('WATERMARKS_DIR  ->', WATERMARKS_DIR)

# Where the rebuilt training targets / training run will live.
TARGETS_DIR = pathlib.Path('/kaggle/working/alpha_targets')
ALPHA_RUN_DIR = pathlib.Path('/kaggle/working/runs/alpha_net')


### 10.2 Write the dataset generator and the replay script

Both files are byte-identical to their repo copies (`scripts/kaggle_dataset/generate_dataset.py`, `scripts/kaggle_dataset/replay_alpha_targets.py`). Like `!nvidia-smi` above, these are tools invoked from the shell (10.3 runs the replay script as a subprocess) rather than modules imported into this kernel.


In [ ]:
%%writefile /kaggle/working/generate_dataset.py
"""
generate_dataset.py
Main dataset generation pipeline:
  1. Loads all backgrounds (2000 total) and 3 processed watermarks
  2. Assigns watermark combinations (0/1/2/3 watermarks per image)
  3. Applies extensive augmentations to watermarks
  4. Composites watermarks onto backgrounds
  5. Generates COCO JSON annotations, binary segmentation masks, and CSV labels
  6. Organizes into train/val splits (85/15)

Usage:
    python scripts/generate_dataset.py
    python scripts/generate_dataset.py --verify   # run verification checks
"""

import os
import sys
import json
import random
import math
import argparse
import csv
from datetime import datetime
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance, ImageChops
import cv2


# ─── Configuration ────────────────────────────────────────────────────────────

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
BG_DIR = os.path.join(BASE_DIR, "wm_backgrounds_v2")
WM_DIR = os.path.join(BASE_DIR, "watermarks_processed")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

TARGET_SIZE = (1024, 1024)  # All output images are 1024x1024
SEED = 42
TOTAL_IMAGES = 2000

# Watermark combination distribution
DIST_NO_WM = 0.10    # 10% = 200 images
DIST_1_WM = 0.40     # 40% = 800 images
DIST_2_WM = 0.30     # 30% = 600 images
DIST_3_WM = 0.20     # 20% = 400 images

# Split ratio
TRAIN_RATIO = 0.85    # 85% train, 15% val

# Augmentation ranges
OPACITY_RANGE = (0.10, 0.70)
ROTATION_RANGE = (-45, 45)
SCALE_RANGE = (0.15, 3.0)
BLUR_SIGMA_RANGE = (0, 3.0)
BRIGHTNESS_RANGE = (0.5, 1.5)
CONTRAST_RANGE = (0.5, 1.5)
PERSPECTIVE_SKEW = 5  # degrees
FEATHER_SIGMA_RANGE = (0, 5)
JPEG_QUALITY_RANGE = (50, 95)
NOISE_SIGMA_RANGE = (0, 10)

# Blend mode weights
BLEND_MODES = {
    "normal": 0.50,
    "multiply": 0.20,
    "screen": 0.20,
    "overlay": 0.10,
}

# Watermark names and IDs
WATERMARK_INFO = {
    "AriaTendernet": {"id": 1, "file": "AriaTendernet.png"},
    "ariatender_logo": {"id": 2, "file": "ariatender_logo.png"},
    "ariatender_persian": {"id": 3, "file": "ariatender_persian.png"},
}

CATEGORIES = [
    {"id": 0, "name": "no_watermark", "supercategory": "background"},
    {"id": 1, "name": "AriaTendernet", "supercategory": "watermark"},
    {"id": 2, "name": "ariatender_logo", "supercategory": "watermark"},
    {"id": 3, "name": "ariatender_persian", "supercategory": "watermark"},
]


# ─── Utility Functions ───────────────────────────────────────────────────────

def load_backgrounds() -> List[str]:
    """Load all background image paths."""
    files = sorted([
        os.path.join(BG_DIR, f)
        for f in os.listdir(BG_DIR)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])
    print(f"Found {len(files)} background images")
    return files


def load_watermarks() -> Dict[str, Image.Image]:
    """Load all processed watermark images."""
    wms = {}
    for name, info in WATERMARK_INFO.items():
        path = os.path.join(WM_DIR, info["file"])
        wm = Image.open(path).convert("RGBA")
        wms[name] = wm
        print(f"  Loaded watermark '{name}': {wm.size}")
    return wms


def choose_blend_mode() -> str:
    """Randomly choose a blend mode based on weights."""
    modes = list(BLEND_MODES.keys())
    weights = list(BLEND_MODES.values())
    return random.choices(modes, weights=weights, k=1)[0]


def apply_blend(base_rgb: np.ndarray, wm_rgb: np.ndarray, mode: str) -> np.ndarray:
    """
    Apply blending between base (background under watermark) and watermark RGB.
    Both inputs are float [0, 1]. Returns blended RGB [0, 1].
    """
    if mode == "normal":
        return wm_rgb
    elif mode == "multiply":
        return base_rgb * wm_rgb
    elif mode == "screen":
        return 1.0 - (1.0 - base_rgb) * (1.0 - wm_rgb)
    elif mode == "overlay":
        # Overlay: multiply if base < 0.5, screen if base >= 0.5
        mask = base_rgb < 0.5
        result = np.where(
            mask,
            2.0 * base_rgb * wm_rgb,
            1.0 - 2.0 * (1.0 - base_rgb) * (1.0 - wm_rgb)
        )
        return result
    return wm_rgb


def apply_perspective_warp(img: Image.Image, max_skew_deg: float = 5) -> Image.Image:
    """Apply a slight perspective/affine warp to the image."""
    w, h = img.size
    
    # Random skew amounts in pixels
    max_offset = int(max(w, h) * math.tan(math.radians(max_skew_deg)) * 0.1)
    if max_offset < 1:
        return img
    
    # Define source and destination corners
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = np.float32([
        [random.randint(0, max_offset), random.randint(0, max_offset)],
        [w - random.randint(0, max_offset), random.randint(0, max_offset)],
        [w - random.randint(0, max_offset), h - random.randint(0, max_offset)],
        [random.randint(0, max_offset), h - random.randint(0, max_offset)],
    ])
    
    # Convert to numpy for cv2
    img_np = np.array(img)
    M = cv2.getPerspectiveTransform(src, dst)
    result = cv2.warpPerspective(img_np, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
    
    return Image.fromarray(result)


def apply_color_jitter(img: Image.Image) -> Image.Image:
    """Apply slight color jitter (hue/saturation shift) to watermark."""
    # Only jitter sometimes
    if random.random() > 0.3:
        return img
    
    # Split channels
    r, g, b, a = img.split()
    rgb = Image.merge("RGB", (r, g, b))
    
    # Slight hue shift via HSV
    hsv = np.array(rgb.convert("HSV"), dtype=np.float32)
    hsv[:, :, 0] = (hsv[:, :, 0] + random.uniform(-10, 10)) % 180
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.8, 1.2), 0, 255)
    hsv = hsv.astype(np.uint8)
    
    rgb_shifted = Image.fromarray(hsv, "HSV").convert("RGB")
    return Image.merge("RGBA", (*rgb_shifted.split(), a))


def augment_watermark(
    wm: Image.Image,
    bg_size: Tuple[int, int],
) -> Tuple[Image.Image, dict]:
    """
    Apply all augmentations to a single watermark instance.
    Returns the augmented watermark (RGBA) and a dict of augmentation parameters.
    """
    params = {}
    
    # 1. Scale
    scale = random.uniform(*SCALE_RANGE)
    # Bias toward moderate scales (0.3-1.5) with 70% probability
    if random.random() < 0.7:
        scale = random.uniform(0.3, 1.5)
    params["scale"] = round(scale, 3)
    
    new_w = max(1, int(wm.width * scale))
    new_h = max(1, int(wm.height * scale))
    # Cap to prevent memory issues
    new_w = min(new_w, bg_size[0] * 4)
    new_h = min(new_h, bg_size[1] * 4)
    wm_aug = wm.resize((new_w, new_h), Image.LANCZOS)
    
    # 2. Rotation
    rotation = random.uniform(*ROTATION_RANGE)
    # Bias toward common angles
    if random.random() < 0.4:
        rotation = random.choice([0, 0, 0, -45, 45, -30, 30, -15, 15])
    params["rotation"] = round(rotation, 1)
    wm_aug = wm_aug.rotate(rotation, expand=True, resample=Image.BICUBIC, fillcolor=(0, 0, 0, 0))
    
    # 3. Resolution/blur
    blur_sigma = random.uniform(*BLUR_SIGMA_RANGE)
    if blur_sigma > 0.3:
        wm_aug = wm_aug.filter(ImageFilter.GaussianBlur(radius=blur_sigma))
    params["blur_sigma"] = round(blur_sigma, 2)
    
    # 4. Brightness
    brightness = random.uniform(*BRIGHTNESS_RANGE)
    params["brightness"] = round(brightness, 2)
    r, g, b, a = wm_aug.split()
    rgb = Image.merge("RGB", (r, g, b))
    rgb = ImageEnhance.Brightness(rgb).enhance(brightness)
    wm_aug = Image.merge("RGBA", (*rgb.split(), a))
    
    # 5. Contrast
    contrast = random.uniform(*CONTRAST_RANGE)
    params["contrast"] = round(contrast, 2)
    r, g, b, a = wm_aug.split()
    rgb = Image.merge("RGB", (r, g, b))
    rgb = ImageEnhance.Contrast(rgb).enhance(contrast)
    wm_aug = Image.merge("RGBA", (*rgb.split(), a))
    
    # 6. Color jitter
    wm_aug = apply_color_jitter(wm_aug)
    
    # 7. Opacity
    opacity = random.uniform(*OPACITY_RANGE)
    params["opacity"] = round(opacity, 2)
    # We'll apply opacity during compositing, not here
    
    # 8. Perspective warp (30% of the time)
    if random.random() < 0.3:
        wm_aug = apply_perspective_warp(wm_aug, PERSPECTIVE_SKEW)
        params["perspective_warp"] = True
    else:
        params["perspective_warp"] = False
    
    # 9. Edge feathering (40% of the time)
    feather = random.uniform(*FEATHER_SIGMA_RANGE)
    if feather > 0.5 and random.random() < 0.4:
        r, g, b, a = wm_aug.split()
        a = a.filter(ImageFilter.GaussianBlur(radius=feather))
        wm_aug = Image.merge("RGBA", (r, g, b, a))
        params["edge_feather"] = round(feather, 2)
    else:
        params["edge_feather"] = 0
    
    # 10. Blend mode
    blend_mode = choose_blend_mode()
    params["blend_mode"] = blend_mode
    
    return wm_aug, params


def decide_tiling(bg_size: Tuple[int, int]) -> Tuple[bool, int, int, int, int]:
    """Decide if and how to tile the watermark."""
    # 20% chance of tiling
    if random.random() < 0.2:
        cols = random.randint(2, 5)
        rows = random.randint(2, 5)
        gap_x = random.randint(20, 150)
        gap_y = random.randint(20, 100)
        return True, rows, cols, gap_x, gap_y
    return False, 1, 1, 0, 0


def composite_watermark_on_background(
    bg: Image.Image,
    wm_aug: Image.Image,
    opacity: float,
    blend_mode: str,
    position: Tuple[int, int],
) -> Tuple[Image.Image, Image.Image]:
    """
    Composite a single augmented watermark onto the background.
    Returns (composited_bg, mask_layer).
    
    mask_layer is a single-channel image where non-zero pixels indicate watermark presence.
    """
    bg_w, bg_h = bg.size
    wm_w, wm_h = wm_aug.size
    px, py = position
    
    # Create the mask layer (same size as bg)
    mask_layer = Image.new("L", (bg_w, bg_h), 0)
    
    # Calculate the visible region (intersection of watermark and background)
    # Watermark region in bg coordinates
    x1 = max(0, px)
    y1 = max(0, py)
    x2 = min(bg_w, px + wm_w)
    y2 = min(bg_h, py + wm_h)
    
    if x2 <= x1 or y2 <= y1:
        return bg, mask_layer  # No overlap
    
    # Crop the watermark to the visible region
    wm_x1 = x1 - px
    wm_y1 = y1 - py
    wm_x2 = x2 - px
    wm_y2 = y2 - py
    
    wm_crop = wm_aug.crop((wm_x1, wm_y1, wm_x2, wm_y2))
    
    if wm_crop.size[0] == 0 or wm_crop.size[1] == 0:
        return bg, mask_layer
    
    # Extract channels
    wm_r, wm_g, wm_b, wm_a = wm_crop.split()
    
    # Apply opacity to alpha
    wm_a_np = np.array(wm_a, dtype=np.float32) / 255.0
    wm_a_np = wm_a_np * opacity
    
    # Get the background region
    bg_region = bg.crop((x1, y1, x2, y2)).convert("RGB")
    bg_np = np.array(bg_region, dtype=np.float32) / 255.0
    
    # Get watermark RGB
    wm_rgb_np = np.stack([
        np.array(wm_r, dtype=np.float32) / 255.0,
        np.array(wm_g, dtype=np.float32) / 255.0,
        np.array(wm_b, dtype=np.float32) / 255.0,
    ], axis=-1)
    
    # Apply blend mode
    blended = apply_blend(bg_np, wm_rgb_np, blend_mode)
    
    # Alpha composite: result = bg * (1 - alpha) + blended * alpha
    alpha_3ch = np.stack([wm_a_np] * 3, axis=-1)
    result = bg_np * (1.0 - alpha_3ch) + blended * alpha_3ch
    result = np.clip(result * 255, 0, 255).astype(np.uint8)
    
    # Paste result back
    bg_result = bg.copy()
    bg_result.paste(Image.fromarray(result), (x1, y1))
    
    # Build mask: where alpha > threshold
    mask_region = (wm_a_np > 0.02).astype(np.uint8) * 255
    mask_layer.paste(Image.fromarray(mask_region, "L"), (x1, y1))
    
    return bg_result, mask_layer


def apply_post_processing(img: Image.Image) -> Image.Image:
    """Apply post-composition augmentations to the final image."""
    # Convert to numpy for some ops
    img_np = np.array(img, dtype=np.float32)
    
    # 1. Gaussian noise (60% of the time)
    if random.random() < 0.6:
        noise_sigma = random.uniform(*NOISE_SIGMA_RANGE)
        if noise_sigma > 1:
            noise = np.random.normal(0, noise_sigma, img_np.shape)
            img_np = np.clip(img_np + noise, 0, 255)
    
    # 2. Color temperature shift (30% of the time)
    if random.random() < 0.3:
        shift = random.uniform(-15, 15)
        img_np[:, :, 0] = np.clip(img_np[:, :, 0] + shift, 0, 255)      # R channel
        img_np[:, :, 2] = np.clip(img_np[:, :, 2] - shift * 0.7, 0, 255)  # B channel
    
    img = Image.fromarray(img_np.astype(np.uint8))
    
    # 3. Background quality degradation (resize down then up) - 20% of the time
    if random.random() < 0.2:
        factor = random.uniform(0.4, 0.8)
        small_size = (max(64, int(img.width * factor)), max(64, int(img.height * factor)))
        img = img.resize(small_size, Image.BILINEAR).resize(img.size, Image.BILINEAR)
    
    return img


def save_as_jpeg_with_quality(img: Image.Image, path: str) -> None:
    """Save image as JPEG with random quality to simulate compression artifacts."""
    quality = random.randint(*JPEG_QUALITY_RANGE)
    img.convert("RGB").save(path, "JPEG", quality=quality)


def generate_single_image(
    idx: int,
    bg_path: str,
    watermarks: Dict[str, Image.Image],
    wm_combo: List[str],  # list of watermark names to apply
) -> dict:
    """
    Generate a single dataset image with watermarks and masks.
    Returns annotation metadata dict.
    """
    # Load and resize background to 1024x1024
    bg = Image.open(bg_path).convert("RGB")
    bg = bg.resize(TARGET_SIZE, Image.LANCZOS)
    
    bg_w, bg_h = bg.size
    
    # Determine output filename
    img_filename = f"wm_{idx:04d}.jpg"
    mask_filename = f"wm_{idx:04d}_mask.png"
    
    # Track annotations for this image
    annotations = []
    mask_layers = {}  # category_id -> combined mask
    
    if len(wm_combo) == 0:
        # No watermark — just save the clean image with post-processing
        bg = apply_post_processing(bg)
    else:
        for wm_name in wm_combo:
            wm_src = watermarks[wm_name].copy()
            cat_id = WATERMARK_INFO[wm_name]["id"]
            
            # Decide tiling
            is_tiled, tile_rows, tile_cols, gap_x, gap_y = decide_tiling((bg_w, bg_h))
            
            # Augment the watermark
            wm_aug, aug_params = augment_watermark(wm_src, (bg_w, bg_h))
            opacity = aug_params["opacity"]
            blend_mode = aug_params["blend_mode"]
            
            if is_tiled:
                # Place in a grid pattern
                wm_w, wm_h = wm_aug.size
                total_w = tile_cols * wm_w + (tile_cols - 1) * gap_x
                total_h = tile_rows * wm_h + (tile_rows - 1) * gap_y
                
                # Center the grid on the image
                start_x = (bg_w - total_w) // 2
                start_y = (bg_h - total_h) // 2
                
                combined_mask = Image.new("L", (bg_w, bg_h), 0)
                
                for r in range(tile_rows):
                    for c in range(tile_cols):
                        px = start_x + c * (wm_w + gap_x)
                        py = start_y + r * (wm_h + gap_y)
                        bg, ml = composite_watermark_on_background(
                            bg, wm_aug, opacity, blend_mode, (px, py)
                        )
                        # Merge mask
                        combined_mask = Image.fromarray(
                            np.maximum(np.array(combined_mask), np.array(ml))
                        )
                
                mask_layers[cat_id] = combined_mask
                aug_params["is_tiled"] = True
                aug_params["tile_grid"] = f"{tile_rows}x{tile_cols}"
                
                # Bounding box = extent of all tiles
                mask_np = np.array(combined_mask)
                if mask_np.max() > 0:
                    ys, xs = np.where(mask_np > 0)
                    bbox = [int(xs.min()), int(ys.min()),
                            int(xs.max() - xs.min()), int(ys.max() - ys.min())]
                    area = int(mask_np.sum() / 255)
                    is_partial = (bbox[0] <= 0 or bbox[1] <= 0 or
                                  bbox[0] + bbox[2] >= bg_w - 1 or
                                  bbox[1] + bbox[3] >= bg_h - 1)
                    
                    # Generate polygon from mask (simplified contour)
                    contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    segmentation = []
                    for contour in contours:
                        if len(contour) >= 3:
                            # Simplify contour
                            epsilon = 0.005 * cv2.arcLength(contour, True)
                            approx = cv2.approxPolyDP(contour, epsilon, True)
                            if len(approx) >= 3:
                                seg = approx.flatten().tolist()
                                segmentation.append(seg)
                    
                    annotations.append({
                        "category_id": cat_id,
                        "bbox": bbox,
                        "segmentation": segmentation,
                        "area": area,
                        "iscrowd": 0,
                        "attributes": {
                            "watermark_name": wm_name,
                            "is_partial": is_partial,
                            **aug_params,
                        }
                    })
            else:
                # Single placement
                wm_w, wm_h = wm_aug.size
                
                # Random position (allow partial off-screen)
                margin = 50
                px = random.randint(-wm_w + margin, bg_w - margin)
                py = random.randint(-wm_h + margin, bg_h - margin)
                
                bg, mask_layer = composite_watermark_on_background(
                    bg, wm_aug, opacity, blend_mode, (px, py)
                )
                mask_layers[cat_id] = mask_layer
                aug_params["is_tiled"] = False
                
                # Compute bounding box from mask
                mask_np = np.array(mask_layer)
                if mask_np.max() > 0:
                    ys, xs = np.where(mask_np > 0)
                    bbox = [int(xs.min()), int(ys.min()),
                            int(xs.max() - xs.min()), int(ys.max() - ys.min())]
                    area = int(mask_np.sum() / 255)
                    is_partial = (bbox[0] <= 0 or bbox[1] <= 0 or
                                  bbox[0] + bbox[2] >= bg_w - 1 or
                                  bbox[1] + bbox[3] >= bg_h - 1)
                    
                    # Generate polygon
                    contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    segmentation = []
                    for contour in contours:
                        if len(contour) >= 3:
                            epsilon = 0.005 * cv2.arcLength(contour, True)
                            approx = cv2.approxPolyDP(contour, epsilon, True)
                            if len(approx) >= 3:
                                seg = approx.flatten().tolist()
                                segmentation.append(seg)
                    
                    annotations.append({
                        "category_id": cat_id,
                        "bbox": bbox,
                        "segmentation": segmentation,
                        "area": area,
                        "iscrowd": 0,
                        "attributes": {
                            "watermark_name": wm_name,
                            "is_partial": is_partial,
                            **aug_params,
                        }
                    })
        
        # Post-processing
        bg = apply_post_processing(bg)
    
    # Build combined segmentation mask (class-encoded)
    # 0 = background, 1/2/3 = watermark class
    combined_mask = np.zeros((bg_h, bg_w), dtype=np.uint8)
    for cat_id, ml in mask_layers.items():
        ml_np = np.array(ml)
        combined_mask[ml_np > 0] = cat_id
    
    # Infer background type from filename
    bg_basename = os.path.basename(bg_path)
    
    return {
        "image": bg,
        "mask": combined_mask,
        "img_filename": img_filename,
        "mask_filename": mask_filename,
        "annotations": annotations,
        "bg_filename": bg_basename,
        "wm_combo": wm_combo,
        "has_watermark": len(wm_combo) > 0,
    }


def build_coco_dataset(
    all_results: List[dict],
    split_name: str,
) -> dict:
    """Build a COCO-format annotation dict from results."""
    images_list = []
    annotations_list = []
    ann_id = 1
    
    for r in all_results:
        img_id = int(r["img_filename"].split("_")[1].split(".")[0])
        images_list.append({
            "id": img_id,
            "file_name": r["img_filename"],
            "width": TARGET_SIZE[0],
            "height": TARGET_SIZE[1],
        })
        
        for ann in r["annotations"]:
            ann_entry = {
                "id": ann_id,
                "image_id": img_id,
                "category_id": ann["category_id"],
                "bbox": ann["bbox"],
                "segmentation": ann["segmentation"],
                "area": ann["area"],
                "iscrowd": ann["iscrowd"],
                "attributes": ann.get("attributes", {}),
            }
            annotations_list.append(ann_entry)
            ann_id += 1
    
    return {
        "info": {
            "description": f"Watermark Detection Dataset - {split_name}",
            "version": "1.0",
            "year": 2026,
            "date_created": datetime.now().isoformat(),
        },
        "images": images_list,
        "annotations": annotations_list,
        "categories": CATEGORIES,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--verify", action="store_true", help="Run verification checks only")
    args = parser.parse_args()
    
    if args.verify:
        run_verification()
        return
    
    random.seed(SEED)
    np.random.seed(SEED)
    
    print("=" * 70)
    print("  WATERMARK DATASET GENERATOR")
    print("=" * 70)
    print(f"  Target: {TOTAL_IMAGES} images @ {TARGET_SIZE[0]}x{TARGET_SIZE[1]}")
    print(f"  Split: {TRAIN_RATIO*100:.0f}% train / {(1-TRAIN_RATIO)*100:.0f}% val")
    print()
    
    # ─── Load assets ──────────────────────────────────────────────────────
    print("Loading backgrounds...")
    bg_paths = load_backgrounds()
    
    if len(bg_paths) < TOTAL_IMAGES:
        print(f"WARNING: Only {len(bg_paths)} backgrounds found, need {TOTAL_IMAGES}.")
        print("Will reuse backgrounds with different augmentations.")
        # Extend by cycling
        while len(bg_paths) < TOTAL_IMAGES:
            bg_paths.extend(bg_paths[:TOTAL_IMAGES - len(bg_paths)])
        bg_paths = bg_paths[:TOTAL_IMAGES]
    else:
        bg_paths = bg_paths[:TOTAL_IMAGES]
    
    random.shuffle(bg_paths)
    
    print("\nLoading watermarks...")
    watermarks = load_watermarks()
    wm_names = list(watermarks.keys())
    
    # ─── Assign watermark combinations ────────────────────────────────────
    print("\nAssigning watermark combinations...")
    n_no_wm = int(TOTAL_IMAGES * DIST_NO_WM)
    n_1_wm = int(TOTAL_IMAGES * DIST_1_WM)
    n_2_wm = int(TOTAL_IMAGES * DIST_2_WM)
    n_3_wm = TOTAL_IMAGES - n_no_wm - n_1_wm - n_2_wm
    
    combos = []
    # No watermark
    combos.extend([[] for _ in range(n_no_wm)])
    # 1 watermark
    for _ in range(n_1_wm):
        combos.append([random.choice(wm_names)])
    # 2 watermarks
    for _ in range(n_2_wm):
        combos.append(random.sample(wm_names, 2))
    # 3 watermarks
    combos.extend([list(wm_names) for _ in range(n_3_wm)])
    
    random.shuffle(combos)
    
    combo_counts = Counter([len(c) for c in combos])
    print(f"  No watermark: {combo_counts[0]}")
    print(f"  1 watermark:  {combo_counts[1]}")
    print(f"  2 watermarks: {combo_counts[2]}")
    print(f"  3 watermarks: {combo_counts[3]}")
    
    # ─── Create output directories ────────────────────────────────────────
    for split in ["train", "val"]:
        os.makedirs(os.path.join(OUTPUT_DIR, "images", split), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR, "masks", split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "annotations"), exist_ok=True)
    
    # ─── Split assignments ────────────────────────────────────────────────
    indices = list(range(TOTAL_IMAGES))
    random.shuffle(indices)
    n_train = int(TOTAL_IMAGES * TRAIN_RATIO)
    train_indices = set(indices[:n_train])
    val_indices = set(indices[n_train:])
    
    print(f"\n  Train: {len(train_indices)} images")
    print(f"  Val:   {len(val_indices)} images")
    
    # ─── Generate images ──────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("  GENERATING DATASET")
    print(f"{'='*70}\n")
    
    train_results = []
    val_results = []
    csv_rows = []
    
    for idx in range(TOTAL_IMAGES):
        try:
            result = generate_single_image(
                idx, bg_paths[idx], watermarks, combos[idx]
            )
            
            # Determine split
            split = "train" if idx in train_indices else "val"
            
            # Save image (JPEG with quality augmentation)
            img_path = os.path.join(OUTPUT_DIR, "images", split, result["img_filename"])
            save_as_jpeg_with_quality(result["image"], img_path)
            
            # Save mask (PNG, lossless)
            mask_path = os.path.join(OUTPUT_DIR, "masks", split, result["mask_filename"])
            Image.fromarray(result["mask"]).save(mask_path, "PNG")
            
            # Track for COCO JSON
            if split == "train":
                train_results.append(result)
            else:
                val_results.append(result)
            
            # CSV row
            wm_types = ";".join(result["wm_combo"]) if result["wm_combo"] else ""
            csv_rows.append({
                "image_name": result["img_filename"],
                "split": split,
                "has_watermark": result["has_watermark"],
                "num_watermarks": len(result["wm_combo"]),
                "watermark_types": wm_types,
                "background_source": result["bg_filename"],
            })
            
            if (idx + 1) % 50 == 0:
                print(f"  [{idx+1:4d}/{TOTAL_IMAGES}] Generated {result['img_filename']} "
                      f"({split}, {len(result['wm_combo'])} wm)")
        
        except Exception as e:
            print(f"  ERROR generating image {idx}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # ─── Save COCO JSON annotations ──────────────────────────────────────
    print("\nSaving COCO JSON annotations...")
    
    train_coco = build_coco_dataset(train_results, "train")
    val_coco = build_coco_dataset(val_results, "val")
    
    train_json_path = os.path.join(OUTPUT_DIR, "annotations", "train.json")
    val_json_path = os.path.join(OUTPUT_DIR, "annotations", "val.json")
    
    with open(train_json_path, "w", encoding="utf-8") as f:
        json.dump(train_coco, f, indent=2, ensure_ascii=False)
    print(f"  Saved: {train_json_path} ({len(train_coco['annotations'])} annotations)")
    
    with open(val_json_path, "w", encoding="utf-8") as f:
        json.dump(val_coco, f, indent=2, ensure_ascii=False)
    print(f"  Saved: {val_json_path} ({len(val_coco['annotations'])} annotations)")
    
    # ─── Save CSV labels ─────────────────────────────────────────────────
    csv_path = os.path.join(OUTPUT_DIR, "annotations", "image_labels.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "image_name", "split", "has_watermark", "num_watermarks",
            "watermark_types", "background_source"
        ])
        writer.writeheader()
        writer.writerows(csv_rows)
    print(f"  Saved: {csv_path} ({len(csv_rows)} rows)")
    
    # ─── Summary ─────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("  GENERATION COMPLETE")
    print(f"{'='*70}")
    print(f"  Total images: {len(csv_rows)}")
    print(f"  Train: {len(train_results)}, Val: {len(val_results)}")
    print(f"  Output directory: {OUTPUT_DIR}")
    print()
    
    # Print distribution stats
    wm_count_dist = Counter(r["num_watermarks"] for r in csv_rows)
    print("  Watermark count distribution:")
    for k in sorted(wm_count_dist.keys()):
        print(f"    {k} watermarks: {wm_count_dist[k]}")
    print()


def run_verification():
    """Run verification checks on the generated dataset."""
    print("=" * 70)
    print("  DATASET VERIFICATION")
    print("=" * 70)
    
    errors = []
    
    # Check image counts
    for split in ["train", "val"]:
        img_dir = os.path.join(OUTPUT_DIR, "images", split)
        mask_dir = os.path.join(OUTPUT_DIR, "masks", split)
        
        if not os.path.exists(img_dir):
            errors.append(f"Missing directory: {img_dir}")
            continue
        
        imgs = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
        masks = [f for f in os.listdir(mask_dir) if f.endswith('.png')]
        
        print(f"\n  {split}: {len(imgs)} images, {len(masks)} masks")
        
        # Check mask-image correspondence
        img_bases = {os.path.splitext(f)[0] for f in imgs}
        mask_bases = {f.replace("_mask.png", "") for f in masks}
        
        missing_masks = img_bases - mask_bases
        if missing_masks:
            errors.append(f"{split}: {len(missing_masks)} images missing masks")
        
        # Check mask dimensions
        for mask_file in masks[:10]:  # spot check
            mask = Image.open(os.path.join(mask_dir, mask_file))
            if mask.size != TARGET_SIZE:
                errors.append(f"{split}/{mask_file}: mask size {mask.size} != {TARGET_SIZE}")
    
    # Check COCO JSON
    for split in ["train", "val"]:
        json_path = os.path.join(OUTPUT_DIR, "annotations", f"{split}.json")
        if not os.path.exists(json_path):
            errors.append(f"Missing annotation file: {json_path}")
            continue
        
        with open(json_path, "r") as f:
            coco = json.load(f)
        
        print(f"\n  {split}.json: {len(coco['images'])} images, {len(coco['annotations'])} annotations")
        print(f"  Categories: {[c['name'] for c in coco['categories']]}")
    
    # Check CSV
    csv_path = os.path.join(OUTPUT_DIR, "annotations", "image_labels.csv")
    if os.path.exists(csv_path):
        with open(csv_path, "r") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
        print(f"\n  image_labels.csv: {len(rows)} rows")
        
        wm_dist = Counter(int(r["num_watermarks"]) for r in rows)
        print("  Watermark distribution:")
        for k in sorted(wm_dist.keys()):
            print(f"    {k} watermarks: {wm_dist[k]}")
    
    # Report
    if errors:
        print(f"\n  [FAIL] VERIFICATION FAILED: {len(errors)} errors")
        for e in errors:
            print(f"    - {e}")
    else:
        print(f"\n  [SUCCESS] VERIFICATION PASSED")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/replay_alpha_targets.py
"""Deterministic replay of ``generate_dataset.py`` to recover EXACT
per-pixel alpha-regression targets (``clean``, per-pixel accumulated
watermark opacity ``A``) for the Kaggle "train and test2" dataset, whose
reference JPEGs/masks give no such ground truth on their own (see
``scripts/kaggle_dataset/README.md`` for why).

STANDALONE by design: stdlib + numpy + PIL + cv2 only, no repo imports.

How it works
------------
``generate_dataset.py`` seeds ``random``/``np.random`` with a fixed seed
and then draws every augmentation parameter (background shuffle, watermark
combo assignment, scale/rotation/blur/opacity/blend-mode/perspective/
feather/tiling/JPEG quality/noise/colour-shift/resize) from those two
global generators, in a fixed call order. If we run the *exact same file*
against the *exact same input assets* with the *exact same library
versions*, every draw reproduces bit-for-bit, so we can recover the exact
per-image compositing that produced each reference JPEG -- including the
otherwise-unrecorded per-pixel alpha and the pre-post-processing clean
background -- by monkeypatching four of its functions to *also* carry a
parallel "clean" image and an alpha accumulator ``A`` through the same
sequence of operations, using ONLY parameters that are already fixed by
the point they're read (no extra ``random``/``np.random`` calls are ever
added, removed, or reordered by the patches -- see the exactness checks
below, which are what prove that promise held).

This script imports the generator module fresh via ``importlib`` (so it
never pollutes ``sys.modules`` with a stale copy across repeated runs in
the same interpreter... except it does register under a fixed name --
re-running this script's ``run_replay`` twice in one process reloads the
module each time, which is fine since generator functions are pure
w.r.t. the patches installed just before each run), points its
``BG_DIR``/``WM_DIR``/``OUTPUT_DIR`` globals at the given paths, and
calls its unmodified ``main()``.

Exactness proof, per verified image (every ``--verify-every`` images):
JPEG-encode the replayed composite in memory at the SAME quality
``generate_dataset.py`` drew for it, decode it back, and compare pixel-
for-pixel against the reference JPEG decoded the same way. Only an
environment whose Pillow/libjpeg encoder is bit-identical to whatever
produced the reference dataset will show ``exact: true`` for every
verified image -- see the module docstring's honesty note and this run's
``replay_report.json`` for which local Python actually achieves that.
"""

from __future__ import annotations

import argparse
import csv
import importlib.util
import io
import json
import math
import random
import shutil
import sys
import time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

try:
    import PIL
    _PIL_VERSION = getattr(PIL, "__version__", "unknown")
except Exception:  # pragma: no cover
    _PIL_VERSION = "unknown"


class _StopReplay(BaseException):
    """Raised (as a BaseException, NOT Exception) once ``--limit`` images
    have been produced, so it escapes the generator's own
    ``except Exception: continue`` inside ``main()``'s per-image loop
    instead of being silently swallowed there -- see module docstring.
    Never caught anywhere except around the ``main()`` call in
    ``run_replay``.
    """


# ---------------------------------------------------------------------------
# Generator module loading
# ---------------------------------------------------------------------------

def _load_generator_module(path: Path):
    spec = importlib.util.spec_from_file_location("kaggle_replay_generator", str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    return mod


# ---------------------------------------------------------------------------
# Histogram-based percentile (memory-safe over arbitrarily many images)
# ---------------------------------------------------------------------------

def _percentile_from_hist(hist_counts: np.ndarray, bin_edges: np.ndarray, q: float) -> float:
    total = int(hist_counts.sum())
    if total == 0:
        return 0.0
    target = q / 100.0 * total
    cum = np.cumsum(hist_counts)
    idx = int(np.searchsorted(cum, target, side="left"))
    idx = min(idx, len(bin_edges) - 2)
    return float(bin_edges[idx + 1])


# ---------------------------------------------------------------------------
# Core replay
# ---------------------------------------------------------------------------

def run_replay(args) -> dict:
    out_root = Path(args.out)
    gen_out_dir = out_root / "_replay_generator_out"
    clean_root = out_root / "clean"
    alpha_root = out_root / "alpha"

    if args.overwrite and out_root.exists():
        shutil.rmtree(out_root)

    for split in ("train", "val"):
        (clean_root / split).mkdir(parents=True, exist_ok=True)
        (alpha_root / split).mkdir(parents=True, exist_ok=True)
    gen_out_dir.mkdir(parents=True, exist_ok=True)

    gen_path = Path(args.generator) if args.generator else (Path(__file__).parent / "generate_dataset.py")
    mod = _load_generator_module(gen_path)
    mod.BG_DIR = str(Path(args.backgrounds_dir))
    mod.WM_DIR = str(Path(args.watermarks_dir))
    mod.OUTPUT_DIR = str(gen_out_dir)

    orig_composite = mod.composite_watermark_on_background
    orig_gen_single = mod.generate_single_image

    limit = args.limit
    state = {"clean": None, "A": None}
    captured_results = []  # one dict per successfully-generated image, in idx order
    per_image_records = []  # verification / stats records, one per image, same order

    hist_counts = np.zeros(256, dtype=np.int64)
    bin_edges = np.linspace(0.0, 1.0, 257)
    stats_acc = {"total_pixels": 0, "total_pos_pixels": 0, "total_gt_09": 0, "per_image_max": []}

    # -- patched composite_watermark_on_background ---------------------------
    def patched_composite(bg, wm_aug, opacity, blend_mode, position):
        bg_w, bg_h = bg.size
        wm_w, wm_h = wm_aug.size
        px, py = position
        x1 = max(0, px)
        y1 = max(0, py)
        x2 = min(bg_w, px + wm_w)
        y2 = min(bg_h, py + wm_h)
        if x2 > x1 and y2 > y1:
            wm_x1 = x1 - px
            wm_y1 = y1 - py
            wm_x2 = x2 - px
            wm_y2 = y2 - py
            wm_crop = wm_aug.crop((wm_x1, wm_y1, wm_x2, wm_y2))
            if wm_crop.size[0] > 0 and wm_crop.size[1] > 0:
                wm_a = wm_crop.split()[3]
                # Exactly mirrors generate_dataset.composite_watermark_on_background
                # lines computing wm_a_np -- no RNG involved, deterministic
                # given (wm_aug, opacity, position), which are already fixed
                # by the time this function is called.
                wm_a_np = np.asarray(wm_a, dtype=np.float32) / 255.0 * opacity
                A = state["A"]
                region = A[y1:y2, x1:x2]
                A[y1:y2, x1:x2] = 1.0 - (1.0 - region) * (1.0 - wm_a_np)
        return orig_composite(bg, wm_aug, opacity, blend_mode, position)

    # -- patched apply_post_processing ---------------------------------------
    def patched_post_processing(img):
        # MUST mirror generate_dataset.apply_post_processing LINE BY LINE:
        # same random()/np.random calls, in the same order, with the same
        # short-circuiting. Any deviation here desyncs random.Random's
        # internal state from the reference run and silently corrupts every
        # draw for the rest of the dataset -- the per-image exact-decode
        # checks in this script's report are what prove this mirror is
        # faithful; never trust it without them.
        clean = state["clean"]
        A = state["A"]

        img_np = np.array(img, dtype=np.float32)
        clean_np = np.array(clean, dtype=np.float32)

        # 1. Gaussian noise (60% of the time) -- identical noise array
        # added to both img and clean, then independently clipped (a real
        # source of small clipping-induced nonlinearity right at 0/255,
        # same as JPEG's own inability to represent out-of-range values).
        if random.random() < 0.6:
            noise_sigma = random.uniform(*mod.NOISE_SIGMA_RANGE)
            if noise_sigma > 1:
                noise = np.random.normal(0, noise_sigma, img_np.shape)
                img_np = np.clip(img_np + noise, 0, 255)
                clean_np = np.clip(clean_np + noise, 0, 255)

        # 2. Colour-temperature shift (30% of the time) -- identical shift.
        if random.random() < 0.3:
            shift = random.uniform(-15, 15)
            img_np[:, :, 0] = np.clip(img_np[:, :, 0] + shift, 0, 255)
            img_np[:, :, 2] = np.clip(img_np[:, :, 2] - shift * 0.7, 0, 255)
            clean_np[:, :, 0] = np.clip(clean_np[:, :, 0] + shift, 0, 255)
            clean_np[:, :, 2] = np.clip(clean_np[:, :, 2] - shift * 0.7, 0, 255)

        img_out = Image.fromarray(img_np.astype(np.uint8))
        clean_out = Image.fromarray(clean_np.astype(np.uint8))

        # 3. Down/up resize (20% of the time) -- same factor/filter applied
        # to img, clean AND the alpha accumulator (PIL mode "F", BILINEAR).
        if random.random() < 0.2:
            factor = random.uniform(0.4, 0.8)
            small_size = (max(64, int(img_out.width * factor)), max(64, int(img_out.height * factor)))
            img_out = img_out.resize(small_size, Image.BILINEAR).resize(img_out.size, Image.BILINEAR)
            clean_out = clean_out.resize(small_size, Image.BILINEAR).resize(clean_out.size, Image.BILINEAR)
            a_img = Image.fromarray(np.ascontiguousarray(A, dtype=np.float32), mode="F")
            a_img = a_img.resize(small_size, Image.BILINEAR).resize(mod.TARGET_SIZE, Image.BILINEAR)
            A = np.asarray(a_img, dtype=np.float32)

        state["clean"] = clean_out
        state["A"] = np.clip(A, 0.0, 1.0).astype(np.float32)
        return img_out

    # -- patched generate_single_image ---------------------------------------
    def patched_generate_single_image(idx, bg_path, watermarks, wm_combo):
        if limit is not None and idx >= limit:
            raise _StopReplay()
        clean0 = Image.open(bg_path).convert("RGB").resize(mod.TARGET_SIZE, Image.LANCZOS)
        state["clean"] = clean0
        state["A"] = np.zeros((mod.TARGET_SIZE[1], mod.TARGET_SIZE[0]), dtype=np.float32)
        result = orig_gen_single(idx, bg_path, watermarks, wm_combo)
        captured_results.append({
            "idx": idx,
            "img_filename": result["img_filename"],
            "bg_filename": result["bg_filename"],
            "wm_combo": list(result["wm_combo"]),
            "has_watermark": result["has_watermark"],
            "annotations": result["annotations"],
        })
        return result

    # -- patched save_as_jpeg_with_quality ------------------------------------
    def patched_save_as_jpeg(img, path):
        # Same draw, same position in the RNG sequence as the original --
        # we just redirect what gets written.
        quality = random.randint(*mod.JPEG_QUALITY_RANGE)
        path = Path(path)
        split = path.parent.name
        stem = path.stem

        clean = state["clean"]
        A = np.clip(state["A"], 0.0, 1.0).astype(np.float32)

        clean.save(clean_root / split / f"{stem}.png", "PNG")
        alpha_u8 = np.clip(np.round(A * 255.0), 0, 255).astype(np.uint8)
        Image.fromarray(alpha_u8, mode="L").save(alpha_root / split / f"{stem}.png", "PNG")

        if captured_results:
            captured_results[-1]["split"] = split
            captured_results[-1]["stem"] = stem
            captured_results[-1]["quality"] = int(quality)

        rec = {
            "idx": captured_results[-1]["idx"] if captured_results else None,
            "stem": stem,
            "split": split,
            "quality": int(quality),
            "alpha_max": float(A.max()) if A.size else 0.0,
            "alpha_p50": float(np.percentile(A, 50)),
            "alpha_p99": float(np.percentile(A, 99)),
        }
        pos = A[A > 0]
        rec["alpha_pos_p99"] = float(np.percentile(pos, 99)) if pos.size else 0.0

        stats_acc["per_image_max"].append(rec["alpha_max"])
        stats_acc["total_pixels"] += int(A.size)
        stats_acc["total_pos_pixels"] += int(pos.size)
        stats_acc["total_gt_09"] += int(np.sum(A > 0.9))
        if pos.size:
            h, _ = np.histogram(pos, bins=256, range=(0.0, 1.0))
            hist_counts[:] += h

        n_so_far = len(per_image_records)
        do_verify = (n_so_far % max(1, args.verify_every) == 0)
        if do_verify:
            buf = io.BytesIO()
            img.convert("RGB").save(buf, "JPEG", quality=quality)
            buf.seek(0)
            replay_decoded = np.asarray(Image.open(buf).convert("RGB"), dtype=np.int16)

            ref_path = Path(args.reference_root) / "images" / split / f"{stem}.jpg"
            if ref_path.exists():
                ref_decoded = np.asarray(Image.open(ref_path).convert("RGB"), dtype=np.int16)
                if replay_decoded.shape == ref_decoded.shape:
                    diff = np.abs(replay_decoded - ref_decoded)
                    rec["max_abs_diff"] = int(diff.max())
                    rec["mean_abs_diff"] = float(diff.mean())
                    rec["exact"] = bool(diff.max() == 0)
                else:
                    rec["max_abs_diff"] = None
                    rec["mean_abs_diff"] = None
                    rec["exact"] = False
                    rec["shape_mismatch"] = [list(replay_decoded.shape), list(ref_decoded.shape)]

                pre_jpeg = np.asarray(img.convert("RGB"), dtype=np.float64)
                ref_f = ref_decoded.astype(np.float64)
                if pre_jpeg.shape == ref_f.shape:
                    mse = float(np.mean((pre_jpeg - ref_f) ** 2))
                    rec["psnr_pre_jpeg_vs_ref"] = (float("inf") if mse == 0 else 10.0 * math.log10((255.0 ** 2) / mse))
                rec["verified"] = True
            else:
                rec["verified"] = False
                rec["verify_error"] = f"reference image not found: {ref_path}"
        else:
            rec["verified"] = False

        per_image_records.append(rec)
        return None

    mod.composite_watermark_on_background = patched_composite
    mod.apply_post_processing = patched_post_processing
    mod.generate_single_image = patched_generate_single_image
    mod.save_as_jpeg_with_quality = patched_save_as_jpeg

    old_argv = sys.argv
    sys.argv = [str(gen_path)]
    t0 = time.time()
    try:
        mod.main()
    except _StopReplay:
        pass
    finally:
        sys.argv = old_argv
    elapsed = time.time() - t0

    # ------------------------------------------------------------------
    # Cross-check against reference CSV / COCO attributes. Built from our
    # OWN captured_results rather than gen_out_dir's own annotations/*.json,
    # because when --limit triggers an early _StopReplay, main()'s own
    # post-loop COCO/CSV-writing code (which sits after the per-image for
    # loop) never runs -- captured_results is populated incrementally
    # during the loop instead and works identically with or without
    # --limit.
    # ------------------------------------------------------------------
    csv_mismatches = []
    attr_mismatches = []
    ATTR_KEYS = ("scale", "rotation", "opacity", "blend_mode", "is_tiled")

    ref_csv_path = Path(args.reference_root) / "annotations" / "image_labels.csv"
    ref_rows = {}
    if ref_csv_path.exists():
        with open(ref_csv_path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                ref_rows[row["image_name"]] = row
    else:
        csv_mismatches.append({"reason": f"reference CSV not found: {ref_csv_path}"})

    ref_coco = {}
    for split in ("train", "val"):
        p = Path(args.reference_root) / "annotations" / f"{split}.json"
        if not p.exists():
            attr_mismatches.append({"reason": f"reference COCO file not found: {p}"})
            continue
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        by_img = {img["id"]: [] for img in data["images"]}
        for ann in data["annotations"]:
            by_img.setdefault(ann["image_id"], []).append(ann)
        for anns in by_img.values():
            anns.sort(key=lambda a: a["id"])
        ref_coco[split] = by_img

    for r in captured_results:
        fname = r["img_filename"]
        split = r.get("split")

        ref_row = ref_rows.get(fname)
        if ref_row is None:
            csv_mismatches.append({"image": fname, "reason": "missing in reference CSV"})
        elif ref_row["background_source"] != r["bg_filename"]:
            csv_mismatches.append({
                "image": fname, "reason": "background_source mismatch",
                "reference": ref_row["background_source"], "replay": r["bg_filename"],
            })

        idx = r["idx"]
        ref_anns = ref_coco.get(split, {}).get(idx) if split else None
        if ref_anns is None:
            if split is not None:
                attr_mismatches.append({"image": fname, "reason": f"image id {idx} not found in reference {split}.json"})
            continue
        my_anns = r["annotations"]
        if len(ref_anns) != len(my_anns):
            attr_mismatches.append({
                "image": fname, "reason": "annotation count mismatch",
                "reference_count": len(ref_anns), "replay_count": len(my_anns),
            })
            continue
        for j, (ra, ma) in enumerate(zip(ref_anns, my_anns)):
            rattrs = ra.get("attributes", {})
            mattrs = ma.get("attributes", {})
            for k in ATTR_KEYS:
                rv, mv = rattrs.get(k), mattrs.get(k)
                if rv != mv:
                    attr_mismatches.append({
                        "image": fname, "ann_index": j, "key": k,
                        "reference": rv, "replay": mv,
                    })

    # ------------------------------------------------------------------
    # Alpha distribution stats
    # ------------------------------------------------------------------
    per_image_max = np.asarray(stats_acc["per_image_max"], dtype=np.float64)
    alpha_stats = {
        "per_image_max_p50": float(np.percentile(per_image_max, 50)) if per_image_max.size else 0.0,
        "per_image_max_p99": float(np.percentile(per_image_max, 99)) if per_image_max.size else 0.0,
        "per_image_max_true_max": float(per_image_max.max()) if per_image_max.size else 0.0,
        "positive_pixel_p99": _percentile_from_hist(hist_counts, bin_edges, 99.0),
        "frac_pixels_gt_0.9": (stats_acc["total_gt_09"] / stats_acc["total_pixels"]) if stats_acc["total_pixels"] else 0.0,
        "frac_pixels_positive": (stats_acc["total_pos_pixels"] / stats_acc["total_pixels"]) if stats_acc["total_pixels"] else 0.0,
    }

    # ------------------------------------------------------------------
    # Disk usage
    # ------------------------------------------------------------------
    def _dir_bytes(p: Path) -> int:
        return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

    n_total = len(captured_results)
    disk_bytes = _dir_bytes(clean_root) + _dir_bytes(alpha_root)

    verified = [r for r in per_image_records if r.get("verified")]
    n_exact = sum(1 for r in verified if r.get("exact"))
    max_abs_diffs = [r["max_abs_diff"] for r in verified if r.get("max_abs_diff") is not None]
    mean_abs_diffs = [r["mean_abs_diff"] for r in verified if r.get("mean_abs_diff") is not None]

    n_train = sum(1 for r in captured_results if r.get("split") == "train")
    n_val = sum(1 for r in captured_results if r.get("split") == "val")

    report = {
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "python_version": sys.version,
        "pillow_version": _PIL_VERSION,
        "numpy_version": np.__version__,
        "cv2_version": cv2.__version__,
        "generator_path": str(gen_path),
        "backgrounds_dir": str(args.backgrounds_dir),
        "watermarks_dir": str(args.watermarks_dir),
        "reference_root": str(args.reference_root),
        "limit": limit,
        "verify_every": args.verify_every,
        "elapsed_seconds": elapsed,
        "seconds_per_image": (elapsed / n_total) if n_total else None,
        "disk_bytes_total": disk_bytes,
        "disk_bytes_per_image": (disk_bytes / n_total) if n_total else None,
        "counts": {"total_processed": n_total, "train": n_train, "val": n_val},
        "exact_match": {
            "n_verified": len(verified),
            "n_exact": n_exact,
            "rate": (n_exact / len(verified)) if verified else None,
            "max_abs_diff_overall": max(max_abs_diffs) if max_abs_diffs else None,
            "mean_abs_diff_mean": (sum(mean_abs_diffs) / len(mean_abs_diffs)) if mean_abs_diffs else None,
        },
        "csv_mismatches": {"count": len(csv_mismatches), "examples": csv_mismatches[:10]},
        "attr_mismatches": {"count": len(attr_mismatches), "examples": attr_mismatches[:10]},
        "alpha_stats": alpha_stats,
        "images": per_image_records,
    }

    out_root.mkdir(parents=True, exist_ok=True)
    with open(out_root / "replay_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    _print_summary(report)

    ok = (report["exact_match"]["rate"] in (None, 1.0)) and not csv_mismatches and not attr_mismatches
    report["_ok"] = ok
    return report


def _print_summary(report: dict) -> None:
    print("=" * 70)
    print("  REPLAY REPORT SUMMARY")
    print("=" * 70)
    print(f"  Python: {report['python_version'].splitlines()[0]}")
    print(f"  Pillow {report['pillow_version']}  numpy {report['numpy_version']}  cv2 {report['cv2_version']}")
    c = report["counts"]
    print(f"  Processed: {c['total_processed']} (train={c['train']}, val={c['val']})")
    em = report["exact_match"]
    rate = f"{em['rate']*100:.2f}%" if em["rate"] is not None else "n/a"
    print(f"  Exact-decode match: {em['n_exact']}/{em['n_verified']} ({rate})")
    print(f"  max_abs_diff (over verified): {em['max_abs_diff_overall']}")
    print(f"  mean_abs_diff (avg over verified): {em['mean_abs_diff_mean']}")
    print(f"  CSV mismatches: {report['csv_mismatches']['count']}")
    print(f"  COCO attribute mismatches: {report['attr_mismatches']['count']}")
    a = report["alpha_stats"]
    print(f"  Alpha (per-image max): p50={a['per_image_max_p50']:.4f} p99={a['per_image_max_p99']:.4f} "
          f"max={a['per_image_max_true_max']:.4f}")
    print(f"  Alpha (positive pixels p99): {a['positive_pixel_p99']:.4f}  "
          f"frac>0.9: {a['frac_pixels_gt_0.9']:.6f}")
    print(f"  Elapsed: {report['elapsed_seconds']:.1f}s "
          f"({report['seconds_per_image']:.3f}s/image)" if report['seconds_per_image'] else "")
    if report["disk_bytes_per_image"]:
        print(f"  Disk: {report['disk_bytes_total']/1e6:.1f} MB total, "
              f"{report['disk_bytes_per_image']/1e3:.1f} KB/image")
    print("=" * 70)


def build_arg_parser() -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(
        description="Deterministically replay generate_dataset.py to recover exact alpha/clean targets."
    )
    ap.add_argument("--generator", default=None, help="Path to generate_dataset.py (default: sibling file)")
    ap.add_argument("--backgrounds-dir", required=True)
    ap.add_argument("--watermarks-dir", required=True)
    ap.add_argument("--reference-root", required=True,
                     help="A wm_dataset_out dir (contains images/, annotations/) to verify replay against")
    ap.add_argument("--out", required=True, help="Output targets root (clean/, alpha/, replay_report.json)")
    ap.add_argument("--limit", type=int, default=None, help="Stop after the first N indices")
    ap.add_argument("--verify-every", type=int, default=1, help="Verify every Kth processed image (default 1)")
    ap.add_argument("--overwrite", action="store_true", help="Delete --out first")
    return ap


def main(argv=None):
    args = build_arg_parser().parse_args(argv)
    report = run_replay(args)
    sys.exit(0 if report.get("_ok") else 1)


if __name__ == "__main__":
    main()


### 10.3 Rebuild the training targets (skips if already done)


In [ ]:
import os, sys, subprocess, json, time, pathlib

# Set e.g. 50 for a quick test; None replays all 2000 images (~23 min on a
# Kaggle T4 kernel, ~2.2 GB written under /kaggle/working).
REPLAY_LIMIT = None

def replay_ok(report):
    em = report.get('exact_match', {})
    rate = em.get('rate')
    max_diff = em.get('max_abs_diff_overall')
    csv_mm = report.get('csv_mismatches', {}).get('count', 0)
    attr_mm = report.get('attr_mismatches', {}).get('count', 0)
    return (rate is not None and rate >= 0.99
            and max_diff is not None and max_diff <= 3
            and csv_mm == 0 and attr_mm == 0)

def replay_complete(report):
    counts = report.get('counts', {})
    return report.get('limit') is None and counts.get('total_processed', 0) == 2000

report_path = TARGETS_DIR / 'replay_report.json'
need_replay = True
if report_path.exists():
    report = json.loads(report_path.read_text())
    if replay_complete(report) and replay_ok(report):
        print(f'Found a complete, verified replay report at {report_path} -- skipping replay.')
        need_replay = False
    else:
        print(f'Found an incomplete or non-exact replay report at {report_path} -- re-running.')

if need_replay:
    assert BACKGROUNDS_DIR is not None and WATERMARKS_DIR is not None, (
        'Need wm_backgrounds_v2/ and watermarks_processed/ under /kaggle/input to run the replay -- '
        'see section 10 markdown for what to upload.'
    )

    # Exact replay depends on the image-library build: locally, Pillow 10.4.0
    # + opencv 4.10.0.84 reproduced the reference JPEGs pixel-for-pixel.
    # Kaggle's kernel ships other versions, so install the matching ones into
    # an ISOLATED directory and put it first on the replay subprocess's
    # PYTHONPATH only -- the notebook kernel (torch, ultralytics, training)
    # keeps its own packages. --no-deps keeps Kaggle's numpy; --upgrade avoids
    # "target directory already exists" warnings on a re-run.
    REPLAY_ENV = pathlib.Path('/tmp/replay_env')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--upgrade', '--target', str(REPLAY_ENV),
         'pillow==10.4.0', 'opencv-python-headless==4.10.0.84'],
        check=True,
    )
    replay_env = dict(os.environ)
    replay_env['PYTHONPATH'] = str(REPLAY_ENV) + os.pathsep + replay_env.get('PYTHONPATH', '')

    cmd = [
        sys.executable, '/kaggle/working/replay_alpha_targets.py',
        '--generator', '/kaggle/working/generate_dataset.py',
        '--backgrounds-dir', str(BACKGROUNDS_DIR),
        '--watermarks-dir', str(WATERMARKS_DIR),
        '--reference-root', str(WM_DATASET),
        '--out', str(TARGETS_DIR),
        '--overwrite',
    ]
    if REPLAY_LIMIT is not None:
        cmd += ['--limit', str(REPLAY_LIMIT)]

    print('Running:', ' '.join(cmd))
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=replay_env)
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
    print(f'\nReplay subprocess exited {ret} after {time.time() - t0:.1f}s')

    # The replay script exits 1 whenever it isn't 100.00% pixel-exact (see its
    # own module docstring) -- that alone is not necessarily a failure for us
    # (see replay_ok's tolerance below), so the report on disk, not the exit
    # code, is what decides whether to proceed.
    if not report_path.exists():
        raise RuntimeError(
            f'The replay exited with code {ret} without writing {report_path} -- it crashed before '
            'finishing (see the log above, e.g. a missing background or watermark file).'
        )
    report = json.loads(report_path.read_text())

em = report['exact_match']
print('\n=== replay report summary ===')
print(f"processed: {report['counts']['total_processed']} (train={report['counts']['train']}, val={report['counts']['val']})")
print(f"exact-decode match: {em['n_exact']}/{em['n_verified']} (rate={em['rate']})")
print(f"max_abs_diff_overall: {em['max_abs_diff_overall']}")
print(f"CSV mismatches: {report['csv_mismatches']['count']}, COCO attribute mismatches: {report['attr_mismatches']['count']}")
print('alpha_stats:', report['alpha_stats'])

if not replay_ok(report):
    raise RuntimeError(
        'Replay did not reproduce the reference dataset closely enough to trust as training supervision -- '
        'this looks like a real desync (mismatched generator/library parameters, or large pixel differences) '
        'rather than ordinary JPEG-decode rounding. Check the replay log above and '
        'scripts/kaggle_dataset/README.md.'
    )

if report.get('limit') is not None:
    print(f"\nNOTE: this replay report has limit={report['limit']} -- targets are PARTIAL "
          f"({report['counts']['total_processed']} of 2000 images). Training below will only use those images.")


### 10.4 Check the targets


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

pos_sample = None  # (split, stem) of the first sample found with a real mark

for split in ['train', 'val']:
    img_dir = WM_DATASET / 'images' / split
    alpha_dir = TARGETS_DIR / 'alpha' / split
    clean_dir = TARGETS_DIR / 'clean' / split
    img_stems = {p.stem for p in img_dir.glob('*.jpg')}
    alpha_stems = {p.stem for p in alpha_dir.glob('*.png')} if alpha_dir.is_dir() else set()
    clean_stems = {p.stem for p in clean_dir.glob('*.png')} if clean_dir.is_dir() else set()
    complete = img_stems & alpha_stems & clean_stems
    print(f'{split:5} images={len(img_stems):4d} alpha={len(alpha_stems):4d} clean={len(clean_stems):4d} '
          f'complete={len(complete):4d}')
    assert complete, f'{split}: no stem has both a replayed alpha and clean target -- run 10.3 first.'
    if pos_sample is None:
        for stem in sorted(complete):
            a = cv2.imread(str(alpha_dir / f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
            if a is not None and a.max() / 255.0 >= 0.1:
                pos_sample = (split, stem)
                break

assert pos_sample is not None, 'No positive (visibly watermarked) sample found across train/val to sanity-check.'
split, stem = pos_sample

obs_path = next((WM_DATASET / 'images' / split).glob(f'{stem}.*'))
obs = cv2.cvtColor(cv2.imread(str(obs_path)), cv2.COLOR_BGR2RGB).astype(np.float32)
clean = cv2.cvtColor(cv2.imread(str(TARGETS_DIR / 'clean' / split / f'{stem}.png')), cv2.COLOR_BGR2RGB).astype(np.float32)
alpha = cv2.imread(str(TARGETS_DIR / 'alpha' / split / f'{stem}.png'), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

diff = np.abs(obs - clean).mean(axis=-1)
inside = alpha >= 0.08
outside = alpha < 1e-6
print(f'\nSample {split}/{stem}: alpha max = {alpha.max():.3f}')
print(f'  mean |obs-clean| outside mark (JPEG floor): {diff[outside].mean():.3f}')
print(f'  mean |obs-clean| inside mark (signal)      : {diff[inside].mean():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(np.clip(obs / 255.0, 0, 1)); axes[0].set_title('observed')
axes[1].imshow(np.clip(clean / 255.0, 0, 1)); axes[1].set_title('clean (replayed)')
axes[2].imshow(alpha, cmap='inferno', vmin=0, vmax=1); axes[2].set_title('alpha (replayed)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/alpha_targets_preview.png', dpi=120)
plt.show()


### 10.5 Model

`AlphaUNet` inline, copied from `watermark_remover/alpha_net.py` (see that module's docstring for the full physics/architecture writeup and the calibration behind `ALPHA_MAX = 0.9`). Kept identical (module names, parameter shapes, forward math) so a checkpoint trained here loads there via `alpha_net.load_model`, and vice versa.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

ALPHA_MAX = 0.9                    # calibrated for this dataset -- see watermark_remover/alpha_net.py docstring
WIDTHS = (32, 64, 128, 256)
FORMAT_VERSION = 1                 # bump together with watermark_remover/alpha_net.py if the checkpoint format changes
ALPHA_BIAS_INIT = -4.0             # sigmoid(-4) ~= 0.018 -> alpha_max*sigmoid(-4) ~= 0.016 at init (near-identity)

class ConvBlock(nn.Module):
    """Two x (Conv3x3 -> GroupNorm(8) -> SiLU), same spatial size in/out."""

    def __init__(self, in_ch, out_ch, dilation=1):
        super().__init__()
        padding = dilation
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=padding, dilation=dilation)
        self.gn1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=padding, dilation=dilation)
        self.gn2 = nn.GroupNorm(8, out_ch)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        x = self.act(self.gn1(self.conv1(x)))
        x = self.act(self.gn2(self.conv2(x)))
        return x

class AlphaUNet(nn.Module):
    """Small U-Net predicting per-pixel watermark opacity (alpha) and ink
    colour from an RGB crop (N,3,H,W in [0,1], H/W multiples of 16). Encoder
    of len(widths) ConvBlocks with MaxPool2d(2) between levels; a dilated
    bottleneck at widths[-1]; a decoder mirroring the encoder (bilinear
    upsample -> concat skip -> ConvBlock); a 1x1 head -> 4 channels (alpha via
    alpha_max*sigmoid, ink RGB via sigmoid). Must stay identical (attribute
    names, parameter shapes, forward math) to watermark_remover/alpha_net.py's
    AlphaUNet -- state_dicts are interchangeable between the two."""

    def __init__(self, widths=WIDTHS, alpha_max=ALPHA_MAX):
        super().__init__()
        widths = tuple(widths)
        assert len(widths) >= 2, f'AlphaUNet needs at least 2 widths, got {widths!r}'
        self.widths = widths
        self.alpha_max = float(alpha_max)
        self.pool = nn.MaxPool2d(2)

        enc_in = (3,) + widths[:-1]
        self.encoders = nn.ModuleList([ConvBlock(c_in, c_out) for c_in, c_out in zip(enc_in, widths)])
        self.bottleneck = ConvBlock(widths[-1], widths[-1], dilation=2)

        dec_in_ch = [widths[i + 1] + widths[i] for i in range(len(widths) - 1)]
        dec_out_ch = list(widths[:-1])
        self.decoders = nn.ModuleList([ConvBlock(c_in, c_out) for c_in, c_out in zip(dec_in_ch, dec_out_ch)])

        self.head = nn.Conv2d(widths[0], 4, kernel_size=1)
        with torch.no_grad():
            self.head.bias.zero_()
            self.head.bias[0] = ALPHA_BIAS_INIT

    def forward(self, x):
        skips = []
        h = x
        n = len(self.encoders)
        for i, enc in enumerate(self.encoders):
            h = enc(h)
            skips.append(h)
            if i < n - 1:
                h = self.pool(h)
        h = self.bottleneck(h)
        for dec, skip in zip(self.decoders[::-1], skips[-2::-1]):
            h = F.interpolate(h, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            h = torch.cat([h, skip], dim=1)
            h = dec(h)
        out = self.head(h)
        alpha = self.alpha_max * torch.sigmoid(out[:, 0:1])
        ink = torch.sigmoid(out[:, 1:4])
        return alpha, ink

def recover(obs, alpha, ink, alpha_max):
    """Closed-form inverse of obs = alpha*ink + (1-alpha)*true, solved for
    true. Denominator floored at 1 - alpha_max so a worst-case prediction can
    never blow up the division beyond what alpha_max itself bounds."""
    denom = torch.clamp(1.0 - alpha, min=1.0 - alpha_max)
    return torch.clamp((obs - alpha * ink) / denom, 0.0, 1.0)

def save_checkpoint(path, model, epoch, val_metrics, config):
    """Writes exactly the repo's alpha_net.checkpoint_dict format -- only
    tensors/plain Python primitives, so both torch.load(weights_only=True)
    and watermark_remover.alpha_net.load_model can read the result. Optimizer
    /scheduler/scaler state (needed only to resume, not to just USE the
    weights) is saved separately, in last.pt."""
    ckpt = {
        'format_version': FORMAT_VERSION,
        'arch': 'AlphaUNet',
        'widths': list(model.widths),
        'alpha_max': float(model.alpha_max),
        'model_state': model.state_dict(),
        'epoch': epoch,
        'val_metrics': val_metrics,
        'train_config': config,
    }
    torch.save(ckpt, path)

_probe_model = AlphaUNet()
_n_params = sum(p.numel() for p in _probe_model.parameters())
print(f'AlphaUNet: widths={WIDTHS} alpha_max={ALPHA_MAX} params={_n_params:,}')
del _probe_model


### 10.6 Dataset and loss


In [ ]:
import cv2
import numpy as np
import torch

IMG_EXTS = ('.jpg', '.jpeg', '.png')

def dilate_bool_mask(mask_bool, ksize):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    return cv2.dilate(mask_bool.astype(np.uint8), kernel).astype(np.float32)

class AlphaCropDataset(torch.utils.data.Dataset):
    """Random (train) or deterministic (val) crops pairing
    WM_DATASET/images/<split> with TARGETS_DIR/{alpha,clean}/<split> by
    filename stem. Returns per item: obs (3,crop,crop), clean (3,crop,crop),
    alpha (1,crop,crop, clamped to ALPHA_MAX), w_alpha (1,crop,crop), w_rec
    (1,crop,crop) -- float32 tensors, obs/clean in [0, 1].

    w_alpha and w_rec both start from weight = 1 + 4*dilated(alpha > 0.01)
    (mark pixels, and a halo around them, count more), then diverge:
    w_alpha also requires either alpha<=0.01 (true background, still
    supervised) or the pixel is actually VISIBLE after gain jitter (a pixel
    with alpha>0 that is not visibly different from clean, e.g. "multiply"
    onto a near-white patch, carries no recoverable alpha signal). w_rec
    excludes alpha > ALPHA_MAX entirely -- recover's denominator is floored
    at 1 - ALPHA_MAX, so those pixels are architecturally unrecoverable and
    penalising the network for that ceiling would teach the wrong thing.
    """

    def __init__(self, split, crop, samples_per_epoch=None, deterministic=False, seed=0):
        self.split = split
        self.crop = int(crop)
        self.samples_per_epoch = samples_per_epoch
        self.deterministic = deterministic
        self.seed = seed

        img_dir = WM_DATASET / 'images' / split
        alpha_dir = TARGETS_DIR / 'alpha' / split
        clean_dir = TARGETS_DIR / 'clean' / split
        imgs = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS}
        alphas = {p.stem: p for p in alpha_dir.glob('*.png')} if alpha_dir.is_dir() else {}
        cleans = {p.stem: p for p in clean_dir.glob('*.png')} if clean_dir.is_dir() else {}
        stems = sorted(set(imgs) & set(alphas) & set(cleans))
        n_skipped = len(set(imgs) - set(stems))
        print(f'[AlphaCropDataset:{split}] {len(stems)} usable stem(s), {n_skipped} skipped (no replayed target)')
        assert stems, f'No stems with both a replayed alpha and clean target for split={split!r} -- run 10.3 first.'
        self.items = [(imgs[s], alphas[s], cleans[s]) for s in stems]

    def __len__(self):
        if self.samples_per_epoch is not None:
            return int(self.samples_per_epoch)
        if self.deterministic:
            return 2 * len(self.items)
        return len(self.items)

    def _read_triple(self, i):
        img_p, alpha_p, clean_p = self.items[i]
        obs = cv2.cvtColor(cv2.imread(str(img_p), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        clean = cv2.cvtColor(cv2.imread(str(clean_p), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        alpha = cv2.imread(str(alpha_p), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        return obs, clean, alpha

    def __getitem__(self, idx):
        n = len(self.items)
        if self.deterministic:
            # Fixed function of (seed, idx) -- independent of worker count or call order.
            rng = np.random.default_rng([int(self.seed), int(idx)])
            img_idx = (idx // 2) % n
            prefer_wm = (idx % 2 == 0)
        else:
            rng = np.random.default_rng()
            img_idx = int(rng.integers(0, n))
            prefer_wm = rng.random() < 0.7

        obs, clean, alpha = self._read_triple(img_idx)
        crop = self.crop
        h, w = alpha.shape[:2]
        pad_h, pad_w = max(0, crop - h), max(0, crop - w)
        if pad_h or pad_w:
            obs = np.pad(obs, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
            clean = np.pad(clean, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
            alpha = np.pad(alpha, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0.0)

        h, w = alpha.shape[:2]
        max_y0, max_x0 = max(0, h - crop), max(0, w - crop)
        has_wm_px = bool(np.any(alpha > 0.05))
        if prefer_wm and has_wm_px:
            ys, xs = np.nonzero(alpha > 0.05)
            i = int(rng.integers(0, len(ys)))
            y0 = int(np.clip(ys[i] - crop // 2, 0, max_y0))
            x0 = int(np.clip(xs[i] - crop // 2, 0, max_x0))
        else:
            y0 = int(rng.integers(0, max_y0 + 1))
            x0 = int(rng.integers(0, max_x0 + 1))

        obs_c = obs[y0:y0 + crop, x0:x0 + crop, :].astype(np.float32).copy()
        clean_c = clean[y0:y0 + crop, x0:x0 + crop, :].astype(np.float32).copy()
        alpha_c = alpha[y0:y0 + crop, x0:x0 + crop].astype(np.float32).copy()

        gain = rng.uniform(0.85, 1.0, size=(1, 1, 3)).astype(np.float32)
        obs_c = obs_c * gain
        clean_c = clean_c * gain

        weight_c = 1.0 + 4.0 * dilate_bool_mask(alpha_c > 0.01, 7)
        diff_mean = np.abs(obs_c - clean_c).mean(axis=2)
        visible = dilate_bool_mask(diff_mean > (4.0 / 255.0), 3)
        w_alpha_c = weight_c * np.maximum((alpha_c <= 0.01).astype(np.float32), visible)
        w_rec_c = weight_c * (alpha_c <= ALPHA_MAX).astype(np.float32)
        alpha_c = np.clip(alpha_c, 0.0, ALPHA_MAX)

        obs_t = torch.from_numpy(np.ascontiguousarray(obs_c.transpose(2, 0, 1)))
        clean_t = torch.from_numpy(np.ascontiguousarray(clean_c.transpose(2, 0, 1)))
        alpha_t = torch.from_numpy(np.ascontiguousarray(alpha_c[None, :, :]))
        w_alpha_t = torch.from_numpy(np.ascontiguousarray(w_alpha_c[None, :, :]))
        w_rec_t = torch.from_numpy(np.ascontiguousarray(w_rec_c[None, :, :]))
        return obs_t, clean_t, alpha_t, w_alpha_t, w_rec_t


def alpha_loss(pred_alpha, pred_ink, obs, clean, alpha, w_alpha, w_rec):
    """5*L_alpha + 1*L_rec + 0.5*L_comp -- alpha target is already clamped to
    ALPHA_MAX by the dataset. Returns (total, {'L_alpha':..., 'L_rec':...,
    'L_comp':...}) with the parts already .item()'d."""
    rec = recover(obs, pred_alpha, pred_ink, ALPHA_MAX)
    l_alpha = (w_alpha * (pred_alpha - alpha).abs()).mean()
    l_rec = (w_rec * (rec - clean).abs().mean(dim=1, keepdim=True)).mean()
    comp = pred_alpha * pred_ink + (1.0 - pred_alpha) * clean
    l_comp = (w_rec * (comp - obs).abs().mean(dim=1, keepdim=True)).mean()
    total = 5.0 * l_alpha + 1.0 * l_rec + 0.5 * l_comp
    parts = {'L_alpha': float(l_alpha.item()), 'L_rec': float(l_rec.item()), 'L_comp': float(l_comp.item())}
    return total, parts


def evaluate(model, loader, device):
    """Pixel-accurate validation metrics accumulated over the whole loader,
    split into watermark (alpha_gt > 0.01) and background regions; pixels
    with alpha_gt > ALPHA_MAX are excluded from every metric (architecturally
    unrecoverable -- frac_alpha_gt_max reports how much of the set that was).
    val_score is lower-is-better and penalises recover damaging clean pixels.
    """
    model.eval()
    sums = dict(alpha_err_wm=0.0, n_wm=0, alpha_bg=0.0, n_bg=0,
                rec_err_wm=0.0, id_err_wm=0.0, rec_err_bg=0.0, id_err_bg=0.0,
                n_excluded=0, n_total=0)
    with torch.no_grad():
        for obs, clean, alpha, _w_alpha, _w_rec in loader:
            obs, clean, alpha = obs.to(device), clean.to(device), alpha.to(device)
            pred_alpha, pred_ink = model(obs)
            rec = recover(obs, pred_alpha, pred_ink, ALPHA_MAX)

            recoverable = alpha <= ALPHA_MAX
            wm_mask = (alpha > 0.01) & recoverable
            bg_mask = (~(alpha > 0.01)) & recoverable

            alpha_err = (pred_alpha - alpha).abs()
            rec_err = (rec - clean).abs().mean(dim=1, keepdim=True) * 255.0
            id_err = (obs - clean).abs().mean(dim=1, keepdim=True) * 255.0

            n_wm, n_bg = int(wm_mask.sum().item()), int(bg_mask.sum().item())
            if n_wm:
                sums['alpha_err_wm'] += float(alpha_err[wm_mask].sum().item())
                sums['rec_err_wm'] += float(rec_err[wm_mask].sum().item())
                sums['id_err_wm'] += float(id_err[wm_mask].sum().item())
                sums['n_wm'] += n_wm
            if n_bg:
                sums['alpha_bg'] += float(pred_alpha[bg_mask].sum().item())
                sums['rec_err_bg'] += float(rec_err[bg_mask].sum().item())
                sums['id_err_bg'] += float(id_err[bg_mask].sum().item())
                sums['n_bg'] += n_bg
            sums['n_excluded'] += int((~recoverable).sum().item())
            sums['n_total'] += int(alpha.numel())

    n_wm, n_bg = max(sums['n_wm'], 1), max(sums['n_bg'], 1)
    alpha_mae_wm = sums['alpha_err_wm'] / n_wm
    alpha_mean_bg = sums['alpha_bg'] / n_bg
    rec_mae_wm_255 = sums['rec_err_wm'] / n_wm
    identity_mae_wm_255 = sums['id_err_wm'] / n_wm
    rec_mae_bg_255 = sums['rec_err_bg'] / n_bg
    identity_mae_bg_255 = sums['id_err_bg'] / n_bg
    frac_alpha_gt_max = sums['n_excluded'] / max(sums['n_total'], 1)
    improvement_wm = (1.0 - rec_mae_wm_255 / identity_mae_wm_255) if identity_mae_wm_255 > 1e-8 else 0.0
    val_score = rec_mae_wm_255 + 2.0 * max(0.0, rec_mae_bg_255 - identity_mae_bg_255)

    return {
        'alpha_mae_wm': alpha_mae_wm, 'alpha_mean_bg': alpha_mean_bg,
        'rec_mae_wm_255': rec_mae_wm_255, 'identity_mae_wm_255': identity_mae_wm_255,
        'improvement_wm': improvement_wm, 'rec_mae_bg_255': rec_mae_bg_255,
        'identity_mae_bg_255': identity_mae_bg_255, 'frac_alpha_gt_max': frac_alpha_gt_max,
        'val_score': val_score,
    }


PROBE_CROP = 512       # only for this cell's sanity-check batch -- 10.7 rebuilds the datasets with its own constants
PROBE_SAMPLES = 8
train_ds = AlphaCropDataset('train', PROBE_CROP, samples_per_epoch=PROBE_SAMPLES, deterministic=False, seed=0)
val_ds = AlphaCropDataset('val', PROBE_CROP, samples_per_epoch=None, deterministic=True, seed=0)
print(f'train_ds: {len(train_ds)} samples/epoch, {len(train_ds.items)} source image(s)')
print(f'val_ds:   {len(val_ds)} crops, {len(val_ds.items)} source image(s)')

probe_loader = torch.utils.data.DataLoader(train_ds, batch_size=2, shuffle=False, num_workers=0)
obs_b, clean_b, alpha_b, w_alpha_b, w_rec_b = next(iter(probe_loader))
print('batch shapes:', obs_b.shape, clean_b.shape, alpha_b.shape, w_alpha_b.shape, w_rec_b.shape)


### 10.7 Train

This is the actual training cell. Expected wall-clock time is **unknown until measured** -- read the first epoch's `time=...s` in the printed log and scale by `EPOCHS` to estimate the whole run. A partial run is still useful: `best.pt`/`last.pt` are updated every epoch, and re-running this cell resumes from `last.pt` where it left off.

`ALPHA_MAX = 0.9` (10.5) is calibrated for THIS dataset's per-mark opacity range stacked up to 3 layers -- see `watermark_remover/alpha_net.py`'s module docstring for the measured alpha distribution and why the ceiling matters (recovery gets noisier as predicted alpha approaches it).

If you hit a CUDA out-of-memory error: first drop `BATCH_SIZE` from 8 to 4, then (if still OOM) drop `CROP` from 512 to 384.


In [ ]:
import csv, math, time, shutil, pathlib
import torch
from IPython.display import FileLink, display

EPOCHS = 40
BATCH_SIZE = 8
CROP = 512
SAMPLES_PER_EPOCH = 3000
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
PATIENCE = 12                 # early-stop after this many epochs with no val_score improvement
SEED = 0
SNAPSHOT_EVERY = 10           # copy best.pt -> alpha_net_best_epoch_N.pt every N epochs

torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

ALPHA_RUN_DIR.mkdir(parents=True, exist_ok=True)
LAST_PATH = ALPHA_RUN_DIR / 'last.pt'
BEST_PATH = ALPHA_RUN_DIR / 'best.pt'
METRICS_PATH = ALPHA_RUN_DIR / 'metrics.csv'

train_ds = AlphaCropDataset('train', CROP, samples_per_epoch=SAMPLES_PER_EPOCH, deterministic=False, seed=SEED)
val_ds = AlphaCropDataset('val', CROP, samples_per_epoch=None, deterministic=True, seed=SEED)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(device == 'cuda'), drop_last=True, persistent_workers=(NUM_WORKERS > 0),
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device == 'cuda'), persistent_workers=(NUM_WORKERS > 0),
)
iters_per_epoch = len(train_loader)
assert iters_per_epoch > 0, 'Training loader produced zero batches -- reduce BATCH_SIZE or raise SAMPLES_PER_EPOCH.'

model = AlphaUNet(widths=WIDTHS, alpha_max=ALPHA_MAX).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

warmup_iters = iters_per_epoch  # 1-epoch linear warmup
total_iters = max(1, EPOCHS * iters_per_epoch)

def lr_lambda(it):
    if it < warmup_iters:
        return (it + 1) / max(1, warmup_iters)
    progress = min(max((it - warmup_iters) / max(1, total_iters - warmup_iters), 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
use_amp = (device == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

start_epoch = 1
best_score = float('inf')
epochs_no_improve = 0

if LAST_PATH.exists():
    print('Continuing training from checkpoint:', LAST_PATH)
    # weights_only=False: last.pt is written exclusively by this same cell and
    # carries optimizer/scheduler/scaler state (not just tensors/primitives),
    # so it cannot use the stricter weights_only=True loading best.pt uses.
    ckpt = torch.load(LAST_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    opt.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    best_score = ckpt.get('best_score', float('inf'))
    epochs_no_improve = ckpt.get('epochs_no_improve', 0)
    start_epoch = int(ckpt['epoch']) + 1
else:
    print('Starting fresh training')

TRAIN_CONFIG = dict(
    epochs=EPOCHS, batch_size=BATCH_SIZE, crop=CROP, samples_per_epoch=SAMPLES_PER_EPOCH,
    lr=LR, weight_decay=WEIGHT_DECAY, num_workers=NUM_WORKERS, patience=PATIENCE, seed=SEED,
    widths=list(WIDTHS), alpha_max=ALPHA_MAX,
)

best_epoch = None
best_metrics = None

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t_epoch0 = time.time()
    part_sums = {'total': 0.0, 'L_alpha': 0.0, 'L_rec': 0.0, 'L_comp': 0.0}
    n_batches = 0

    for obs, clean, alpha, w_alpha, w_rec in train_loader:
        obs = obs.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)
        alpha = alpha.to(device, non_blocking=True)
        w_alpha = w_alpha.to(device, non_blocking=True)
        w_rec = w_rec.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        try:
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                pred_alpha, pred_ink = model(obs)
                loss, parts = alpha_loss(pred_alpha, pred_ink, obs, clean, alpha, w_alpha, w_rec)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                raise RuntimeError(
                    f'CUDA out of memory during a training step (batch_size={BATCH_SIZE}, crop={CROP}). '
                    'Try a smaller BATCH_SIZE (e.g. 4) and/or CROP (e.g. 384).'
                ) from e
            raise

        scheduler.step()
        part_sums['total'] += float(loss.item())
        for k in ('L_alpha', 'L_rec', 'L_comp'):
            part_sums[k] += parts[k]
        n_batches += 1

    n_batches = max(1, n_batches)
    train_parts = {k: v / n_batches for k, v in part_sums.items()}
    val_metrics = evaluate(model, val_loader, device)
    epoch_seconds = time.time() - t_epoch0
    lr_now = scheduler.get_last_lr()[0]

    print(
        f'epoch {epoch}/{EPOCHS} train_loss={train_parts["total"]:.4f} '
        f'(alpha={train_parts["L_alpha"]:.4f} rec={train_parts["L_rec"]:.4f} comp={train_parts["L_comp"]:.4f}) '
        f'val_score={val_metrics["val_score"]:.4f} alpha_mae_wm={val_metrics["alpha_mae_wm"]:.4f} '
        f'rec_mae_wm_255={val_metrics["rec_mae_wm_255"]:.2f} lr={lr_now:.2e} time={epoch_seconds:.1f}s'
    )

    is_new_file = not METRICS_PATH.exists()
    fieldnames = (['epoch', 'train_loss', 'train_L_alpha', 'train_L_rec', 'train_L_comp', 'lr', 'seconds']
                  + list(val_metrics.keys()))
    with open(METRICS_PATH, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if is_new_file:
            w.writeheader()
        row = {'epoch': epoch, 'train_loss': train_parts['total'], 'train_L_alpha': train_parts['L_alpha'],
               'train_L_rec': train_parts['L_rec'], 'train_L_comp': train_parts['L_comp'],
               'lr': lr_now, 'seconds': epoch_seconds}
        row.update(val_metrics)
        w.writerow(row)

    improved = val_metrics['val_score'] < best_score
    if improved:
        best_score = val_metrics['val_score']
        epochs_no_improve = 0
        save_checkpoint(BEST_PATH, model, epoch, val_metrics, TRAIN_CONFIG)
        best_epoch, best_metrics = epoch, val_metrics
    else:
        epochs_no_improve += 1

    last_ckpt = {
        'format_version': FORMAT_VERSION, 'arch': 'AlphaUNet', 'widths': list(model.widths),
        'alpha_max': float(model.alpha_max), 'model_state': model.state_dict(),
        'epoch': epoch, 'val_metrics': val_metrics, 'train_config': TRAIN_CONFIG,
        'optimizer': opt.state_dict(), 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
        'best_score': best_score, 'epochs_no_improve': epochs_no_improve,
    }
    torch.save(last_ckpt, LAST_PATH)

    if epoch % SNAPSHOT_EVERY == 0 and BEST_PATH.exists():
        snap_path = pathlib.Path(f'/kaggle/working/alpha_net_best_epoch_{epoch}.pt')
        shutil.copy2(BEST_PATH, snap_path)
        print(f'\n[Checkpoint] Saved best weights at epoch {epoch} to {snap_path}')
        display(FileLink(snap_path.name))

    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch} ({epochs_no_improve} epochs without improvement)')
        break

print('\n=== training summary ===')
print('best epoch:', best_epoch)
print('best val metrics:', best_metrics)


### 10.8 Validation examples


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

BEST_PATH = ALPHA_RUN_DIR / 'best.pt'
assert BEST_PATH.exists(), f'No trained weights at {BEST_PATH} -- run the training cell (10.7) first.'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(BEST_PATH, map_location=device, weights_only=True)
val_model = AlphaUNet(widths=tuple(ckpt['widths']), alpha_max=float(ckpt['alpha_max'])).to(device)
val_model.load_state_dict(ckpt['model_state'])
val_model.eval()

val_ds = AlphaCropDataset('val', CROP, samples_per_epoch=None, deterministic=True, seed=SEED)

N_SHOW = 4
fig, axes = plt.subplots(N_SHOW, 4, figsize=(16, 4 * N_SHOW))
for row in range(N_SHOW):
    idx = row % len(val_ds)
    obs, clean, alpha_gt, _w_alpha, _w_rec = val_ds[idx]
    with torch.no_grad():
        pred_alpha, pred_ink = val_model(obs.unsqueeze(0).to(device))
        rec = recover(obs.unsqueeze(0).to(device), pred_alpha, pred_ink, float(ckpt['alpha_max']))

    obs_img = obs.permute(1, 2, 0).numpy()
    clean_img = clean.permute(1, 2, 0).numpy()
    alpha_img = pred_alpha[0, 0].cpu().numpy()
    rec_img = rec[0].permute(1, 2, 0).cpu().numpy()

    axes[row, 0].imshow(np.clip(obs_img, 0, 1)); axes[row, 0].set_title('observed')
    axes[row, 1].imshow(alpha_img, cmap='inferno', vmin=0, vmax=float(ckpt['alpha_max'])); axes[row, 1].set_title('predicted alpha')
    axes[row, 2].imshow(np.clip(rec_img, 0, 1)); axes[row, 2].set_title('recovered')
    axes[row, 3].imshow(np.clip(clean_img, 0, 1)); axes[row, 3].set_title('clean')
    for c in range(4):
        axes[row, c].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/alpha_net_val_examples.png', dpi=120)
plt.show()

print('best epoch:', ckpt['epoch'])
print('best val metrics:', ckpt['val_metrics'])


### 10.9 Real documents (images_scraped)

These panels show whether the synthetic training transfers to real scanned/scraped pages -- there is no ground truth here, so this is descriptive only, not an accuracy measurement. A genuinely clean real page should get alpha close to 0 everywhere; if it doesn't, that is a sign of synthetic-to-real gap, not necessarily a code bug.


In [ ]:
import pathlib, time
import numpy as np
import torch
from PIL import Image

def feather_weight(tile, overlap):
    overlap = max(1, min(overlap, tile // 2))
    ramp = np.linspace(1.0 / (overlap + 1), 1.0, overlap, dtype=np.float32)
    w = np.ones(tile, dtype=np.float32)
    w[:overlap] = ramp
    w[-overlap:] = ramp[::-1]
    return np.outer(w, w)

def predict_page(model, img_rgb_uint8, device, tile=512, overlap=64):
    """Tiled full-page inference: reflect-pads to at least one tile and a
    multiple of 16, runs the model over overlapping tiles, blends alpha/ink
    with a feathered weight so tile seams don't show, then calls `recover`
    ONCE on the merged maps (not per-tile) -- same behaviour as
    watermark_remover/alpha_net.py's predict_image."""
    alpha_max = model.alpha_max
    h, w = img_rgb_uint8.shape[:2]
    pad_h = max(tile, int(np.ceil(h / 16.0) * 16))
    pad_w = max(tile, int(np.ceil(w / 16.0) * 16))
    img_f = img_rgb_uint8.astype(np.float32) / 255.0
    img_padded = np.pad(img_f, ((0, pad_h - h), (0, pad_w - w), (0, 0)), mode='reflect')

    alpha_acc = np.zeros((pad_h, pad_w), dtype=np.float32)
    ink_acc = np.zeros((pad_h, pad_w, 3), dtype=np.float32)
    weight_acc = np.zeros((pad_h, pad_w), dtype=np.float32)

    stride = max(1, tile - overlap)
    ys = list(range(0, max(1, pad_h - tile + 1), stride))
    if ys[-1] != pad_h - tile:
        ys.append(pad_h - tile)
    xs = list(range(0, max(1, pad_w - tile + 1), stride))
    if xs[-1] != pad_w - tile:
        xs.append(pad_w - tile)
    ramp2d = feather_weight(tile, overlap)

    with torch.no_grad():
        for y in ys:
            for x in xs:
                crop = img_padded[y:y + tile, x:x + tile, :]
                t = torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0).to(device)
                a, ink = model(t)
                a_np = a[0, 0].float().cpu().numpy()
                ink_np = ink[0].float().permute(1, 2, 0).cpu().numpy()
                alpha_acc[y:y + tile, x:x + tile] += a_np * ramp2d
                ink_acc[y:y + tile, x:x + tile, :] += ink_np * ramp2d[..., None]
                weight_acc[y:y + tile, x:x + tile] += ramp2d

    weight_safe = np.clip(weight_acc, 1e-6, None)
    alpha_full = (alpha_acc / weight_safe)[:h, :w]
    ink_full = (ink_acc / weight_safe[..., None])[:h, :w, :]

    obs_t = torch.from_numpy(img_f).permute(2, 0, 1).unsqueeze(0)
    alpha_t = torch.from_numpy(alpha_full).unsqueeze(0).unsqueeze(0)
    ink_t = torch.from_numpy(ink_full).permute(2, 0, 1).unsqueeze(0)
    recovered_t = recover(obs_t, alpha_t, ink_t, alpha_max)
    recovered = np.clip(np.round(recovered_t[0].permute(1, 2, 0).numpy() * 255.0), 0, 255).astype(np.uint8)
    return alpha_full.astype(np.float32), ink_full.astype(np.float32), recovered

assert SCRAPED_DIR is not None, 'Could not find images_scraped -- see 10.1.'
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
real_images = sorted(p for p in SCRAPED_DIR.rglob('*') if p.suffix.lower() in EXTS and p.is_file())
assert real_images, f'No images found under {SCRAPED_DIR}'

OUT_DIR = pathlib.Path('/kaggle/working/alpha_net_real')
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(ALPHA_RUN_DIR / 'best.pt', map_location=device, weights_only=True)
real_model = AlphaUNet(widths=tuple(ckpt['widths']), alpha_max=float(ckpt['alpha_max'])).to(device)
real_model.load_state_dict(ckpt['model_state'])
real_model.eval()

panel_paths = []
for p in real_images[:6]:
    img = np.array(Image.open(p).convert('RGB'))
    t0 = time.time()
    alpha, ink, recovered = predict_page(real_model, img, device)
    dt = time.time() - t0

    alpha_vis = (np.clip(alpha / max(float(ckpt['alpha_max']), 1e-6), 0, 1) * 255).astype(np.uint8)
    alpha_vis = np.stack([alpha_vis] * 3, axis=-1)
    panel = np.concatenate([img, alpha_vis, recovered], axis=1)
    panel_path = OUT_DIR / f'{p.stem}_alpha_panel.png'
    Image.fromarray(panel).save(panel_path)
    panel_paths.append(panel_path)

    frac_marked = float((alpha > 0.05).mean()) * 100
    print(f'{p.name}: alpha_max={alpha.max():.3f} marked={frac_marked:.2f}% time={dt:.2f}s')

from IPython.display import Image as IPyImage, display
for path in panel_paths[:3]:
    display(IPyImage(filename=str(path), width=1200))


### 10.10 Download the final model

Copies the best checkpoint to a single, stably-named file and shows its epoch/metrics/size. To wire this into the app, place the downloaded file at **`weights/alpha_net_best_final.pt`** in the repo -- that enables Method 4's **"Alpha Network (per box)"** strategy (`watermark_remover/doc_detect.py`).


In [ ]:
import pathlib, shutil
import torch
from IPython.display import FileLink, display

best_src = ALPHA_RUN_DIR / 'best.pt'
final_path = pathlib.Path('/kaggle/working/alpha_net_best_final.pt')

if best_src.exists():
    shutil.copy2(best_src, final_path)
    ckpt = torch.load(final_path, map_location='cpu', weights_only=True)
    size_mb = final_path.stat().st_size / 1e6
    print('epoch:', ckpt['epoch'])
    print('val metrics:', ckpt['val_metrics'])
    print(f'size: {size_mb:.2f} MB')
    display(FileLink(final_path.name))
else:
    print(f'No best.pt found at {best_src} -- run the training cell (10.7) first.')


## 11. Alpha network v2 (improved training)

**Why**: the section-10 model (v1, best checkpoint = epoch 9/40, early-stopped) under-performs on real screenshots even though it was diagnosed as never having the wrong watermark colour/opacity:

- It removes only ~62% of the mark even on its own synthetic validation data (predicted alpha ~0.26 vs. true ~0.43) -- alpha is systematically under-predicted, not mis-coloured.
- It is fragile to page scale/sharpness: a crisp 788x713 real screenshot removed only 18% of its mark at native size, but 60% once resized to 1024 -- the model has implicitly learned "1024px downsampled page", not "watermarked page at any resolution".
- Every v1 training image was a 1024x1024 **downsampled** page, with noise added to 60% of them and JPEG re-encoding on 100% -- real inputs (screenshots) are crisp, noise-free, and shown at their native resolution, a distribution the model never saw.
- v1 stopped improving (early-stopped) well before its scheduled 40 epochs, on a single-domain dataset -- there was no second, harder validation signal to push it further or to select a checkpoint that generalises off that one domain.

**What changed here**:
- **Mixed data**: every training batch draws from two sources -- the existing Kaggle `wm_dataset_out` pages (10.6, downsampled/noisy/JPEG, with scale augmentation added -- 11.2) and new **crisp, on-the-fly composites** built directly from native-resolution `wm_backgrounds_v2` pages and the real, calibrated AriaTender mark opacity (11.1) -- so the model sees both regimes every epoch, not just the downsampled one.
- **Scale augmentation**: both data sources now randomly rescale the page (0.6x-1.6x) before cropping, so the model can no longer just memorise "watermark at 1024px scale".
- **Longer training with EMA**: up to 60 epochs (vs. v1's 40, and v1 only reached epoch 9 before stalling) with an exponential moving average of the weights, which is what actually gets saved/evaluated -- smooths out the noisy, higher-variance signal that comes from mixing two data regimes per batch.
- **Checkpoint selection on crisp validation**: the epoch picked as `best.pt` is chosen by validation score on the crisp, native-resolution held-out set (11.1's deterministic mode) -- the domain closest to real screenshots -- while the Kaggle-page validation score is still computed and logged every epoch for comparison.
- **Both GPUs**: this notebook's Kaggle GPU setting trains on both T4s via `torch.nn.DataParallel` (11.3 explains why DP rather than DDP here).

**What to upload**: a `stamps/` folder at the top level of the Kaggle dataset, containing `ariatender_stacked.png` and `ariatender_wide.png` -- two RGBA PNGs where the alpha channel is the real, measured AriaTender opacity (not a 0/1 mask). If `stamps/` is not found, 11.1 falls back to the 3 `watermarks_processed` marks alone for the crisp composites (their alpha is a 0/1 coverage mask, so a plausible opacity is sampled instead of measured) and prints a warning -- crisp composites still work, just without the calibrated real-opacity signal.

**Accelerator**: before running this section, set **Settings -> Accelerator -> GPU T4 x2** (top right of the Kaggle kernel editor) so both GPUs are visible to `torch.cuda.device_count()`.

**Run order**: this section is fully independent of 10.7-10.10 (v1's training/validation/download cells) but depends on 10.1-10.6 having already been run in this session -- it reuses their dataset locations, the rebuilt training targets, `AlphaUNet`/`recover`/`save_checkpoint`, and `AlphaCropDataset`/`alpha_loss`/`evaluate` rather than redefining them. Run 10.1 -> 10.2 -> 10.3 -> 10.4 -> 10.5 -> 10.6, then 11.1 -> 11.2 -> 11.3 -> 11.4 -> 11.5.


### 11.1 Stamps and crisp on-the-fly composites

`CrispCompositeDataset` composites the calibrated AriaTender stamps (or, as a fallback, the `watermarks_processed` marks) onto **native-resolution** `wm_backgrounds_v2` pages, entirely in memory, with no noise and no blur -- these samples exist specifically to be crisp, the regime v1 never saw. Placement (pattern, scale range, rotation) mirrors `scripts/wm_dataset/compositor.py`'s `MARK_REGISTRY`, inlined here (not imported) to keep this notebook self-contained.


In [ ]:
import math
import pathlib
import cv2
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

for _name in ('AlphaUNet', 'recover', 'AlphaCropDataset', 'alpha_loss', 'evaluate',
              'dilate_bool_mask', 'WM_DATASET', 'BACKGROUNDS_DIR', 'WATERMARKS_DIR',
              'ALPHA_MAX', 'TARGETS_DIR'):
    assert _name in globals(), (
        f'{_name!r} is not defined -- section 11 depends on 10.1-10.6 having already '
        f'run in this session (this notebook is self-contained per cell, but 11.x '
        f'reuses 10.x definitions rather than redefining them). Run 10.1 through 10.6 first.'
    )

P_REAL_STAMP = 0.7     # probability of drawing a calibrated stamp over a watermarks_processed extra
P_JPEG = 0.4            # probability of JPEG re-encoding the observed crop (target stays clean)
JPEG_QUALITY_RANGE = (75, 95)
PAGE_SCALE_RANGE = (0.6, 1.6)
VAL_BG_FRACTION = 0.10  # last 10% of backgrounds held out for crisp validation, never used in training

# --- 11.1a: locate stamps/ ---------------------------------------------------
STAMPS_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/stamps'),
    pathlib.Path('/kaggle/input/train-and-test2/stamps'),
]
STAMPS_DIR = None
for cand in STAMPS_CANDIDATES:
    if cand.exists():
        STAMPS_DIR = cand
        break
if STAMPS_DIR is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('ariatender_wide.png') if p.is_file()]
    if matches:
        STAMPS_DIR = matches[0].parent
if STAMPS_DIR is not None:
    print('STAMPS_DIR ->', STAMPS_DIR)
else:
    print('[warn] stamps/ not found under /kaggle/input -- upload it (see section 11 markdown) for '
          'calibrated-opacity crisp composites. Falling back to watermarks_processed marks only.')

# --- 11.1b: load calibrated stamps + watermarks_processed extras ------------
# scale_ranges: fraction of the SCALED PAGE WIDTH the placed instance should
# span, copied verbatim from scripts/wm_dataset/compositor.py's MARK_REGISTRY
# (inlined, not imported, to keep this notebook self-contained -- see 10.5's
# markdown for the same reasoning applied to AlphaUNet).
PATTERNS = ('single', 'oversize', 'corner', 'lattice', 'diagonal')
PATTERN_WEIGHTS = {'single': 0.30, 'oversize': 0.18, 'lattice': 0.24, 'diagonal': 0.16, 'corner': 0.12}
SCALE_RANGES = {
    'ariatender_stacked': {'single': (0.60, 1.00), 'oversize': (1.05, 1.9), 'corner': (0.12, 0.28),
                            'lattice': (0.07, 0.14), 'diagonal': (0.07, 0.14)},
    'ariatender_wide': {'single': (0.55, 1.00), 'oversize': (1.05, 1.90), 'corner': (0.20, 0.42),
                         'lattice': (0.14, 0.28), 'diagonal': (0.14, 0.28)},
}
DEFAULT_SCALE_RANGE = SCALE_RANGES['ariatender_stacked']  # used for watermarks_processed extras too

CALIBRATED_STAMP_NAMES = ('ariatender_stacked', 'ariatender_wide')
calibrated_stamps = []  # [{'name', 'ink': (H,W,3) float32 [0,1], 'alpha': (H,W) float32 [0,1]}]
if STAMPS_DIR is not None:
    for name in CALIBRATED_STAMP_NAMES:
        p = STAMPS_DIR / f'{name}.png'
        if not p.is_file():
            print(f'[warn] {p} missing, skipping')
            continue
        rgba = np.asarray(Image.open(p).convert('RGBA'), dtype=np.float32) / 255.0
        calibrated_stamps.append({'name': name, 'ink': rgba[..., :3].copy(), 'alpha': rgba[..., 3].copy()})

extra_stamps = []  # [{'name', 'ink': (H,W,3) float32 [0,1], 'coverage': (H,W) float32 {0,1}}]
if WATERMARKS_DIR is not None and WATERMARKS_DIR.exists():
    for p in sorted(WATERMARKS_DIR.glob('*.png')):
        rgba = np.asarray(Image.open(p).convert('RGBA'), dtype=np.float32) / 255.0
        extra_stamps.append({'name': p.stem, 'ink': rgba[..., :3].copy(),
                              'coverage': (rgba[..., 3] > 0.5).astype(np.float32)})

assert calibrated_stamps or extra_stamps, (
    'No stamps available at all (neither stamps/ nor watermarks_processed/) -- '
    'cannot build crisp composites. See section 11 markdown for what to upload.'
)
print(f'{len(calibrated_stamps)} calibrated stamp(s), {len(extra_stamps)} watermarks_processed extra stamp(s)')

# --- 11.1c: backgrounds, held out slice for crisp validation -----------------
BG_EXTS = ('.png', '.jpg', '.jpeg')
all_backgrounds = sorted(p for p in BACKGROUNDS_DIR.iterdir() if p.suffix.lower() in BG_EXTS)
assert all_backgrounds, f'No background images found in {BACKGROUNDS_DIR}'
n_val_bg = max(1, int(round(VAL_BG_FRACTION * len(all_backgrounds))))
TRAIN_BACKGROUNDS = all_backgrounds[:-n_val_bg]
VAL_BACKGROUNDS = all_backgrounds[-n_val_bg:]
assert TRAIN_BACKGROUNDS and VAL_BACKGROUNDS
print(f'{len(all_backgrounds)} background page(s): {len(TRAIN_BACKGROUNDS)} train, '
      f'{len(VAL_BACKGROUNDS)} held out for crisp validation (never used in training)')

# --- 11.1d: shared weight/target computation (same definitions as 10.6's ----
# AlphaCropDataset.__getitem__ -- reuses 10.6's dilate_bool_mask rather than
# redefining dilation, per this notebook's rule of not editing section 10).
def make_weights(alpha_crop, obs_crop, clean_crop, alpha_max):
    weight = 1.0 + 4.0 * dilate_bool_mask(alpha_crop > 0.01, 7)
    diff_mean = np.abs(obs_crop - clean_crop).mean(axis=2)
    visible = dilate_bool_mask(diff_mean > (4.0 / 255.0), 3)
    w_alpha = weight * np.maximum((alpha_crop <= 0.01).astype(np.float32), visible)
    w_rec = weight * (alpha_crop <= alpha_max).astype(np.float32)
    alpha_clamped = np.clip(alpha_crop, 0.0, alpha_max)
    return alpha_clamped, w_alpha, w_rec

# --- 11.1e: placement patterns (ported from compositor.py's private helpers,
# adapted to take an explicit rng and operate in SCALED page pixel coords) ---
def choose_pattern(rng):
    w = np.array([PATTERN_WEIGHTS[p] for p in PATTERNS], dtype=np.float64)
    return str(rng.choice(PATTERNS, p=w / w.sum()))

def placements_for(pattern, w, h, rng, frac_range):
    if pattern == 'single':
        frac = float(rng.uniform(*frac_range))
        angle = float(rng.uniform(-8, 8)) if rng.random() < 0.7 else float(rng.uniform(20, 45) * rng.choice([-1, 1]))
        cx, cy = float(rng.uniform(0.35, 0.65) * w), float(rng.uniform(0.35, 0.65) * h)
        return [(cx, cy, frac, angle)]
    if pattern == 'oversize':
        frac = float(rng.uniform(*frac_range))
        angle = float(rng.uniform(-10, 10)) if rng.random() < 0.75 else float(rng.uniform(15, 40) * rng.choice([-1, 1]))
        cx, cy = float(rng.uniform(0.20, 0.80) * w), float(rng.uniform(0.18, 0.82) * h)
        return [(cx, cy, frac, angle)]
    if pattern == 'corner':
        frac = float(rng.uniform(*frac_range))
        angle = float(rng.uniform(-10, 10))
        margin, corner = 0.06, int(rng.integers(0, 4))
        cx = margin * w if corner in (0, 3) else (1 - margin) * w
        cy = margin * h if corner in (0, 1) else (1 - margin) * h
        return [(float(cx), float(cy), frac, angle)]
    # lattice / diagonal -- a tiled grid, optionally rotated as a whole
    diagonal = (pattern == 'diagonal')
    frac = float(rng.uniform(*frac_range))
    grid_angle = float(rng.uniform(30, 50) * rng.choice([-1, 1])) if diagonal else float(rng.uniform(-6, 6))
    stamp_w = frac * w
    spacing_x = stamp_w * float(rng.uniform(1.6, 2.4))
    spacing_y = spacing_x * float(rng.uniform(0.7, 1.1))
    jitter = spacing_x * 0.08
    diag = math.hypot(w, h)
    n_cols, n_rows = int(diag / spacing_x) + 4, int(diag / spacing_y) + 4
    theta = math.radians(grid_angle)
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    cx0, cy0 = w / 2.0, h / 2.0
    out = []
    for i in range(-n_rows // 2, n_rows // 2 + 1):
        for j in range(-n_cols // 2, n_cols // 2 + 1):
            gx = j * spacing_x + rng.uniform(-jitter, jitter)
            gy = i * spacing_y + rng.uniform(-jitter, jitter)
            px = cx0 + gx * cos_t - gy * sin_t
            py = cy0 + gx * sin_t + gy * cos_t
            if -stamp_w < px < w + stamp_w and -stamp_w < py < h + stamp_w:
                out.append((float(px), float(py), frac, grid_angle))
    return out

def scale_stamp(stamp_ink, stamp_alpha, scale):
    """Resizes a stamp to `scale`, premultiplied: ink*alpha and alpha are
    resized, then divided back, so the (arbitrary) ink colour stored under
    fully transparent pixels can't bleed into the mark's edge."""
    sh, sw = stamp_alpha.shape[:2]
    tw, th = max(1, int(round(sw * scale))), max(1, int(round(sh * scale)))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_CUBIC
    a = np.clip(cv2.resize(stamp_alpha, (tw, th), interpolation=interp), 0.0, 1.0)
    pm = cv2.resize(stamp_ink * stamp_alpha[..., None], (tw, th), interpolation=interp)
    ink = np.clip(pm / np.maximum(a, 1e-4)[..., None], 0.0, 1.0)
    return ink.astype(np.float32), a.astype(np.float32)


def warp_tile_into_crop(s_ink, s_alpha, angle_deg, opacity_mult, cx, cy, crop):
    """Rotates a scaled stamp about its centre (counter-clockwise for positive
    angles, like PIL's rotate) and places that centre at (cx, cy) in crop
    coordinates, rendering only the tile's bounding box clipped to the crop.
    Returns (ink, alpha, x, y) for paste_over_crop, or None if it misses."""
    th, tw = s_alpha.shape[:2]
    M = cv2.getRotationMatrix2D((tw / 2.0, th / 2.0), float(angle_deg), 1.0)
    M[0, 2] += cx - tw / 2.0
    M[1, 2] += cy - th / 2.0
    corners = np.array([[0, 0, 1], [tw, 0, 1], [0, th, 1], [tw, th, 1]], dtype=np.float64) @ M.T
    x0 = max(0, int(np.floor(corners[:, 0].min())))
    y0 = max(0, int(np.floor(corners[:, 1].min())))
    x1 = min(crop, int(np.ceil(corners[:, 0].max())) + 1)
    y1 = min(crop, int(np.ceil(corners[:, 1].max())) + 1)
    if x1 <= x0 or y1 <= y0:
        return None
    M[0, 2] -= x0
    M[1, 2] -= y0
    size = (x1 - x0, y1 - y0)
    a = cv2.warpAffine(s_alpha, M, size, flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
    ink = cv2.warpAffine(s_ink, M, size, flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0.0, 0.0, 0.0))
    a = np.clip(a * float(opacity_mult), 0.0, 1.0).astype(np.float32)
    return np.clip(ink, 0.0, 1.0).astype(np.float32), a, x0, y0

def paste_over_crop(canvas_rgb, canvas_a, tile_ink, tile_alpha, x, y):
    th, tw = tile_alpha.shape[:2]
    ch, cw = canvas_a.shape[:2]
    src_x0, src_y0 = max(0, -x), max(0, -y)
    dst_x0, dst_y0 = max(0, x), max(0, y)
    dst_x1, dst_y1 = min(cw, x + tw), min(ch, y + th)
    if dst_x1 <= dst_x0 or dst_y1 <= dst_y0:
        return
    src_x1, src_y1 = src_x0 + (dst_x1 - dst_x0), src_y0 + (dst_y1 - dst_y0)
    t_rgb = tile_ink[src_y0:src_y1, src_x0:src_x1, :]
    t_a = tile_alpha[src_y0:src_y1, src_x0:src_x1]
    dst_rgb = canvas_rgb[dst_y0:dst_y1, dst_x0:dst_x1, :]
    dst_a = canvas_a[dst_y0:dst_y1, dst_x0:dst_x1]
    canvas_rgb[dst_y0:dst_y1, dst_x0:dst_x1, :] = t_rgb * t_a[..., None] + dst_rgb * (1.0 - t_a[..., None])
    canvas_a[dst_y0:dst_y1, dst_x0:dst_x1] = t_a + dst_a * (1.0 - t_a)

# --- 11.1f: the dataset -------------------------------------------------------
class CrispCompositeDataset(torch.utils.data.Dataset):
    """On-the-fly crisp composites: native-resolution backgrounds + calibrated
    (or watermarks_processed-extra) marks, NO noise, NO blur -- these samples
    exist to teach the network the crisp, native-resolution screenshot regime
    that v1 (trained only on downsampled/noisy/JPEG Kaggle pages, 10.6) never
    saw. Returns the same 5 tensors as 10.6's AlphaCropDataset (obs, clean,
    alpha, w_alpha, w_rec), with the same weight definitions (make_weights).

    Deterministic (val) items are a fixed function of (seed, idx): background
    drawn from a held-out slice of backgrounds that training never sees.
    """

    def __init__(self, crop, samples_per_epoch=None, deterministic=False, seed=0):
        self.crop = int(crop)
        self.samples_per_epoch = samples_per_epoch
        self.deterministic = deterministic
        self.seed = int(seed)
        self.backgrounds = VAL_BACKGROUNDS if deterministic else TRAIN_BACKGROUNDS

    def __len__(self):
        if self.samples_per_epoch is not None:
            return int(self.samples_per_epoch)
        return len(self.backgrounds) if self.deterministic else 4000

    def _sample_mark(self, rng):
        use_calibrated = bool(calibrated_stamps) and (not extra_stamps or rng.random() < P_REAL_STAMP)
        if use_calibrated:
            s = calibrated_stamps[int(rng.integers(0, len(calibrated_stamps)))]
            opacity_mult = float(rng.uniform(0.5, 1.5))
            return s['ink'], s['alpha'], opacity_mult, SCALE_RANGES.get(s['name'], DEFAULT_SCALE_RANGE)
        s = extra_stamps[int(rng.integers(0, len(extra_stamps)))]
        opacity_mult = float(rng.uniform(0.10, 0.45))
        return s['ink'], s['coverage'], opacity_mult, DEFAULT_SCALE_RANGE

    def __getitem__(self, idx, _force_no_jpeg=False):
        crop = self.crop
        if self.deterministic:
            rng = np.random.default_rng([self.seed, int(idx)])
            bg_path = self.backgrounds[idx % len(self.backgrounds)]
        else:
            rng = np.random.default_rng()
            bg_path = self.backgrounds[int(rng.integers(0, len(self.backgrounds)))]

        # cv2 decodes PNG/JPEG noticeably faster than PIL; the page is only
        # needed at full size to find its dimensions and cut the crop window.
        bg_u8 = cv2.cvtColor(cv2.imread(str(bg_path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        bh, bw = bg_u8.shape[:2]
        page_scale = float(rng.uniform(*PAGE_SCALE_RANGE))
        Ws, Hs = max(16, int(round(bw * page_scale))), max(16, int(round(bh * page_scale)))
        interp = cv2.INTER_AREA if page_scale < 1.0 else cv2.INTER_CUBIC
        # Pages smaller than the crop (rare) are resized whole and reflect-padded,
        # as before. Otherwise only the source window that maps onto the crop is
        # resized (see below) -- resizing a whole 1800-px page for one 512-px crop
        # dominated the per-sample cost.
        small_page = (Ws < crop) or (Hs < crop)
        if small_page:
            bg_np = cv2.resize(bg_u8, (Ws, Hs), interpolation=interp).astype(np.float32) / 255.0
            pad_h, pad_w = max(0, crop - Hs), max(0, crop - Ws)
            page = np.pad(bg_np, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect') if (pad_h or pad_w) else bg_np
            Hp, Wp = page.shape[:2]
        else:
            Hp, Wp = Hs, Ws

        ink, alpha_full, opacity_mult, scale_range = self._sample_mark(rng)
        pattern = choose_pattern(rng)
        frac_range = scale_range[pattern]
        placements = placements_for(pattern, Ws, Hs, rng, frac_range)

        centre_on_mark = bool(placements) and (rng.random() < 0.70)
        if centre_on_mark:
            cx, cy, _, _ = placements[int(rng.integers(0, len(placements)))]
        else:
            cx, cy = float(rng.uniform(0, Wp)), float(rng.uniform(0, Hp))
        y0 = int(np.clip(cy - crop / 2.0, 0, max(0, Hp - crop)))
        x0 = int(np.clip(cx - crop / 2.0, 0, max(0, Wp - crop)))

        if small_page:
            clean_crop = page[y0:y0 + crop, x0:x0 + crop, :].copy()
        else:
            # Source-pixel window covering the crop (+2 px so the resampling
            # kernel sees real neighbours), resized by the same page_scale, then
            # the exact crop cut out of it. obs is composited onto this same
            # clean_crop below, so the targets stay exact.
            sx0 = max(0, int(np.floor(x0 / page_scale)) - 2)
            sy0 = max(0, int(np.floor(y0 / page_scale)) - 2)
            sx1 = min(bw, int(np.ceil((x0 + crop) / page_scale)) + 2)
            sy1 = min(bh, int(np.ceil((y0 + crop) / page_scale)) + 2)
            win = bg_u8[sy0:sy1, sx0:sx1]
            ox, oy = int(round(sx0 * page_scale)), int(round(sy0 * page_scale))
            ww = max(crop + (x0 - ox), int(round((sx1 - sx0) * page_scale)))
            wh = max(crop + (y0 - oy), int(round((sy1 - sy0) * page_scale)))
            win_s = cv2.resize(win, (ww, wh), interpolation=interp)
            clean_crop = win_s[y0 - oy:y0 - oy + crop, x0 - ox:x0 - ox + crop].astype(np.float32) / 255.0
            assert clean_crop.shape[:2] == (crop, crop), (clean_crop.shape, crop)
        obs_crop = clean_crop.copy()
        alpha_crop = np.zeros((crop, crop), dtype=np.float32)

        aspect = alpha_full.shape[0] / alpha_full.shape[1]
        scaled_cache = {}
        for (pcx, pcy, frac, angle) in placements:
            approx_w = frac * Ws
            approx_h = approx_w * aspect
            radius = 0.75 * math.hypot(approx_w, approx_h) + 8
            if pcx + radius < x0 or pcx - radius > x0 + crop or pcy + radius < y0 or pcy - radius > y0 + crop:
                continue
            scale = (frac * Ws) / alpha_full.shape[1]
            if scale <= 0.01:
                continue
            # Resize the stamp once per distinct scale (lattice/diagonal patterns
            # reuse one scale for every grid cell), then rotate + translate it
            # straight into just the part of the crop it covers -- a big single
            # or oversize tile is never rendered in full.
            if scale not in scaled_cache:
                scaled_cache[scale] = scale_stamp(ink, alpha_full, scale)
            s_ink, s_alpha = scaled_cache[scale]
            placed = warp_tile_into_crop(s_ink, s_alpha, angle, opacity_mult, pcx - x0, pcy - y0, crop)
            if placed is None:
                continue
            tile_ink, tile_alpha, tx, ty = placed
            paste_over_crop(obs_crop, alpha_crop, tile_ink, tile_alpha, tx, ty)

        if not _force_no_jpeg and rng.random() < P_JPEG:
            q = int(rng.integers(JPEG_QUALITY_RANGE[0], JPEG_QUALITY_RANGE[1] + 1))
            obs_u8 = np.clip(np.round(obs_crop * 255.0), 0, 255).astype(np.uint8)
            ok, enc = cv2.imencode('.jpg', cv2.cvtColor(obs_u8, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), q])
            assert ok, 'JPEG encode failed'
            dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
            obs_crop = cv2.cvtColor(dec, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

        alpha_c, w_alpha_c, w_rec_c = make_weights(alpha_crop, obs_crop, clean_crop, ALPHA_MAX)

        obs_t = torch.from_numpy(np.ascontiguousarray(obs_crop.transpose(2, 0, 1)).astype(np.float32))
        clean_t = torch.from_numpy(np.ascontiguousarray(clean_crop.transpose(2, 0, 1)).astype(np.float32))
        alpha_t = torch.from_numpy(np.ascontiguousarray(alpha_c[None, :, :]))
        w_alpha_t = torch.from_numpy(np.ascontiguousarray(w_alpha_c[None, :, :]))
        w_rec_t = torch.from_numpy(np.ascontiguousarray(w_rec_c[None, :, :]))
        return obs_t, clean_t, alpha_t, w_alpha_t, w_rec_t

# --- 11.1g: physics self-check + preview -------------------------------------
CROP_PREVIEW = 512
_check_ds = CrispCompositeDataset(CROP_PREVIEW, samples_per_epoch=20, deterministic=False, seed=123)
_max_alpha, _n_centered, _n_centered_with_mark, _bad_zero = 0.0, 0, 0, 0
for _i in range(len(_check_ds)):
    _obs, _clean, _alpha, _wa, _wr = _check_ds.__getitem__(_i, _force_no_jpeg=True)
    _obs_np, _clean_np, _alpha_np = _obs.numpy(), _clean.numpy(), _alpha.numpy()[0]
    _max_alpha = max(_max_alpha, float(_alpha_np.max()))
    _zero_mask = (_alpha_np <= 1e-8)
    _diff = np.abs(_obs_np - _clean_np)[:, _zero_mask] if _zero_mask.any() else np.zeros((3, 0))
    if _diff.size and float(_diff.max()) > 1e-6:
        _bad_zero += 1
print(f'[physics check] 20 samples (no JPEG): max alpha={_max_alpha:.3f} (<=1 required), '
      f'{_bad_zero}/20 samples had obs!=clean where alpha==0 (should be 0)')
assert _max_alpha <= 1.0 + 1e-6
assert _bad_zero == 0, 'obs must equal clean exactly where alpha==0 (before JPEG)'

_centered_ds = CrispCompositeDataset(CROP_PREVIEW, samples_per_epoch=100, deterministic=False, seed=321)
_n_has_mark = 0
for _i in range(len(_centered_ds)):
    _, _, _alpha, _, _ = _centered_ds[_i]
    if float((_alpha.numpy()[0] > 0.05).mean()) > 0:
        _n_has_mark += 1
print(f'[coverage check] {_n_has_mark}/100 crops contain any alpha>0.05 pixel '
      f'({100 * _n_has_mark / 100:.0f}% -- most crops are drawn near a mark placement)')

_preview_ds = CrispCompositeDataset(CROP_PREVIEW, samples_per_epoch=4, deterministic=True, seed=0)
fig, axes = plt.subplots(4, 3, figsize=(9, 12))
for row in range(4):
    obs, clean, alpha, _wa, _wr = _preview_ds[row]
    axes[row, 0].imshow(np.clip(obs.permute(1, 2, 0).numpy(), 0, 1)); axes[row, 0].set_title('observed')
    axes[row, 1].imshow(alpha[0].numpy(), cmap='inferno', vmin=0, vmax=ALPHA_MAX); axes[row, 1].set_title('alpha')
    axes[row, 2].imshow(np.clip(clean.permute(1, 2, 0).numpy(), 0, 1)); axes[row, 2].set_title('clean')
    for c in range(3):
        axes[row, c].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/alpha_v2_crisp_preview.png', dpi=110)
plt.show()
print('Saved preview to /kaggle/working/alpha_v2_crisp_preview.png')


### 11.2 Scale augmentation for the Kaggle pages

`ScaledAlphaCropDataset` subclasses 10.6's `AlphaCropDataset`, overriding only `_read_triple` to rescale `obs`/`clean`/`alpha` together (same interpolation for `obs`/`clean`, bilinear for `alpha`) before the parent class's existing crop/weight logic runs unchanged -- so every random-crop, weighting, and clamping rule from 10.6 still applies. Deterministic (validation) items skip scaling entirely (scale = 1.0), so validation stays comparable across epochs.


In [ ]:
import cv2
import numpy as np

for _name in ('AlphaCropDataset',):
    assert _name in globals(), f'{_name!r} is not defined -- run 10.6 first.'

class ScaledAlphaCropDataset(AlphaCropDataset):
    """10.6's AlphaCropDataset + a random page-scale augmentation (0.6x-1.6x)
    applied to obs/clean/alpha together BEFORE the parent class's crop/weight
    logic runs -- only _read_triple is overridden, nothing else changes, so
    every rule 10.6 already applies (crop selection biased toward mark pixels,
    gain jitter, weight computation, ALPHA_MAX clamping) is inherited as-is.
    Deterministic (validation) items are NOT scaled, so validation numbers
    stay comparable run to run.
    """

    def __init__(self, split, crop, samples_per_epoch=None, deterministic=False, seed=0,
                 scale_range=(0.6, 1.6)):
        super().__init__(split, crop, samples_per_epoch=samples_per_epoch,
                          deterministic=deterministic, seed=seed)
        self.scale_range = scale_range

    def _read_triple(self, i):
        obs, clean, alpha = super()._read_triple(i)
        if self.deterministic:
            return obs, clean, alpha
        rng = np.random.default_rng()
        s = float(rng.uniform(*self.scale_range))
        h, w = alpha.shape[:2]
        nh, nw = max(16, int(round(h * s))), max(16, int(round(w * s)))
        interp_img = cv2.INTER_AREA if s < 1.0 else cv2.INTER_CUBIC
        obs = cv2.resize(obs, (nw, nh), interpolation=interp_img)
        clean = cv2.resize(clean, (nw, nh), interpolation=interp_img)
        alpha = cv2.resize(alpha, (nw, nh), interpolation=cv2.INTER_LINEAR)
        return obs, clean, alpha

print('ScaledAlphaCropDataset defined (subclasses AlphaCropDataset, overrides _read_triple only).')


### 11.3 Train v2 (both GPUs)

This is the actual training cell. Expected wall-clock time is **unknown until measured** -- read the first epoch's `time=...s` and `img/s=...` in the printed log and scale by `EPOCHS` to estimate the whole run. A partial run is still useful: `best.pt`/`last.pt` are updated every epoch, and re-running this cell resumes from `last.pt` (`V2_LAST_PATH`) where it left off.

If you hit a CUDA out-of-memory error: first drop `BATCH_PER_GPU` (e.g. 8 -> 4), then (if still OOM) drop `CROP` (e.g. 512 -> 384).


In [ ]:
import csv
import copy
import math
import pathlib
import random
import shutil
import time
import torch
from IPython.display import FileLink, display

for _name in ('AlphaUNet', 'recover', 'save_checkpoint', 'AlphaCropDataset', 'evaluate',
              'CrispCompositeDataset', 'ScaledAlphaCropDataset', 'WIDTHS', 'ALPHA_MAX', 'FORMAT_VERSION'):
    assert _name in globals(), f'{_name!r} is not defined -- run 10.1-10.6 and 11.1-11.2 first.'

EPOCHS = 60
BATCH_PER_GPU = 8
CROP = 512
SAMPLES_PER_EPOCH = 4000
P_CRISP = 0.5                 # fraction of each training batch-stream drawn from CrispCompositeDataset
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
PATIENCE = 25                 # early-stop after this many epochs with no crisp-val improvement
EMA_DECAY = 0.999
SEED = 0
SNAPSHOT_EVERY = 10
LAMBDAS = (5.0, 2.0, 0.5)     # alpha, rec, comp -- rec doubled vs v1 (10.6: 1.0) to counter under-removal
INIT_FROM = None              # optional: path to a v1 checkpoint (10.7's best.pt) to warm-start from
CRISP_VAL_SIZE = 300
KAGGLE_VAL_SIZE = 300

torch.manual_seed(SEED)
random.seed(SEED)

N_GPUS = torch.cuda.device_count()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if N_GPUS:
    print(f'GPUs ({N_GPUS}):', [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
else:
    print('No GPU visible -- training on CPU (slow; fine for a smoke test, not for a real run).')

V2_RUN_DIR = pathlib.Path('/kaggle/working/runs/alpha_net_v2')
V2_RUN_DIR.mkdir(parents=True, exist_ok=True)
V2_LAST_PATH = V2_RUN_DIR / 'last.pt'
V2_BEST_PATH = V2_RUN_DIR / 'best.pt'
V2_METRICS_PATH = V2_RUN_DIR / 'metrics.csv'

# --- data -------------------------------------------------------------------
GLOBAL_BATCH = BATCH_PER_GPU * max(1, N_GPUS)

crisp_train_ds = CrispCompositeDataset(CROP, samples_per_epoch=None, deterministic=False, seed=SEED)
kaggle_train_ds = ScaledAlphaCropDataset('train', CROP, samples_per_epoch=None, deterministic=False, seed=SEED)
crisp_val_ds = CrispCompositeDataset(CROP, samples_per_epoch=CRISP_VAL_SIZE, deterministic=True, seed=SEED)
kaggle_val_ds = ScaledAlphaCropDataset('val', CROP, samples_per_epoch=KAGGLE_VAL_SIZE, deterministic=True, seed=SEED)

class MixedTrainDataset(torch.utils.data.Dataset):
    """Each item independently comes from crisp_ds with probability p_crisp,
    else from kaggle_ds -- a simple mixing dataset rather than a
    ConcatDataset+WeightedRandomSampler, so the epoch length is just
    SAMPLES_PER_EPOCH regardless of either source dataset's own size."""

    def __init__(self, crisp_ds, kaggle_ds, length, p_crisp):
        self.crisp_ds, self.kaggle_ds = crisp_ds, kaggle_ds
        self.length, self.p_crisp = int(length), float(p_crisp)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if random.random() < self.p_crisp:
            return self.crisp_ds[random.randrange(len(self.crisp_ds))]
        return self.kaggle_ds[random.randrange(len(self.kaggle_ds))]

train_ds = MixedTrainDataset(crisp_train_ds, kaggle_train_ds, SAMPLES_PER_EPOCH, P_CRISP)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=GLOBAL_BATCH, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(device == 'cuda'), drop_last=True, persistent_workers=(NUM_WORKERS > 0),
)
crisp_val_loader = torch.utils.data.DataLoader(
    crisp_val_ds, batch_size=GLOBAL_BATCH, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device == 'cuda'), persistent_workers=(NUM_WORKERS > 0),
)
kaggle_val_loader = torch.utils.data.DataLoader(
    kaggle_val_ds, batch_size=GLOBAL_BATCH, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device == 'cuda'), persistent_workers=(NUM_WORKERS > 0),
)
iters_per_epoch = len(train_loader)
assert iters_per_epoch > 0, 'Training loader produced zero batches -- reduce BATCH_PER_GPU or raise SAMPLES_PER_EPOCH.'

# --- loss (10.6's alpha_loss with LAMBDAS parameterised rather than the
# hardcoded 5.0/1.0/0.5 it uses -- see the markdown for why rec is doubled) --
def alpha_loss_v2(pred_alpha, pred_ink, obs, clean, alpha, w_alpha, w_rec):
    rec = recover(obs, pred_alpha, pred_ink, ALPHA_MAX)
    l_alpha = (w_alpha * (pred_alpha - alpha).abs()).mean()
    l_rec = (w_rec * (rec - clean).abs().mean(dim=1, keepdim=True)).mean()
    comp = pred_alpha * pred_ink + (1.0 - pred_alpha) * clean
    l_comp = (w_rec * (comp - obs).abs().mean(dim=1, keepdim=True)).mean()
    total = LAMBDAS[0] * l_alpha + LAMBDAS[1] * l_rec + LAMBDAS[2] * l_comp
    parts = {'L_alpha': float(l_alpha.item()), 'L_rec': float(l_rec.item()), 'L_comp': float(l_comp.item())}
    return total, parts

# --- model, DataParallel, EMA ------------------------------------------------
model = AlphaUNet(widths=WIDTHS, alpha_max=ALPHA_MAX)
if INIT_FROM is not None and not V2_LAST_PATH.exists():
    _init_ckpt = torch.load(INIT_FROM, map_location='cpu', weights_only=True)
    model.load_state_dict(_init_ckpt['model_state'])
    print('Warm-started from', INIT_FROM)
model = model.to(device)

# DataParallel rather than DistributedDataParallel: this cell runs inline in a
# notebook process with no torchrun/launch script to give DDP its process
# group, and AlphaUNet uses GroupNorm (no running batch statistics), so DP's
# per-forward-call replication across GPUs has no cross-GPU norm-sync gap to
# worry about -- the one real downside DP usually has relative to DDP.
USE_DP = (N_GPUS >= 2) and (device == 'cuda')
model_run = torch.nn.DataParallel(model) if USE_DP else model
print(f'DataParallel: {USE_DP} (N_GPUS={N_GPUS}) -- global batch = {GLOBAL_BATCH} ({BATCH_PER_GPU} x {max(1, N_GPUS)})')

def ema_avg_fn(averaged_param, param, num_averaged):
    return EMA_DECAY * averaged_param + (1.0 - EMA_DECAY) * param

ema_model = torch.optim.swa_utils.AveragedModel(model, avg_fn=ema_avg_fn)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
warmup_iters = iters_per_epoch
total_iters = max(1, EPOCHS * iters_per_epoch)

def lr_lambda(it):
    if it < warmup_iters:
        return (it + 1) / max(1, warmup_iters)
    progress = min(max((it - warmup_iters) / max(1, total_iters - warmup_iters), 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
use_amp = (device == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

start_epoch = 1
best_score = float('inf')
epochs_no_improve = 0

if V2_LAST_PATH.exists():
    print('Continuing training from checkpoint:', V2_LAST_PATH)
    # weights_only=False: last.pt is written exclusively by this cell and
    # carries optimizer/scheduler/scaler/EMA state (not just tensors/
    # primitives), so it cannot use the stricter weights_only=True loading
    # best.pt uses.
    ckpt = torch.load(V2_LAST_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    ema_model.module.load_state_dict(ckpt['ema_state'])
    opt.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    best_score = ckpt.get('best_score', float('inf'))
    epochs_no_improve = ckpt.get('epochs_no_improve', 0)
    start_epoch = int(ckpt['epoch']) + 1
else:
    print('Starting fresh training')

TRAIN_CONFIG = dict(
    epochs=EPOCHS, batch_per_gpu=BATCH_PER_GPU, global_batch=GLOBAL_BATCH, n_gpus=N_GPUS,
    crop=CROP, samples_per_epoch=SAMPLES_PER_EPOCH, p_crisp=P_CRISP, lr=LR,
    weight_decay=WEIGHT_DECAY, num_workers=NUM_WORKERS, patience=PATIENCE, ema_decay=EMA_DECAY,
    seed=SEED, widths=list(WIDTHS), alpha_max=ALPHA_MAX, lambdas=list(LAMBDAS), init_from=INIT_FROM,
)

best_epoch, best_metrics = None, None

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t_epoch0 = time.time()
    part_sums = {'total': 0.0, 'L_alpha': 0.0, 'L_rec': 0.0, 'L_comp': 0.0}
    n_batches, n_items = 0, 0

    for obs, clean, alpha, w_alpha, w_rec in train_loader:
        obs = obs.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)
        alpha = alpha.to(device, non_blocking=True)
        w_alpha = w_alpha.to(device, non_blocking=True)
        w_rec = w_rec.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        try:
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                pred_alpha, pred_ink = model_run(obs)
                loss, parts = alpha_loss_v2(pred_alpha, pred_ink, obs, clean, alpha, w_alpha, w_rec)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                raise RuntimeError(
                    f'CUDA out of memory during a training step (BATCH_PER_GPU={BATCH_PER_GPU}, '
                    f'CROP={CROP}). Try a smaller BATCH_PER_GPU (e.g. 4) and/or CROP (e.g. 384).'
                ) from e
            raise

        scheduler.step()
        ema_model.update_parameters(model)
        part_sums['total'] += float(loss.item())
        for k in ('L_alpha', 'L_rec', 'L_comp'):
            part_sums[k] += parts[k]
        n_batches += 1
        n_items += obs.shape[0]

    n_batches = max(1, n_batches)
    train_parts = {k: v / n_batches for k, v in part_sums.items()}
    epoch_seconds = time.time() - t_epoch0
    imgs_per_sec = n_items / max(epoch_seconds, 1e-6)

    crisp_metrics = evaluate(ema_model.module, crisp_val_loader, device)
    kaggle_metrics = evaluate(ema_model.module, kaggle_val_loader, device)
    val_score = crisp_metrics['val_score']
    lr_now = scheduler.get_last_lr()[0]

    print(
        f'epoch {epoch}/{EPOCHS} train_loss={train_parts["total"]:.4f} '
        f'(alpha={train_parts["L_alpha"]:.4f} rec={train_parts["L_rec"]:.4f} comp={train_parts["L_comp"]:.4f}) '
        f'crisp: val_score={crisp_metrics["val_score"]:.4f} alpha_mae_wm={crisp_metrics["alpha_mae_wm"]:.4f} '
        f'improvement_wm={crisp_metrics["improvement_wm"]:.3f} | kaggle: val_score={kaggle_metrics["val_score"]:.4f} '
        f'| lr={lr_now:.2e} time={epoch_seconds:.1f}s img/s={imgs_per_sec:.1f}'
    )

    is_new_file = not V2_METRICS_PATH.exists()
    fieldnames = (['epoch', 'train_loss', 'train_L_alpha', 'train_L_rec', 'train_L_comp', 'lr', 'seconds', 'img_s']
                  + [f'crisp_{k}' for k in crisp_metrics] + [f'kaggle_{k}' for k in kaggle_metrics])
    with open(V2_METRICS_PATH, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if is_new_file:
            w.writeheader()
        row = {'epoch': epoch, 'train_loss': train_parts['total'], 'train_L_alpha': train_parts['L_alpha'],
               'train_L_rec': train_parts['L_rec'], 'train_L_comp': train_parts['L_comp'],
               'lr': lr_now, 'seconds': epoch_seconds, 'img_s': imgs_per_sec}
        row.update({f'crisp_{k}': v for k, v in crisp_metrics.items()})
        row.update({f'kaggle_{k}': v for k, v in kaggle_metrics.items()})
        w.writerow(row)

    combined_val_metrics = {'val_score': val_score, 'crisp': crisp_metrics, 'kaggle': kaggle_metrics}
    improved = val_score < best_score
    if improved:
        best_score = val_score
        epochs_no_improve = 0
        # UNWRAPPED EMA module's state_dict -- no 'module.' prefix, loadable
        # by watermark_remover.alpha_net.load_model(weights_only=True).
        save_checkpoint(V2_BEST_PATH, ema_model.module, epoch, combined_val_metrics, TRAIN_CONFIG)
        best_epoch, best_metrics = epoch, combined_val_metrics
    else:
        epochs_no_improve += 1

    last_ckpt = {
        'format_version': FORMAT_VERSION, 'arch': 'AlphaUNet', 'widths': list(model.widths),
        'alpha_max': float(model.alpha_max), 'model_state': model.state_dict(),
        'ema_state': ema_model.module.state_dict(),
        'epoch': epoch, 'val_metrics': combined_val_metrics, 'train_config': TRAIN_CONFIG,
        'optimizer': opt.state_dict(), 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
        'best_score': best_score, 'epochs_no_improve': epochs_no_improve,
    }
    torch.save(last_ckpt, V2_LAST_PATH)

    if epoch % SNAPSHOT_EVERY == 0 and V2_BEST_PATH.exists():
        snap_path = pathlib.Path(f'/kaggle/working/alpha_net_v2_best_epoch_{epoch}.pt')
        shutil.copy2(V2_BEST_PATH, snap_path)
        print(f'\n[Checkpoint] Saved best weights at epoch {epoch} to {snap_path}')
        display(FileLink(snap_path.name))

    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch} ({epochs_no_improve} epochs without crisp-val improvement)')
        break

print('\n=== v2 training summary ===')
print('best epoch:', best_epoch)
print('best val metrics:', best_metrics)


### 11.4 v2 examples (val + real pages)

Loads `V2_BEST_PATH` into a fresh `AlphaUNet` (`weights_only=True`, same as `watermark_remover.alpha_net.load_model` would use), then shows 4 crisp validation crops and runs tiled full-page inference (`predict_page`, reused from 10.9 if it already ran this session, else defined here) on 6 real pages from `SCRAPED_DIR`.


In [ ]:
import pathlib
import time
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

for _name in ('AlphaUNet', 'recover', 'V2_RUN_DIR', 'CROP', 'CrispCompositeDataset', 'SCRAPED_DIR'):
    assert _name in globals(), f'{_name!r} is not defined -- run 10.1-10.6 and 11.1-11.3 first.'

V2_BEST_PATH = V2_RUN_DIR / 'best.pt'
assert V2_BEST_PATH.exists(), f'No trained v2 weights at {V2_BEST_PATH} -- run the training cell (11.3) first.'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(V2_BEST_PATH, map_location=device, weights_only=True)
v2_model = AlphaUNet(widths=tuple(ckpt['widths']), alpha_max=float(ckpt['alpha_max'])).to(device)
v2_model.load_state_dict(ckpt['model_state'])
v2_model.eval()
print('v2 best epoch:', ckpt['epoch'])
print('v2 val metrics:', ckpt['val_metrics'])

# --- crisp validation crops ---------------------------------------------------
crisp_val_ds_show = CrispCompositeDataset(CROP, samples_per_epoch=None, deterministic=True, seed=SEED)
N_SHOW = 4
fig, axes = plt.subplots(N_SHOW, 4, figsize=(16, 4 * N_SHOW))
for row in range(N_SHOW):
    idx = row % len(crisp_val_ds_show)
    obs, clean, alpha_gt, _w_alpha, _w_rec = crisp_val_ds_show[idx]
    with torch.no_grad():
        pred_alpha, pred_ink = v2_model(obs.unsqueeze(0).to(device))
        rec = recover(obs.unsqueeze(0).to(device), pred_alpha, pred_ink, float(ckpt['alpha_max']))

    obs_img = obs.permute(1, 2, 0).numpy()
    alpha_img = pred_alpha[0, 0].cpu().numpy()
    rec_img = rec[0].permute(1, 2, 0).cpu().numpy()
    clean_img = clean.permute(1, 2, 0).numpy()

    axes[row, 0].imshow(np.clip(obs_img, 0, 1)); axes[row, 0].set_title('observed')
    axes[row, 1].imshow(alpha_img, cmap='inferno', vmin=0, vmax=float(ckpt['alpha_max'])); axes[row, 1].set_title('predicted alpha')
    axes[row, 2].imshow(np.clip(rec_img, 0, 1)); axes[row, 2].set_title('recovered')
    axes[row, 3].imshow(np.clip(clean_img, 0, 1)); axes[row, 3].set_title('clean')
    for c in range(4):
        axes[row, c].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/alpha_net_v2_crisp_val_examples.png', dpi=120)
plt.show()

# --- real pages: tiled full-page inference ------------------------------------
if 'predict_page' not in globals():
    def feather_weight(tile, overlap):
        overlap = max(1, min(overlap, tile // 2))
        ramp = np.linspace(1.0 / (overlap + 1), 1.0, overlap, dtype=np.float32)
        w = np.ones(tile, dtype=np.float32)
        w[:overlap], w[-overlap:] = ramp, ramp[::-1]
        return np.outer(w, w)

    def predict_page(model, img_rgb_uint8, device, tile=512, overlap=64):
        """Tiled full-page inference (same behaviour as 10.9's predict_page /
        watermark_remover/alpha_net.py's predict_image): reflect-pad to at
        least one tile and a multiple of 16, run overlapping tiles, blend
        with a feathered weight, then call recover ONCE on the merged maps."""
        alpha_max = model.alpha_max
        h, w = img_rgb_uint8.shape[:2]
        pad_h = max(tile, int(np.ceil(h / 16.0) * 16))
        pad_w = max(tile, int(np.ceil(w / 16.0) * 16))
        img_f = img_rgb_uint8.astype(np.float32) / 255.0
        img_padded = np.pad(img_f, ((0, pad_h - h), (0, pad_w - w), (0, 0)), mode='reflect')

        alpha_acc = np.zeros((pad_h, pad_w), dtype=np.float32)
        ink_acc = np.zeros((pad_h, pad_w, 3), dtype=np.float32)
        weight_acc = np.zeros((pad_h, pad_w), dtype=np.float32)

        stride = max(1, tile - overlap)
        ys = list(range(0, max(1, pad_h - tile + 1), stride))
        if ys[-1] != pad_h - tile:
            ys.append(pad_h - tile)
        xs = list(range(0, max(1, pad_w - tile + 1), stride))
        if xs[-1] != pad_w - tile:
            xs.append(pad_w - tile)
        ramp2d = feather_weight(tile, overlap)

        with torch.no_grad():
            for y in ys:
                for x in xs:
                    crop = img_padded[y:y + tile, x:x + tile, :]
                    t = torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0).to(device)
                    a, ink = model(t)
                    a_np = a[0, 0].float().cpu().numpy()
                    ink_np = ink[0].float().permute(1, 2, 0).cpu().numpy()
                    alpha_acc[y:y + tile, x:x + tile] += a_np * ramp2d
                    ink_acc[y:y + tile, x:x + tile, :] += ink_np * ramp2d[..., None]
                    weight_acc[y:y + tile, x:x + tile] += ramp2d

        weight_safe = np.clip(weight_acc, 1e-6, None)
        alpha_full = (alpha_acc / weight_safe)[:h, :w]
        ink_full = (ink_acc / weight_safe[..., None])[:h, :w, :]

        obs_t = torch.from_numpy(img_f).permute(2, 0, 1).unsqueeze(0)
        alpha_t = torch.from_numpy(alpha_full).unsqueeze(0).unsqueeze(0)
        ink_t = torch.from_numpy(ink_full).permute(2, 0, 1).unsqueeze(0)
        recovered_t = recover(obs_t, alpha_t, ink_t, alpha_max)
        recovered = np.clip(np.round(recovered_t[0].permute(1, 2, 0).numpy() * 255.0), 0, 255).astype(np.uint8)
        return alpha_full.astype(np.float32), ink_full.astype(np.float32), recovered
    print('predict_page was not defined (10.9 was not run this session) -- defined it locally.')
else:
    print('Reusing predict_page from 10.9.')

assert SCRAPED_DIR is not None, 'Could not find images_scraped -- see 10.1.'
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
real_images = sorted(p for p in SCRAPED_DIR.rglob('*') if p.suffix.lower() in EXTS and p.is_file())
assert real_images, f'No images found under {SCRAPED_DIR}'

V2_REAL_OUT_DIR = pathlib.Path('/kaggle/working/alpha_net_v2_real')
V2_REAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

v2_panel_paths = []
for p in real_images[:6]:
    img = np.array(Image.open(p).convert('RGB'))
    t0 = time.time()
    alpha, ink, recovered = predict_page(v2_model, img, device)
    dt = time.time() - t0

    alpha_vis = (np.clip(alpha / max(float(ckpt['alpha_max']), 1e-6), 0, 1) * 255).astype(np.uint8)
    alpha_vis = np.stack([alpha_vis] * 3, axis=-1)
    panel = np.concatenate([img, alpha_vis, recovered], axis=1)
    panel_path = V2_REAL_OUT_DIR / f'{p.stem}_alpha_panel_v2.png'
    Image.fromarray(panel).save(panel_path)
    v2_panel_paths.append(panel_path)

    frac_marked = float((alpha > 0.05).mean()) * 100
    print(f'{p.name}: alpha_max={alpha.max():.3f} marked={frac_marked:.2f}% time={dt:.2f}s')

for path in v2_panel_paths[:3]:
    display(IPyImage(filename=str(path), width=1200))


### 11.5 Download the v2 model

To use this model in the app: place it at `weights/alpha_net_best_final.pt` (replacing v1 -- keep a copy of v1 first, e.g. `weights/alpha_net_best_final_v1.pt`), then run the repo benchmark, `python scripts/alpha_net/benchmark.py`, to compare v1 and v2.


In [ ]:
import pathlib
import shutil
import torch
from IPython.display import FileLink, display

assert 'V2_RUN_DIR' in globals(), 'V2_RUN_DIR is not defined -- run 11.3 first.'

v2_best_src = V2_RUN_DIR / 'best.pt'
v2_final_path = pathlib.Path('/kaggle/working/alpha_net_v2_best_final.pt')

if v2_best_src.exists():
    shutil.copy2(v2_best_src, v2_final_path)
    ckpt = torch.load(v2_final_path, map_location='cpu', weights_only=True)
    size_mb = v2_final_path.stat().st_size / 1e6
    print('epoch:', ckpt['epoch'])
    print('crisp val metrics:', ckpt['val_metrics']['crisp'])
    print('kaggle val metrics:', ckpt['val_metrics']['kaggle'])
    print(f'size: {size_mb:.2f} MB')
    display(FileLink(v2_final_path.name))
else:
    print(f'No best.pt found at {v2_best_src} -- run the training cell (11.3) first.')


## 12. Alpha network v3 (fixing the collapsed ink head)

**Diagnosis** (measured on v2, `weights/alpha_net_v2_best_final.pt`): v2's alpha prediction is excellent (96-101% of ground truth, up from v1's 63-79%), but its ink head collapsed to a constant mid-grey and never trained --

- predicted ink saturation on genuinely pink mark pixels: 0.009 vs. true 0.381; median predicted ink saturation across real pages ~= 0.025.
- Oracle ablation on `ariatender_wide`: with predicted ink, the PINK shield glyph is only 70.4% removed while the GREY lettering is 94.9% removed; substituting the TRUE ink (same predicted alpha) takes the shield to 95.8% -- ink accounts for 100% of that gap.
- `head.weight` mean-|w| for the ink channels was ~0.13 at v1 epoch 9 and STILL ~0.13 at v2 epoch 56, with ink bias ~= 0 -> `sigmoid(0) = 0.5`, i.e. the ink head never moved from its initialisation.

**Two causes**: (1) `alpha_loss`/`alpha_loss_v2` give the ink head no direct supervision -- it only leaks in via `L_rec`/`L_comp` at roughly 6x weaker gradient than a direct term would; (2) ~84% of training mark pixels are flat grey ink (`ariatender_stacked` is a flat-grey stamp; `ariatender_wide` is 68% grey lettering / 32% pink shield; the two calibrated stamps are drawn 50/50), so predicting constant grey was already close to optimal under the old loss.

**What changes here**:
- **12.1** -- `ColorAugmentedCrispCompositeDataset` (11.1's `CrispCompositeDataset` + a random ink re-tint) so a large, controlled fraction of training marks carry strongly saturated ink of arbitrary hue -- "always predict grey" stops being a winning strategy. `alpha_loss_v3` adds a direct, alpha-weighted `L_ink` term computed from an exact closed-form ink target (no dataset changes needed). `evaluate_v3` adds ink/colour metrics (`ink_sat_pred_wm` vs `ink_sat_true_wm` alone expose the collapse) and folds a coloured-subset recovery error into `val_score`, so checkpoint selection is sensitive to this specific failure mode -- **v3's `val_score` is NOT comparable to v2's**. 12.1 also adds checkpoint plumbing (`locate_alpha_checkpoint`, `save_full_state_checkpoint`) since `/kaggle/working` is wiped between Kaggle sessions but the Kaggle dataset the user uploads to is not -- see 12.2.
- **12.2** -- trains v3 with the same dual-GPU `DataParallel` + EMA structure as 11.3, mixing the colour-augmented crisp stream with 11.2's `ScaledAlphaCropDataset` Kaggle stream. Unlike 11.3, 12.2 can **resume exactly across Kaggle sessions**: its periodic snapshot (every 10 epochs, still a `FileLink` download) is now a FULL-STATE checkpoint (model + EMA + optimizer + LR-scheduler + AMP scaler + early-stopping bookkeeping), not just weights -- download it, re-upload it into the Kaggle dataset, and a later session's 12.2 finds and resumes from it via `locate_alpha_checkpoint`, continuing training exactly rather than from scratch. `EPOCHS` is always a TOTAL target epoch count, not a number of additional epochs. If only a model-only checkpoint is available (e.g. the `alpha_net_v2_best_epoch_60.pt` / `alpha_net_best_epoch_10.pt` files already in the dataset, both written before this change, or 12.4's own slim final download), 12.2 still resumes, but APPROXIMATELY, and says so loudly -- see 12.2's markdown for exactly what that means.
- **12.3** -- like 11.4, plus a panel that isolates a strongly coloured mark (`ariatender_wide`'s pink shield, zoomed) so the colour fix is visible at a glance, not just in aggregate metrics. Loads its checkpoint through the same locator as 12.2, so it also works in a fresh session where `/kaggle/working` is empty but the dataset has a v3 snapshot.
- **12.4** -- final, slim, model-only download for the app -- unaffected by 12.2's full-state snapshots.

**Run order**: 12.1 -> 12.2 -> 12.3 -> 12.4, and (like section 11) this depends on 10.1-10.6 and 11.1-11.2 having already run in this session (12.1 reuses `CrispCompositeDataset`, `ScaledAlphaCropDataset`, `AlphaUNet`, `recover`, `save_checkpoint`, `evaluate`, `ALPHA_MAX`, `WIDTHS`, `FORMAT_VERSION` rather than redefining them). Run 10.1 through 10.6, then 11.1 and 11.2, then 12.1 -> 12.2 -> 12.3 -> 12.4 (11.3-11.5, v2's own training/example/download cells, are not required for section 12).

### 12.1 Colour-augmented crisp composites, the v3 loss/metrics, and checkpoint plumbing

`ColorAugmentedCrispCompositeDataset` subclasses 11.1's `CrispCompositeDataset` and overrides ONLY `_sample_mark`, re-tinting the sampled stamp's ink in HSV space (random hue over the full circle, then either forced-up or freely-scaled saturation, plus mild value jitter) before `__getitem__` composites `obs` from it -- `alpha`/`clean` are read directly off the background and the mark's own coverage and never touch the tint, so they stay exact (re-verified below, same invariant 11.1 already checks). `alpha_loss_v3` adds a direct `L_ink` term (`LAMBDAS = (5.0, 2.0, 0.5, 3.0)`) using an ink target derived in closed form from tensors the dataset already returns -- no 5th tensor needed. `evaluate_v3` extends 10.6's `evaluate` with ink/colour metrics and folds a coloured-subset recovery error into `val_score` (NOT comparable to v2's). The cell ends with a self-check: the `alpha==0 => obs==clean` invariant, the sampled ink-saturation distribution, a loss/gradient sanity check on a real batch, and a full-state-checkpoint round trip through `weights_only=True` loading.

The last part of this cell is checkpoint plumbing, not modelling: `locate_alpha_checkpoint` searches `/kaggle/working`, then the Kaggle dataset root (both mount spellings), then all of `/kaggle/input`, for a checkpoint matching a filename pattern -- because `/kaggle/working` is wiped between Kaggle sessions but the dataset the user uploads to is not. `save_full_state_checkpoint` writes a checkpoint carrying everything needed to resume training exactly (model, EMA -- including its `n_averaged` counter, otherwise the very first post-resume EMA update silently discards the restored EMA weights, see the code comment -- optimizer, LR-scheduler, AMP scaler, early-stopping bookkeeping) -- and, because every one of those `state_dict()`s turns out to be plain tensors/primitives/lists/dicts (verified below by actually round-tripping one through `weights_only=True`), the same file still loads through `watermark_remover.alpha_net.load_model`, which simply ignores the extra keys it doesn't recognise. v3 never warm-starts or seeds from v1/v2 -- the locator also reports what it finds for those two, purely informationally, since the user's dataset carries them today; v3 only ever resumes ITS OWN prior progress (12.2).

In [ ]:
import math
import pathlib
import re
import cv2
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

for _name in ('AlphaUNet', 'recover', 'save_checkpoint', 'evaluate', 'dilate_bool_mask',
              'CrispCompositeDataset', 'ScaledAlphaCropDataset', 'calibrated_stamps',
              'VAL_BACKGROUNDS', 'WIDTHS', 'ALPHA_MAX', 'FORMAT_VERSION'):
    assert _name in globals(), (
        f'{_name!r} is not defined -- section 12 depends on 10.1-10.6 and 11.1-11.2 having '
        f'already run in this session (this notebook is self-contained per cell, but 12.x reuses '
        f'10.x/11.x definitions rather than redefining them). Run 10.1 through 10.6, then 11.1 and '
        f'11.2, first.'
    )

# =============================================================================
# 12.1a: ink-colour augmentation -- re-tint the sampled stamp's ink so a
# large, controlled fraction of training marks carry strongly saturated ink
# of arbitrary hue ("always predict grey" stops being a winning strategy).
# Overrides ONLY CrispCompositeDataset._sample_mark; obs is recomposited from
# whatever ink _sample_mark returns, so alpha/clean (read directly off the
# background) stay exact regardless of the tint.
# =============================================================================
P_KEEP_REAL_INK = 0.4   # probability of keeping the stamp's real calibrated colour untouched

def _tint_ink(ink_rgb, rng):
    """Re-tints an (H,W,3) float32 [0,1] ink image in HSV space: a random hue
    shift over the full circle, then EITHER forces saturation up into a
    strongly-coloured band OR scales the existing saturation freely (can
    stay muted or even grey), plus a mild value jitter. Applied uniformly
    across the whole array, so per-pixel STRUCTURE -- e.g. ariatender_wide's
    two-tone pink-shield / grey-lettering split -- is preserved: only each
    pixel's own hue/saturation/value changes, never WHERE ink is placed."""
    hsv = cv2.cvtColor(np.clip(ink_rgb, 0.0, 1.0).astype(np.float32), cv2.COLOR_RGB2HSV)
    hue_shift = float(rng.uniform(0.0, 360.0))
    hsv[..., 0] = (hsv[..., 0] + hue_shift) % 360.0
    if rng.random() < 0.5:
        # "coloured mark" regime: force saturation up regardless of the source.
        target_sat = float(rng.uniform(0.45, 0.95))
        hsv[..., 1] = np.clip(0.2 * hsv[..., 1] + target_sat, 0.0, 1.0)
    else:
        # free scaling: can amplify OR mute what was already there.
        sat_scale = float(rng.uniform(0.0, 2.0))
        hsv[..., 1] = np.clip(hsv[..., 1] * sat_scale, 0.0, 1.0)
    val_jitter = float(rng.uniform(0.85, 1.15))
    hsv[..., 2] = np.clip(hsv[..., 2] * val_jitter, 0.0, 1.0)
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    return np.clip(rgb, 0.0, 1.0).astype(np.float32)

class ColorAugmentedCrispCompositeDataset(CrispCompositeDataset):
    """11.1's CrispCompositeDataset, overriding ONLY _sample_mark: with
    probability P_KEEP_REAL_INK keeps the stamp's real calibrated colour,
    otherwise re-tints it with _tint_ink. Nothing else about placement,
    scale, JPEG re-encoding, or weight computation changes, so every rule
    11.1 already applies is inherited as-is."""

    def _sample_mark(self, rng):
        ink, alpha_full, opacity_mult, scale_range = super()._sample_mark(rng)
        if rng.random() >= P_KEEP_REAL_INK:
            ink = _tint_ink(ink, rng)
        return ink, alpha_full, opacity_mult, scale_range

print('ColorAugmentedCrispCompositeDataset defined (subclasses CrispCompositeDataset, '
      'overrides _sample_mark only).')

# =============================================================================
# 12.1b: the v3 loss -- adds a direct, alpha-weighted ink term. v1/v2's
# alpha_loss(_v2) gave the ink head NO direct supervision (only indirect
# leakage via L_rec/L_comp, ~6x weaker gradient) -- this is the fix.
# =============================================================================
INK_MIN_ALPHA = 0.10   # below this alpha, obs=a*ink+(1-a)*clean is too ill-conditioned to invert for ink
LAMBDAS = (5.0, 2.0, 0.5, 1.0)   # alpha, rec, comp, ink -- ink was 3.0 in the first v3 run, where
                                  # the ink term became over half the total loss and the alpha head
                                  # (already good in v2) got worse; 1.0 still revives the ink head.
# Tag for the val_score definition below. A full-state resume keeps its stored best score only
# if it was computed with the same definition (see 12.2) -- scores from different definitions or
# val sets are not comparable.
SCORE_VERSION = 'v3.1'

def alpha_loss_v3(pred_alpha, pred_ink, obs, clean, alpha, w_alpha, w_rec, lambdas=LAMBDAS):
    """10.6's alpha_loss (l_alpha/l_rec/l_comp, unchanged) + a direct L_ink
    term. ink_gt is the EXACT closed-form ink implied by
    obs = a*ink + (1-a)*clean, i.e. ink = (obs - (1-a)*clean) / a --
    derivable from tensors the dataset already returns, no 5th tensor
    needed (verified numerically in 12.1e: 0.0000 MAE without JPEG, ~5.7/255
    under JPEG q85 at a>=INK_MIN_ALPHA, which covers ~92% of mark pixels).
    ink_w zeroes out L_ink below INK_MIN_ALPHA (where the division is
    ill-conditioned) and otherwise weights by alpha itself. Normalised by
    the WEIGHT SUM, not the image -- normalising over all pixels is exactly
    what diluted the old indirect ink signal to nothing. An all-zero mask
    gives L_ink=0, not NaN (numerator is 0 too; denominator is clamped>=1).
    """
    la, lr, lc, li = lambdas
    rec = recover(obs, pred_alpha, pred_ink, ALPHA_MAX)
    l_alpha = (w_alpha * (pred_alpha - alpha).abs()).mean()
    l_rec = (w_rec * (rec - clean).abs().mean(dim=1, keepdim=True)).mean()
    comp = pred_alpha * pred_ink + (1.0 - pred_alpha) * clean
    l_comp = (w_rec * (comp - obs).abs().mean(dim=1, keepdim=True)).mean()

    ink_w = torch.where(alpha >= INK_MIN_ALPHA, alpha, torch.zeros_like(alpha))
    ink_gt = torch.clamp((obs - (1.0 - alpha) * clean) / alpha.clamp(min=INK_MIN_ALPHA), 0.0, 1.0)
    l_ink = (ink_w * (pred_ink - ink_gt).abs()).sum() / torch.clamp(ink_w.sum() * pred_ink.shape[1], min=1.0)

    total = la * l_alpha + lr * l_rec + lc * l_comp + li * l_ink
    parts = {'L_alpha': float(l_alpha.item()), 'L_rec': float(l_rec.item()),
             'L_comp': float(l_comp.item()), 'L_ink': float(l_ink.item())}
    return total, parts

print(f'alpha_loss_v3 defined. LAMBDAS (alpha, rec, comp, ink) = {LAMBDAS}, INK_MIN_ALPHA = {INK_MIN_ALPHA}')

# =============================================================================
# 12.1c: evaluate_v3 -- 10.6's evaluate (alpha/reconstruction metrics,
# unchanged logic) + ink/colour metrics. ink_sat_pred_wm vs ink_sat_true_wm
# alone expose the v2 grey-ink collapse (v2 reads ~0.03 vs ~0.38). The
# coloured-subset group (true ink saturation > colored_sat_thresh) is folded
# into val_score so checkpoint selection is sensitive to this failure mode
# -- NOT comparable to v2's val_score (different score definition).
# =============================================================================
def rgb_saturation(rgb):
    """(N,3,H,W) float tensor in [0,1] -> (N,1,H,W) HSV saturation =
    (max-min)/max, 0 where max==0 (pure black has no defined hue/sat)."""
    mx, _ = rgb.max(dim=1, keepdim=True)
    mn, _ = rgb.min(dim=1, keepdim=True)
    return torch.where(mx > 1e-6, (mx - mn) / mx.clamp(min=1e-6), torch.zeros_like(mx))

def evaluate_v3(model, loader, device, ink_min_alpha=INK_MIN_ALPHA, colored_sat_thresh=0.06):
    model.eval()
    sums = dict(alpha_err_wm=0.0, n_wm=0, alpha_bg=0.0, n_bg=0,
                rec_err_wm=0.0, id_err_wm=0.0, rec_err_bg=0.0, id_err_bg=0.0,
                n_excluded=0, n_total=0,
                ink_err_wm=0.0, n_ink_wm=0, sat_pred_wm=0.0, sat_true_wm=0.0,
                rec_err_colored=0.0, id_err_colored=0.0, n_colored=0)
    with torch.no_grad():
        for obs, clean, alpha, _w_alpha, _w_rec in loader:
            obs, clean, alpha = obs.to(device), clean.to(device), alpha.to(device)
            pred_alpha, pred_ink = model(obs)
            rec = recover(obs, pred_alpha, pred_ink, ALPHA_MAX)

            recoverable = alpha <= ALPHA_MAX
            wm_mask = (alpha > 0.01) & recoverable
            bg_mask = (~(alpha > 0.01)) & recoverable

            alpha_err = (pred_alpha - alpha).abs()
            rec_err = (rec - clean).abs().mean(dim=1, keepdim=True) * 255.0
            id_err = (obs - clean).abs().mean(dim=1, keepdim=True) * 255.0

            n_wm, n_bg = int(wm_mask.sum().item()), int(bg_mask.sum().item())
            if n_wm:
                sums['alpha_err_wm'] += float(alpha_err[wm_mask].sum().item())
                sums['rec_err_wm'] += float(rec_err[wm_mask].sum().item())
                sums['id_err_wm'] += float(id_err[wm_mask].sum().item())
                sums['n_wm'] += n_wm
            if n_bg:
                sums['alpha_bg'] += float(pred_alpha[bg_mask].sum().item())
                sums['rec_err_bg'] += float(rec_err[bg_mask].sum().item())
                sums['id_err_bg'] += float(id_err[bg_mask].sum().item())
                sums['n_bg'] += n_bg
            sums['n_excluded'] += int((~recoverable).sum().item())
            sums['n_total'] += int(alpha.numel())

            # --- v3 ink metrics: only where alpha is large enough that ink_gt
            # is well-conditioned (matches alpha_loss_v3's ink_w mask) ---
            ink_mask = (alpha >= ink_min_alpha) & recoverable
            n_ink = int(ink_mask.sum().item())
            if n_ink:
                ink_gt = torch.clamp((obs - (1.0 - alpha) * clean) / alpha.clamp(min=ink_min_alpha), 0.0, 1.0)
                ink_err = (pred_ink - ink_gt).abs().mean(dim=1, keepdim=True)
                sums['ink_err_wm'] += float(ink_err[ink_mask].sum().item())
                sums['n_ink_wm'] += n_ink

                sat_pred = rgb_saturation(pred_ink)
                sat_true = rgb_saturation(ink_gt)
                sums['sat_pred_wm'] += float(sat_pred[ink_mask].sum().item())
                sums['sat_true_wm'] += float(sat_true[ink_mask].sum().item())

                colored_mask = ink_mask & (sat_true > colored_sat_thresh)
                n_colored = int(colored_mask.sum().item())
                if n_colored:
                    sums['rec_err_colored'] += float(rec_err[colored_mask].sum().item())
                    sums['id_err_colored'] += float(id_err[colored_mask].sum().item())
                    sums['n_colored'] += n_colored

    n_wm, n_bg = max(sums['n_wm'], 1), max(sums['n_bg'], 1)
    n_ink_wm = max(sums['n_ink_wm'], 1)
    n_colored = max(sums['n_colored'], 1)

    alpha_mae_wm = sums['alpha_err_wm'] / n_wm
    alpha_mean_bg = sums['alpha_bg'] / n_bg
    rec_mae_wm_255 = sums['rec_err_wm'] / n_wm
    identity_mae_wm_255 = sums['id_err_wm'] / n_wm
    rec_mae_bg_255 = sums['rec_err_bg'] / n_bg
    identity_mae_bg_255 = sums['id_err_bg'] / n_bg
    frac_alpha_gt_max = sums['n_excluded'] / max(sums['n_total'], 1)
    improvement_wm = (1.0 - rec_mae_wm_255 / identity_mae_wm_255) if identity_mae_wm_255 > 1e-8 else 0.0

    ink_mae_wm = sums['ink_err_wm'] / n_ink_wm
    ink_sat_pred_wm = sums['sat_pred_wm'] / n_ink_wm
    ink_sat_true_wm = sums['sat_true_wm'] / n_ink_wm
    rec_mae_colored_255 = sums['rec_err_colored'] / n_colored
    identity_mae_colored_255 = sums['id_err_colored'] / n_colored
    improvement_wm_colored = (1.0 - rec_mae_colored_255 / identity_mae_colored_255) if identity_mae_colored_255 > 1e-8 else 0.0
    frac_colored = sums['n_colored'] / max(sums['n_ink_wm'], 1)

    # v2's score (rec_wm + background-damage penalty) PLUS the coloured-subset
    # recovery error, so checkpoint selection is sensitive to the grey-ink
    # collapse -- NOT comparable to v2's val_score.
    val_score = (rec_mae_wm_255 + 2.0 * max(0.0, rec_mae_bg_255 - identity_mae_bg_255)
                 + rec_mae_colored_255)

    return {
        'alpha_mae_wm': alpha_mae_wm, 'alpha_mean_bg': alpha_mean_bg,
        'rec_mae_wm_255': rec_mae_wm_255, 'identity_mae_wm_255': identity_mae_wm_255,
        'improvement_wm': improvement_wm, 'rec_mae_bg_255': rec_mae_bg_255,
        'identity_mae_bg_255': identity_mae_bg_255, 'frac_alpha_gt_max': frac_alpha_gt_max,
        'ink_mae_wm': ink_mae_wm, 'ink_sat_pred_wm': ink_sat_pred_wm, 'ink_sat_true_wm': ink_sat_true_wm,
        'rec_mae_colored_255': rec_mae_colored_255, 'identity_mae_colored_255': identity_mae_colored_255,
        'improvement_wm_colored': improvement_wm_colored, 'frac_colored': frac_colored,
        'val_score': val_score,
    }

print('evaluate_v3 defined (10.6\'s evaluate + ink_mae_wm / ink_sat_pred_wm / ink_sat_true_wm / '
      'coloured-subset recovery error, folded into val_score).')

# =============================================================================
# 12.1d: checkpoint plumbing -- NOT modelling, just where checkpoints live.
# /kaggle/working is wiped between Kaggle sessions; the Kaggle dataset the
# user uploads to is not. locate_alpha_checkpoint finds an alpha_net
# checkpoint wherever it currently lives (this session's own output, a
# checkpoint carried over in the dataset, or anywhere else under
# /kaggle/input); save_full_state_checkpoint writes one that a later
# session's 12.2 can resume EXACTLY from. Used by 12.2 (resume) and 12.3
# (loading a checkpoint to run examples from, in a fresh session).
# =============================================================================
DATASET_ROOT_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2'),
    pathlib.Path('/kaggle/input/train-and-test2'),
]

def _epoch_of(p):
    m = re.search(r'epoch_(\d+)', p.stem)
    return int(m.group(1)) if m else -1

def locate_alpha_checkpoint(patterns, working_last_path=None, explicit_path=None):
    """General alpha_net checkpoint locator. Search order, first hit wins:
      1. explicit_path, if given (must exist -- an explicit override is
         never silently ignored).
      2. working_last_path, if given and it exists (this session's own
         in-progress run -- exact, no epoch-parsing needed).
      3. /kaggle/working, searched recursively for `patterns`, highest
         parsed '..._epoch_<N>...' wins (ties/no-epoch-in-name -> -1).
      4. the dataset root, both mount spellings (DATASET_ROOT_CANDIDATES),
         same highest-epoch-wins rule.
      5. a final rglob(pattern) fallback across all of /kaggle/input,
         mirroring 11.1's STAMPS_CANDIDATES fallback for stamps/.
    `patterns` is one glob pattern or a list tried together at each
    location. NEVER raises for "nothing found" (returns None) -- that just
    means train from scratch / no cross-session checkpoint is available yet.
    """
    patterns = [patterns] if isinstance(patterns, str) else list(patterns)

    if explicit_path is not None:
        p = pathlib.Path(explicit_path)
        assert p.exists(), f'Explicit checkpoint path {explicit_path!r} does not exist.'
        return p
    if working_last_path is not None and pathlib.Path(working_last_path).exists():
        return pathlib.Path(working_last_path)

    def _best(paths):
        paths = sorted(set(paths), key=_epoch_of, reverse=True)
        return paths[0] if paths else None

    working_dir = pathlib.Path('/kaggle/working')
    if working_dir.is_dir():
        found = _best(p for pat in patterns for p in working_dir.rglob(pat))
        if found is not None:
            return found

    for root in DATASET_ROOT_CANDIDATES:
        if root.exists():
            found = _best(p for pat in patterns for p in root.glob(pat))
            if found is not None:
                return found

    return _best(p for pat in patterns for p in pathlib.Path('/kaggle/input').rglob(pat))

def strip_module_prefix(state_dict):
    """Strips a leading 'module.' from every key, if present -- tolerates a
    state dict written by a torch.nn.DataParallel-wrapped model."""
    if any(k.startswith('module.') for k in state_dict):
        return {(k[len('module.'):] if k.startswith('module.') else k): v for k, v in state_dict.items()}
    return state_dict

def classify_checkpoint_tier(ckpt):
    """'full' if the checkpoint carries optimizer/scheduler/scaler/EMA state
    (exact resume is possible), else 'model_only' (weights + epoch only --
    e.g. any checkpoint written by save_checkpoint/10.5, including
    alpha_net_v2_best_epoch_60.pt / alpha_net_best_epoch_10.pt in the
    dataset today -- resume from these is only approximate, see 12.2)."""
    return 'full' if all(k in ckpt for k in ('optimizer', 'scheduler', 'scaler', 'ema_state')) else 'model_only'

def save_full_state_checkpoint(path, model, ema_model, opt, scheduler, scaler, epoch,
                                val_metrics, config, best_score, epochs_no_improve):
    """Everything save_checkpoint's model-only dict has (10.5), PLUS
    optimizer/scheduler/scaler/EMA state and early-stopping bookkeeping --
    enough to resume training EXACTLY, even from a NEW Kaggle session after
    /kaggle/working has been wiped (a model-only dict has no optimizer
    momentum and no LR-schedule position to restore). 'ema_n_averaged' is
    saved explicitly and separately from 'ema_state' (which is only
    ema_model.module's plain, unprefixed weights) because
    AveragedModel.update_parameters special-cases n_averaged==0 to mean
    "just copy the raw model in, discarding whatever EMA state is there" --
    without restoring it, the very first post-resume EMA update would
    silently overwrite the just-restored EMA weights with the plain model's.
    Every value here is a plain tensor/primitive/list/dict (verified in
    12.1e), so the result still loads through
    watermark_remover.alpha_net.load_model (weights_only=True, which simply
    ignores keys it doesn't recognise) -- one file serves both purposes.
    """
    ckpt = {
        'format_version': FORMAT_VERSION,
        'arch': 'AlphaUNet',
        'widths': list(model.widths),
        'alpha_max': float(model.alpha_max),
        # 'model_state' is what watermark_remover.alpha_net.load_model (the app)
        # reads, so it holds the EMA weights -- the ones that get evaluated and
        # selected. The raw training weights, needed only to resume, are kept
        # separately under 'raw_model_state'.
        'model_state': ema_model.module.state_dict(),
        'raw_model_state': model.state_dict(),
        'epoch': epoch,
        'val_metrics': val_metrics,
        'train_config': config,
        'ema_state': ema_model.module.state_dict(),
        'ema_n_averaged': int(ema_model.n_averaged.item()),
        'optimizer': opt.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_score': best_score,
        'epochs_no_improve': epochs_no_improve,
    }
    torch.save(ckpt, path)
    return ckpt

# Informational only -- v3 does NOT warm-start or resume from v1/v2 checkpoints,
# it only ever resumes its OWN prior progress (12.2). This just demonstrates
# the locator against files the user may already have uploaded and confirms
# the dataset is reachable under this session's mount.
_V1_CHECKPOINT_PATH = locate_alpha_checkpoint(['alpha_net_best_final.pt', 'alpha_net_best_epoch_*.pt'])
_V2_CHECKPOINT_PATH = locate_alpha_checkpoint(['alpha_net_v2_best_final.pt', 'alpha_net_v2_best_epoch_*.pt'])
print('V1_CHECKPOINT_PATH ->', _V1_CHECKPOINT_PATH if _V1_CHECKPOINT_PATH else '(none found)')
print('V2_CHECKPOINT_PATH ->', _V2_CHECKPOINT_PATH if _V2_CHECKPOINT_PATH else '(none found)')
print('(informational only -- v3 always trains its own weights; see 12.2 for how IT resumes.)')

# =============================================================================
# 12.1e: self-check -- physics invariant, colour distribution, loss sanity,
# and a plain-dict round-trip proving 12.1d's full-state checkpoint stays
# loadable through weights_only=True.
# =============================================================================
_CROP_CHECK = 512
_check_ds = ColorAugmentedCrispCompositeDataset(_CROP_CHECK, samples_per_epoch=40, deterministic=False, seed=777)
_bad_zero = 0
_sat_true_chunks = []
for _i in range(len(_check_ds)):
    _obs, _clean, _alpha, _wa, _wr = _check_ds.__getitem__(_i, _force_no_jpeg=True)
    _obs_np, _clean_np, _alpha_np = _obs.numpy(), _clean.numpy(), _alpha.numpy()[0]
    _zero_mask = (_alpha_np <= 1e-8)
    _diff = np.abs(_obs_np - _clean_np)[:, _zero_mask] if _zero_mask.any() else np.zeros((3, 0))
    if _diff.size and float(_diff.max()) > 1e-6:
        _bad_zero += 1
    _mask010 = _alpha_np >= INK_MIN_ALPHA
    if _mask010.any():
        _a = _alpha_np[None, ...]
        _ink_gt_np = np.clip((_obs_np - (1.0 - _a) * _clean_np) / np.clip(_a, INK_MIN_ALPHA, None), 0.0, 1.0)
        _ink_rgb = _ink_gt_np.transpose(1, 2, 0)[_mask010]
        _mx, _mn = _ink_rgb.max(axis=1), _ink_rgb.min(axis=1)
        _sat_true_chunks.append(np.where(_mx > 1e-6, (_mx - _mn) / np.clip(_mx, 1e-6, None), 0.0))

print(f'[physics check] {_bad_zero}/{len(_check_ds)} samples had obs != clean where alpha==0 (should be 0, no JPEG)')
assert _bad_zero == 0, 'obs must equal clean exactly where alpha==0 (before JPEG) -- colour augmentation must not violate this invariant.'

_sat_concat = np.concatenate(_sat_true_chunks) if _sat_true_chunks else np.zeros(0)
print(f'[colour-augmentation check] {len(_sat_concat)} mark pixel(s) sampled (alpha>={INK_MIN_ALPHA}) across {len(_check_ds)} crops')
if len(_sat_concat):
    print(f'  true ink saturation: mean={_sat_concat.mean():.3f} median={float(np.median(_sat_concat)):.3f} '
          f'p90={float(np.percentile(_sat_concat, 90)):.3f} frac>0.06={float((_sat_concat > 0.06).mean()):.3f}')

_loader_check = torch.utils.data.DataLoader(_check_ds, batch_size=4, shuffle=False, num_workers=0)
_obs_b, _clean_b, _alpha_b, _wa_b, _wr_b = next(iter(_loader_check))
_probe_model = AlphaUNet(widths=WIDTHS, alpha_max=ALPHA_MAX)
with torch.no_grad():
    _pa, _pi = _probe_model(_obs_b)
_loss_val, _loss_parts = alpha_loss_v3(_pa, _pi, _obs_b, _clean_b, _alpha_b, _wa_b, _wr_b)
print('[loss check] alpha_loss_v3 parts:', _loss_parts, 'total=', float(_loss_val.item()))
assert math.isfinite(float(_loss_val.item())) and _loss_parts['L_ink'] >= 0.0

# Full-state round trip: a fresh optimizer/scheduler/scaler/EMA over the probe
# model, saved via save_full_state_checkpoint, then loaded back with
# weights_only=True (what watermark_remover.alpha_net.load_model uses).
import tempfile
_probe_ema = torch.optim.swa_utils.AveragedModel(
    _probe_model, avg_fn=lambda a, p, n: 0.999 * a + 0.001 * p)
_probe_opt = torch.optim.AdamW(_probe_model.parameters(), lr=1e-3)
(_pa2, _pi2) = _probe_model(_obs_b)
_probe_loss, _ = alpha_loss_v3(_pa2, _pi2, _obs_b, _clean_b, _alpha_b, _wa_b, _wr_b)
_probe_loss.backward()
_probe_opt.step()
_probe_sched = torch.optim.lr_scheduler.LambdaLR(_probe_opt, lambda it: 1.0)
_probe_scaler = torch.amp.GradScaler('cpu', enabled=False)
_probe_ema.update_parameters(_probe_model)
_tmp_ckpt_path = pathlib.Path(tempfile.gettempdir()) / '_alpha_v3_selfcheck_full_state.pt'
save_full_state_checkpoint(_tmp_ckpt_path, _probe_model, _probe_ema, _probe_opt, _probe_sched,
                            _probe_scaler, epoch=1, val_metrics={'val_score': 0.0}, config={},
                            best_score=0.0, epochs_no_improve=0)
_reloaded = torch.load(_tmp_ckpt_path, map_location='cpu', weights_only=True)
print(f'[full-state round trip] saved+reloaded with weights_only=True OK, keys={sorted(_reloaded.keys())}')
assert classify_checkpoint_tier(_reloaded) == 'full'
_tmp_ckpt_path.unlink(missing_ok=True)
del _probe_model, _probe_ema, _probe_opt, _probe_sched, _probe_scaler

# --- small preview of colour-augmented composites -----------------------------
_preview_ds = ColorAugmentedCrispCompositeDataset(_CROP_CHECK, samples_per_epoch=4, deterministic=True, seed=0)
fig, axes = plt.subplots(4, 3, figsize=(9, 12))
for row in range(4):
    obs, clean, alpha, _wa, _wr = _preview_ds[row]
    axes[row, 0].imshow(np.clip(obs.permute(1, 2, 0).numpy(), 0, 1)); axes[row, 0].set_title('observed (colour-augmented)')
    axes[row, 1].imshow(alpha[0].numpy(), cmap='inferno', vmin=0, vmax=ALPHA_MAX); axes[row, 1].set_title('alpha')
    axes[row, 2].imshow(np.clip(clean.permute(1, 2, 0).numpy(), 0, 1)); axes[row, 2].set_title('clean')
    for c in range(3):
        axes[row, c].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/alpha_v3_colour_preview.png', dpi=110)
plt.show()
print('Saved preview to /kaggle/working/alpha_v3_colour_preview.png')

### 12.2 Train v3 (resume-capable, both GPUs, live TensorBoard)

Same dual-GPU/EMA structure as 11.3: `torch.nn.DataParallel` when `N_GPUS >= 2`, an EMA shadow model that is what gets evaluated and saved, mixing `ColorAugmentedCrispCompositeDataset` (12.1) with 11.2's `ScaledAlphaCropDataset`, dual validation with `evaluate_v3` (crisp selects the checkpoint via `val_score`, Kaggle logged alongside it). TensorBoard shows train vs. validation loss and "accuracy" (fraction of the watermark removed) live; if the inline panel stays blank on Kaggle, the logs are still in `runs/alpha_net_v3/tensorboard`.

**`EPOCHS` is a TOTAL target epoch count, not additional epochs** -- raise it and re-run the cell to keep training; if the resumed checkpoint is already at or past `EPOCHS`, the cell says so and exits.

**Resume.** Auto-resume only looks at v3's own files: this session's `last.pt`, else the highest-epoch `alpha_net_v3_resume_epoch_*.pt` in the dataset or under `/kaggle/input` (`/kaggle/working` is wiped between sessions, so download a resume file, add it to the dataset, and a later session continues from it). Two tiers, printed loudly:
- **FULL-STATE** (a v3 resume file): raw weights, EMA, optimizer, LR scheduler and AMP scaler are restored -- an exact continuation. Its stored best score is carried over only if it was computed with the same score definition (`SCORE_VERSION`); otherwise the best score starts fresh.
- **MODEL-ONLY** (a slim checkpoint, e.g. v2's `alpha_net_v2_best_epoch_60.pt`, set via `RESUME_FROM`): weights and epoch are restored, then training runs as a **fine-tune**: peak LR `FINETUNE_LR` (2e-4), 1-epoch warmup, cosine over the remaining epochs, fresh optimizer/EMA, and a fresh best score.

What went wrong in the first v3 run, and what changed:
1. The best score was seeded from v2's own val score (2.2 -- a different score definition on a different val set; v3 scores ~9.4). No epoch could ever "improve", so `best.pt` was never written and early stopping fired after 25 epochs. -> The best score now starts fresh unless the definitions match.
2. The model-only path restarted the from-scratch schedule, so the converged v2 weights got peak LR 1e-3 with a fresh optimizer; alpha accuracy regressed (crisp alpha MAE 0.023 -> 0.036). -> Fine-tune schedule above.
3. `LAMBDAS` had ink = 3.0, making the ink term over half the loss. -> 1.0 (12.1).
4. The every-10-epoch file was a full-state copy of the LATEST epoch (named `..._best_epoch_N`), whose `model_state` held the RAW weights -- which is what the app loads. -> Every 10 epochs you now get two files: **`alpha_net_v3_best_upto_epoch_N.pt`** (slim, the best EMA model so far, for the app) and **`alpha_net_v3_resume_epoch_N.pt`** (full state for resuming, ~50 MB, whose `model_state` is also the EMA weights so it is safe in the app too).


In [ ]:
import csv
import math
import pathlib
import random
import shutil
import time
import torch
from IPython.display import FileLink, display
from torch.utils.tensorboard import SummaryWriter

for _name in ('AlphaUNet', 'recover', 'save_checkpoint', 'ScaledAlphaCropDataset',
              'ColorAugmentedCrispCompositeDataset', 'alpha_loss_v3', 'evaluate_v3',
              'locate_alpha_checkpoint', 'strip_module_prefix', 'classify_checkpoint_tier',
              'save_full_state_checkpoint', 'DATASET_ROOT_CANDIDATES', 'SCORE_VERSION',
              'WIDTHS', 'ALPHA_MAX', 'FORMAT_VERSION', 'LAMBDAS', 'INK_MIN_ALPHA'):
    assert _name in globals(), f'{_name!r} is not defined -- run 10.1-10.6, 11.1-11.2, and 12.1 first.'

EPOCHS = 60                   # TOTAL target epoch count, NOT additional epochs -- resuming an
                               # epoch-60 checkpoint with EPOCHS=80 trains 20 MORE and stops; if the
                               # resumed checkpoint is already >= EPOCHS, this cell says so and exits.
BATCH_PER_GPU = 8
CROP = 512
SAMPLES_PER_EPOCH = 4000
P_CRISP = 0.5                 # fraction of each training batch-stream drawn from the colour-augmented crisp set
LR = 1e-3                     # from-scratch peak LR
FINETUNE_LR = 2e-4            # peak LR when continuing from a MODEL-ONLY checkpoint (see "Resume" below)
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
PATIENCE = 25                 # early-stop after this many epochs with no crisp-val improvement
EMA_DECAY = 0.999
SEED = 0
SNAPSHOT_EVERY = 10           # every N epochs: a slim BEST model + a full-state RESUME file (see markdown)
CRISP_VAL_SIZE = 300
KAGGLE_VAL_SIZE = 300
TB_LOG_EVERY = 20             # log per-batch train loss to TensorBoard every N steps
RESUME_FROM = None            # explicit override path; None = auto-select (this session's own last.pt
                               # first, else the highest-epoch alpha_net_v3_resume_epoch_*.pt found in the
                               # dataset / /kaggle/input). Auto-select never picks up v1/v2 files. To
                               # continue from v2's model-only file deliberately, set e.g.:
                               #   RESUME_FROM = '/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/alpha_net_v2_best_epoch_60.pt'
                               # -> MODEL-ONLY tier: weights + epoch restored, then a FINE-TUNE
                               # schedule (FINETUNE_LR, 1-epoch warmup, cosine over the remaining
                               # epochs) and a fresh best score. Raise EPOCHS above the file's epoch.

torch.manual_seed(SEED)
random.seed(SEED)

N_GPUS = torch.cuda.device_count()
device = 'cuda' if N_GPUS > 0 else 'cpu'
if N_GPUS:
    print(f'GPUs ({N_GPUS}):', [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
else:
    print('No GPU visible -- training on CPU (slow; fine for a smoke test, not for a real run).')

V3_RUN_DIR = pathlib.Path('/kaggle/working/runs/alpha_net_v3')
V3_RUN_DIR.mkdir(parents=True, exist_ok=True)
V3_LAST_PATH = V3_RUN_DIR / 'last.pt'
V3_BEST_PATH = V3_RUN_DIR / 'best.pt'
V3_METRICS_PATH = V3_RUN_DIR / 'metrics.csv'
TB_DIR = V3_RUN_DIR / 'tensorboard'

# --- data ---------------------------------------------------------------------
GLOBAL_BATCH = BATCH_PER_GPU * max(1, N_GPUS)

crisp_train_ds = ColorAugmentedCrispCompositeDataset(CROP, samples_per_epoch=None, deterministic=False, seed=SEED)
kaggle_train_ds = ScaledAlphaCropDataset('train', CROP, samples_per_epoch=None, deterministic=False, seed=SEED)
crisp_val_ds = ColorAugmentedCrispCompositeDataset(CROP, samples_per_epoch=CRISP_VAL_SIZE, deterministic=True, seed=SEED)
kaggle_val_ds = ScaledAlphaCropDataset('val', CROP, samples_per_epoch=KAGGLE_VAL_SIZE, deterministic=True, seed=SEED)

class MixedTrainDataset(torch.utils.data.Dataset):
    """Each item independently comes from crisp_ds with probability p_crisp,
    else from kaggle_ds -- same mixing strategy as 11.3's MixedTrainDataset."""

    def __init__(self, crisp_ds, kaggle_ds, length, p_crisp):
        self.crisp_ds, self.kaggle_ds = crisp_ds, kaggle_ds
        self.length, self.p_crisp = int(length), float(p_crisp)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if random.random() < self.p_crisp:
            return self.crisp_ds[random.randrange(len(self.crisp_ds))]
        return self.kaggle_ds[random.randrange(len(self.kaggle_ds))]

train_ds = MixedTrainDataset(crisp_train_ds, kaggle_train_ds, SAMPLES_PER_EPOCH, P_CRISP)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=GLOBAL_BATCH, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(device == 'cuda'), drop_last=True, persistent_workers=(NUM_WORKERS > 0),
)
crisp_val_loader = torch.utils.data.DataLoader(
    crisp_val_ds, batch_size=GLOBAL_BATCH, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device == 'cuda'), persistent_workers=(NUM_WORKERS > 0),
)
kaggle_val_loader = torch.utils.data.DataLoader(
    kaggle_val_ds, batch_size=GLOBAL_BATCH, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device == 'cuda'), persistent_workers=(NUM_WORKERS > 0),
)
iters_per_epoch = len(train_loader)
assert iters_per_epoch > 0, 'Training loader produced zero batches -- reduce BATCH_PER_GPU or raise SAMPLES_PER_EPOCH.'

# --- model, DataParallel, EMA --------------------------------------------------
model = AlphaUNet(widths=WIDTHS, alpha_max=ALPHA_MAX).to(device)
USE_DP = (N_GPUS >= 2) and (device == 'cuda')
model_run = torch.nn.DataParallel(model) if USE_DP else model
print(f'DataParallel: {USE_DP} (N_GPUS={N_GPUS}) -- global batch = {GLOBAL_BATCH} ({BATCH_PER_GPU} x {max(1, N_GPUS)})')

def ema_avg_fn(averaged_param, param, num_averaged):
    return EMA_DECAY * averaged_param + (1.0 - EMA_DECAY) * param

ema_model = torch.optim.swa_utils.AveragedModel(model, avg_fn=ema_avg_fn)
use_amp = (device == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

def make_optimizer_and_schedule(peak_lr, n_epochs):
    """AdamW + 1-epoch linear warmup + cosine decay over `n_epochs` epochs.
    `n_epochs` is the length of THIS schedule: all EPOCHS for a fresh run,
    only the REMAINING epochs for a fine-tune continuation."""
    opt_ = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=WEIGHT_DECAY)
    warmup = iters_per_epoch
    total = max(warmup + 1, n_epochs * iters_per_epoch)

    def lr_lambda(it):
        if it < warmup:
            return (it + 1) / warmup
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, (it - warmup) / max(1, total - warmup))))

    return opt_, torch.optim.lr_scheduler.LambdaLR(opt_, lr_lambda)

opt, scheduler = make_optimizer_and_schedule(LR, EPOCHS)

# =============================================================================
# Resume -- two tiers, printed loudly so which one happened is never ambiguous.
#   FULL (a v3 full-state file): raw weights, EMA (+ n_averaged), optimizer,
#     scheduler and scaler are restored -> exact continuation. Its best score
#     is kept ONLY if it was computed with this cell's score definition
#     (SCORE_VERSION); otherwise it starts fresh.
#   MODEL_ONLY (any slim checkpoint, e.g. v2's): weights + epoch restored,
#     then a FINE-TUNE schedule over the remaining epochs, and a fresh best
#     score. Two earlier bugs this fixes: (1) seeding the best score from the
#     file's own val_score -- a different score definition on a different val
#     set (v2's 2.2 vs v3's ~9.4), so no epoch could ever "improve", best.pt
#     was never written, and early stopping fired; (2) restarting the
#     from-scratch schedule, i.e. hitting a converged model with peak LR.
# =============================================================================
start_epoch = 1
best_score = float('inf')
epochs_no_improve = 0
resume_tier = None

_resume_path = locate_alpha_checkpoint(
    ['alpha_net_v3_resume_epoch_*.pt', 'alpha_net_v3_best_epoch_*.pt'],
    working_last_path=V3_LAST_PATH, explicit_path=RESUME_FROM,
)

if _resume_path is not None:
    print(f'Resume candidate: {_resume_path}')
    _rckpt = torch.load(_resume_path, map_location=device, weights_only=True)
    assert tuple(_rckpt['widths']) == tuple(WIDTHS) and abs(float(_rckpt['alpha_max']) - ALPHA_MAX) < 1e-6, (
        f'{_resume_path} has widths={_rckpt["widths"]} alpha_max={_rckpt["alpha_max"]}, expected '
        f'widths={list(WIDTHS)} alpha_max={ALPHA_MAX} -- refusing to resume an incompatible architecture.'
    )
    start_epoch = int(_rckpt['epoch']) + 1
    resume_tier = classify_checkpoint_tier(_rckpt)

    if resume_tier == 'full':
        # newer full-state files keep the RAW weights under 'raw_model_state' (their
        # 'model_state' is the EMA, so the app loads the good weights); older ones
        # kept the raw weights under 'model_state'.
        model.load_state_dict(strip_module_prefix(_rckpt.get('raw_model_state', _rckpt['model_state'])))
        ema_model.module.load_state_dict(strip_module_prefix(_rckpt['ema_state']))
        ema_model.n_averaged.fill_(int(_rckpt.get('ema_n_averaged', 1)))
        opt.load_state_dict(_rckpt['optimizer'])
        scheduler.load_state_dict(_rckpt['scheduler'])
        scaler.load_state_dict(_rckpt['scaler'])
        same_score = (_rckpt.get('train_config') or {}).get('score_version') == SCORE_VERSION
        best_score = float(_rckpt.get('best_score', float('inf'))) if same_score else float('inf')
        epochs_no_improve = int(_rckpt.get('epochs_no_improve', 0)) if same_score else 0
        print(f'[resume: FULL-STATE, exact] epoch {_rckpt["epoch"]} -> continuing at epoch {start_epoch}. '
              f'Optimizer/scheduler/scaler/EMA restored. Best score '
              + (f'{best_score:.4f} carried over.' if same_score else
                 'NOT carried over (written with a different score definition) -- starts fresh.'))
    else:
        model.load_state_dict(strip_module_prefix(_rckpt['model_state']))
        ema_model.module.load_state_dict(model.state_dict())
        remaining = max(1, EPOCHS - start_epoch + 1)
        opt, scheduler = make_optimizer_and_schedule(FINETUNE_LR, remaining)
        best_score, epochs_no_improve = float('inf'), 0
        print(f'[resume: MODEL-ONLY -> FINE-TUNE] {_resume_path} has weights + epoch only. Restored epoch '
              f'{_rckpt["epoch"]} -> continuing at epoch {start_epoch} with a fine-tune schedule: peak LR '
              f'{FINETUNE_LR:g}, 1-epoch warmup, cosine over the remaining {remaining} epoch(s); fresh '
              f'optimizer/EMA; best score starts fresh (the file\'s own val score may use a different '
              f'definition and val set, so it is not comparable).')
else:
    print('No resume checkpoint found (checked /kaggle/working, the dataset root, /kaggle/input for '
          'alpha_net_v3_resume_epoch_*.pt) -- starting fresh training from epoch 1. To continue from a v1/v2 '
          'checkpoint, set RESUME_FROM explicitly -- see its comment above for the exact path.')

if start_epoch > EPOCHS:
    print(f'\n[nothing to do] resume checkpoint is already at epoch {start_epoch - 1}, and EPOCHS={EPOCHS} '
          f'is the TOTAL target epoch count (not additional epochs) -- nothing to train. Raise EPOCHS above '
          f'{start_epoch - 1} and re-run this cell to keep training.')

TRAIN_CONFIG = dict(
    epochs=EPOCHS, batch_per_gpu=BATCH_PER_GPU, global_batch=GLOBAL_BATCH, n_gpus=N_GPUS,
    crop=CROP, samples_per_epoch=SAMPLES_PER_EPOCH, p_crisp=P_CRISP, lr=LR, finetune_lr=FINETUNE_LR,
    weight_decay=WEIGHT_DECAY, num_workers=NUM_WORKERS, patience=PATIENCE, ema_decay=EMA_DECAY,
    seed=SEED, widths=list(WIDTHS), alpha_max=ALPHA_MAX, lambdas=list(LAMBDAS),
    ink_min_alpha=INK_MIN_ALPHA, resume_tier=resume_tier, score_version=SCORE_VERSION,
)

# =============================================================================
# TensorBoard -- live train/val loss and "accuracy" curves.
# "Accuracy" = fraction of the watermark removed on watermark pixels
# (1 - |recovered - clean| / |observed - clean|): the same improvement_wm
# evaluate_v3 reports, computed on train batches too so train/val compare.
# Val loss = the SAME alpha_loss_v3 used for training, run on the val sets.
# =============================================================================
TB_DIR.mkdir(parents=True, exist_ok=True)
tb = SummaryWriter(log_dir=str(TB_DIR))
try:
    get_ipython().run_line_magic('load_ext', 'tensorboard')
    get_ipython().run_line_magic('tensorboard', f'--logdir {TB_DIR} --reload_interval 10')
except Exception as e:
    print(f'[TB] could not open the inline TensorBoard panel ({e}); logs are still written to {TB_DIR}')

def removal_fraction_sums(obs, clean, alpha, pred_alpha, pred_ink):
    """(error_sum, identity_sum) over watermark pixels; accuracy = 1 - err/identity."""
    rec = recover(obs.float(), pred_alpha.float(), pred_ink.float(), ALPHA_MAX)
    wm = (alpha > 0.01).float()
    err = ((rec - clean.float()).abs().mean(dim=1, keepdim=True) * wm).sum()
    idn = ((obs.float() - clean.float()).abs().mean(dim=1, keepdim=True) * wm).sum()
    return float(err), float(idn)

def val_loss(net, loader):
    """Mean alpha_loss_v3 over a val loader -- directly comparable to train loss."""
    net.eval()
    tot, n = 0.0, 0
    with torch.no_grad():
        for obs, clean, alpha, w_alpha, w_rec in loader:
            obs, clean, alpha = obs.to(device), clean.to(device), alpha.to(device)
            w_alpha, w_rec = w_alpha.to(device), w_rec.to(device)
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                pa, pi = net(obs)
            loss, _ = alpha_loss_v3(pa.float(), pi.float(), obs, clean, alpha, w_alpha, w_rec)
            tot += float(loss.item())
            n += 1
    return tot / max(1, n)

best_epoch, best_metrics = None, None

if start_epoch <= EPOCHS:
    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        t_epoch0 = time.time()
        part_sums = {'total': 0.0, 'L_alpha': 0.0, 'L_rec': 0.0, 'L_comp': 0.0, 'L_ink': 0.0}
        n_batches, n_items = 0, 0
        train_err_sum, train_idn_sum = 0.0, 0.0

        for batch_idx, (obs, clean, alpha, w_alpha, w_rec) in enumerate(train_loader):
            obs = obs.to(device, non_blocking=True)
            clean = clean.to(device, non_blocking=True)
            alpha = alpha.to(device, non_blocking=True)
            w_alpha = w_alpha.to(device, non_blocking=True)
            w_rec = w_rec.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            try:
                with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                    pred_alpha, pred_ink = model_run(obs)
                    loss, parts = alpha_loss_v3(pred_alpha, pred_ink, obs, clean, alpha, w_alpha, w_rec)
                if use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            except RuntimeError as e:
                if 'out of memory' in str(e).lower():
                    raise RuntimeError(
                        f'CUDA out of memory during a training step (BATCH_PER_GPU={BATCH_PER_GPU}, '
                        f'CROP={CROP}). Try a smaller BATCH_PER_GPU (e.g. 4) and/or CROP (e.g. 384).'
                    ) from e
                raise

            scheduler.step()
            ema_model.update_parameters(model)
            part_sums['total'] += float(loss.item())
            for k in ('L_alpha', 'L_rec', 'L_comp', 'L_ink'):
                part_sums[k] += parts[k]
            n_batches += 1
            n_items += obs.shape[0]

            with torch.no_grad():
                e_sum, i_sum = removal_fraction_sums(obs, clean, alpha, pred_alpha.detach(), pred_ink.detach())
                train_err_sum += e_sum
                train_idn_sum += i_sum
            global_step = (epoch - 1) * iters_per_epoch + batch_idx
            if global_step % TB_LOG_EVERY == 0:
                tb.add_scalar('batch/train_loss', float(loss.item()), global_step)
                tb.add_scalar('batch/train_L_ink', parts['L_ink'], global_step)
                tb.add_scalar('batch/lr', scheduler.get_last_lr()[0], global_step)

        n_batches = max(1, n_batches)
        train_parts = {k: v / n_batches for k, v in part_sums.items()}
        epoch_seconds = time.time() - t_epoch0
        imgs_per_sec = n_items / max(epoch_seconds, 1e-6)

        crisp_metrics = evaluate_v3(ema_model.module, crisp_val_loader, device)
        kaggle_metrics = evaluate_v3(ema_model.module, kaggle_val_loader, device)
        val_score = crisp_metrics['val_score']
        lr_now = scheduler.get_last_lr()[0]

        train_acc = 1.0 - train_err_sum / max(train_idn_sum, 1e-6)
        crisp_val_loss = val_loss(ema_model.module, crisp_val_loader)
        kaggle_val_loss = val_loss(ema_model.module, kaggle_val_loader)
        tb.add_scalars('loss', {'train': train_parts['total'], 'val_crisp': crisp_val_loss,
                                'val_kaggle': kaggle_val_loss}, epoch)
        tb.add_scalars('accuracy_watermark_removed', {'train': train_acc,
                                                       'val_crisp': crisp_metrics['improvement_wm'],
                                                       'val_kaggle': kaggle_metrics['improvement_wm']}, epoch)
        tb.add_scalars('accuracy_coloured_watermark_removed', {'val_crisp': crisp_metrics['improvement_wm_colored'],
                                                                'val_kaggle': kaggle_metrics['improvement_wm_colored']}, epoch)
        tb.add_scalars('ink_saturation_crisp', {'predicted': crisp_metrics['ink_sat_pred_wm'],
                                                'true': crisp_metrics['ink_sat_true_wm']}, epoch)
        tb.add_scalars('loss_parts_train', {k: train_parts[k] for k in ('L_alpha', 'L_rec', 'L_comp', 'L_ink')}, epoch)
        tb.add_scalar('val/crisp_val_score', crisp_metrics['val_score'], epoch)
        tb.add_scalar('val/kaggle_val_score', kaggle_metrics['val_score'], epoch)
        tb.add_scalar('val/crisp_alpha_mae_wm', crisp_metrics['alpha_mae_wm'], epoch)
        tb.add_scalar('epoch/lr', lr_now, epoch)
        tb.add_scalar('epoch/img_per_sec', imgs_per_sec, epoch)
        tb.flush()

        print(
            f'epoch {epoch}/{EPOCHS} train_loss={train_parts["total"]:.4f} '
            f'(alpha={train_parts["L_alpha"]:.4f} rec={train_parts["L_rec"]:.4f} comp={train_parts["L_comp"]:.4f} '
            f'ink={train_parts["L_ink"]:.4f}) val_loss={crisp_val_loss:.4f} train_acc={train_acc:.3f} '
            f'crisp: val_score={crisp_metrics["val_score"]:.4f} '
            f'alpha_mae_wm={crisp_metrics["alpha_mae_wm"]:.4f} ink_sat_pred_wm={crisp_metrics["ink_sat_pred_wm"]:.3f} '
            f'(true={crisp_metrics["ink_sat_true_wm"]:.3f}) improvement_wm_colored={crisp_metrics["improvement_wm_colored"]:.3f} '
            f'| kaggle: val_score={kaggle_metrics["val_score"]:.4f} | lr={lr_now:.2e} time={epoch_seconds:.1f}s img/s={imgs_per_sec:.1f}'
        )

        is_new_file = not V3_METRICS_PATH.exists()
        fieldnames = (['epoch', 'train_loss', 'train_L_alpha', 'train_L_rec', 'train_L_comp', 'train_L_ink',
                        'lr', 'seconds', 'img_s']
                      + [f'crisp_{k}' for k in crisp_metrics] + [f'kaggle_{k}' for k in kaggle_metrics])
        with open(V3_METRICS_PATH, 'a', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            if is_new_file:
                w.writeheader()
            row = {'epoch': epoch, 'train_loss': train_parts['total'], 'train_L_alpha': train_parts['L_alpha'],
                   'train_L_rec': train_parts['L_rec'], 'train_L_comp': train_parts['L_comp'],
                   'train_L_ink': train_parts['L_ink'], 'lr': lr_now, 'seconds': epoch_seconds, 'img_s': imgs_per_sec}
            row.update({f'crisp_{k}': v for k, v in crisp_metrics.items()})
            row.update({f'kaggle_{k}': v for k, v in kaggle_metrics.items()})
            w.writerow(row)

        combined_val_metrics = {'val_score': val_score, 'crisp': crisp_metrics, 'kaggle': kaggle_metrics}
        if val_score < best_score:
            best_score = val_score
            epochs_no_improve = 0
            # UNWRAPPED EMA module -- slim, model-only, what the app loads.
            save_checkpoint(V3_BEST_PATH, ema_model.module, epoch, combined_val_metrics, TRAIN_CONFIG)
            best_epoch, best_metrics = epoch, combined_val_metrics
            print(f'   -> new best (val_score {val_score:.4f}), saved {V3_BEST_PATH}')
        else:
            epochs_no_improve += 1

        # FULL-STATE, for exact resume. This is the LATEST epoch, not the best one.
        save_full_state_checkpoint(V3_LAST_PATH, model, ema_model, opt, scheduler, scaler, epoch,
                                    combined_val_metrics, TRAIN_CONFIG, best_score, epochs_no_improve)

        if epoch % SNAPSHOT_EVERY == 0:
            if V3_BEST_PATH.exists():
                best_snap = pathlib.Path(f'/kaggle/working/alpha_net_v3_best_upto_epoch_{epoch}.pt')
                shutil.copy2(V3_BEST_PATH, best_snap)
                print(f'\n[Checkpoint] best model so far (epoch {best_epoch}), slim, for the app: {best_snap}')
                display(FileLink(best_snap.name))
            resume_snap = pathlib.Path(f'/kaggle/working/alpha_net_v3_resume_epoch_{epoch}.pt')
            shutil.copy2(V3_LAST_PATH, resume_snap)
            print(f'[Checkpoint] full-state resume file (latest epoch {epoch}) -- re-upload to continue exactly: {resume_snap}')
            display(FileLink(resume_snap.name))

        if epochs_no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch} ({epochs_no_improve} epochs without crisp-val improvement)')
            break

    print('\n=== v3 training summary ===')
    print('best epoch:', best_epoch)
    print('best val metrics:', best_metrics)

tb.close()


### 12.3 v3 examples (val + real pages + coloured-mark zoom)

Loads the trained checkpoint through the same `locate_alpha_checkpoint` used to resume in 12.2 (preferring this session's own `V3_RUN_DIR/best.pt`, falling back to whatever the locator finds), so this cell also works in a fresh session where `/kaggle/working` is empty but the dataset has a v3 snapshot. Shows 4 colour-augmented crisp validation crops and runs tiled full-page inference (`predict_page`, reused from an earlier section if it already ran this session, else defined here) on 6 real pages from `SCRAPED_DIR` -- same as 11.4. Then, specifically to make the ink-head fix visible rather than just aggregate: composites `ariatender_wide`'s REAL calibrated ink (not colour-augmented -- the stamp exactly as shipped) onto a held-out background and zooms into the pink shield glyph (the left ~22% of the stamp -- the "AriaTender.neT" wordmark, grey, fills the rest), the part v2's oracle ablation showed was only 70.4% removed (vs. 94.9% for the grey lettering) because of the collapsed ink head.

In [ ]:
import pathlib
import time
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

for _name in ('AlphaUNet', 'recover', 'V3_RUN_DIR', 'CROP', 'ColorAugmentedCrispCompositeDataset',
              'SCRAPED_DIR', 'locate_alpha_checkpoint', 'strip_module_prefix', 'calibrated_stamps',
              'VAL_BACKGROUNDS', 'scale_stamp', 'paste_over_crop'):
    assert _name in globals(), f'{_name!r} is not defined -- run 10.1-10.6, 11.1-11.2, and 12.1-12.2 first.'

# --- load the trained v3 checkpoint --------------------------------------------
# Prefer this session's own V3_RUN_DIR/best.pt; fall back to the locator so
# examples still work in a FRESH session where /kaggle/working is empty but a
# v3 snapshot was uploaded into the dataset (12.2's periodic FileLink output).
V3_BEST_PATH = V3_RUN_DIR / 'best.pt'
if not V3_BEST_PATH.exists():
    _located = locate_alpha_checkpoint(['alpha_net_v3_best_final.pt', 'alpha_net_v3_best_epoch_*.pt'])
    assert _located is not None, (
        f'No trained v3 weights at {V3_BEST_PATH}, and none found via locate_alpha_checkpoint (checked '
        f'/kaggle/working, the dataset root, /kaggle/input) -- run 12.2 first, or upload a v3 snapshot '
        f'into the dataset the way alpha_net_v2_best_epoch_60.pt was.'
    )
    V3_BEST_PATH = _located
    print('V3_RUN_DIR/best.pt not found (fresh session) -- using located checkpoint instead:', V3_BEST_PATH)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(V3_BEST_PATH, map_location=device, weights_only=True)
v3_model = AlphaUNet(widths=tuple(ckpt['widths']), alpha_max=float(ckpt['alpha_max'])).to(device)
v3_model.load_state_dict(strip_module_prefix(ckpt['model_state']))
v3_model.eval()
print('v3 checkpoint:', V3_BEST_PATH)
print('v3 epoch:', ckpt['epoch'])
print('v3 val metrics:', ckpt['val_metrics'])

# --- crisp validation crops (colour-augmented) ---------------------------------
crisp_val_ds_show = ColorAugmentedCrispCompositeDataset(CROP, samples_per_epoch=None, deterministic=True, seed=SEED if 'SEED' in globals() else 0)
N_SHOW = 4
fig, axes = plt.subplots(N_SHOW, 4, figsize=(16, 4 * N_SHOW))
for row in range(N_SHOW):
    idx = row % len(crisp_val_ds_show)
    obs, clean, alpha_gt, _w_alpha, _w_rec = crisp_val_ds_show[idx]
    with torch.no_grad():
        pred_alpha, pred_ink = v3_model(obs.unsqueeze(0).to(device))
        rec = recover(obs.unsqueeze(0).to(device), pred_alpha, pred_ink, float(ckpt['alpha_max']))

    obs_img = obs.permute(1, 2, 0).numpy()
    alpha_img = pred_alpha[0, 0].cpu().numpy()
    rec_img = rec[0].permute(1, 2, 0).cpu().numpy()
    clean_img = clean.permute(1, 2, 0).numpy()

    axes[row, 0].imshow(np.clip(obs_img, 0, 1)); axes[row, 0].set_title('observed')
    axes[row, 1].imshow(alpha_img, cmap='inferno', vmin=0, vmax=float(ckpt['alpha_max'])); axes[row, 1].set_title('predicted alpha')
    axes[row, 2].imshow(np.clip(rec_img, 0, 1)); axes[row, 2].set_title('recovered')
    axes[row, 3].imshow(np.clip(clean_img, 0, 1)); axes[row, 3].set_title('clean')
    for c in range(4):
        axes[row, c].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/alpha_net_v3_crisp_val_examples.png', dpi=120)
plt.show()

# --- real pages: tiled full-page inference (reused from 10.9/11.4 if present) --
if 'predict_page' not in globals():
    def feather_weight(tile, overlap):
        overlap = max(1, min(overlap, tile // 2))
        ramp = np.linspace(1.0 / (overlap + 1), 1.0, overlap, dtype=np.float32)
        w = np.ones(tile, dtype=np.float32)
        w[:overlap], w[-overlap:] = ramp, ramp[::-1]
        return np.outer(w, w)

    def predict_page(model, img_rgb_uint8, device, tile=512, overlap=64):
        alpha_max = model.alpha_max
        h, w = img_rgb_uint8.shape[:2]
        pad_h = max(tile, int(np.ceil(h / 16.0) * 16))
        pad_w = max(tile, int(np.ceil(w / 16.0) * 16))
        img_f = img_rgb_uint8.astype(np.float32) / 255.0
        img_padded = np.pad(img_f, ((0, pad_h - h), (0, pad_w - w), (0, 0)), mode='reflect')

        alpha_acc = np.zeros((pad_h, pad_w), dtype=np.float32)
        ink_acc = np.zeros((pad_h, pad_w, 3), dtype=np.float32)
        weight_acc = np.zeros((pad_h, pad_w), dtype=np.float32)

        stride = max(1, tile - overlap)
        ys = list(range(0, max(1, pad_h - tile + 1), stride))
        if ys[-1] != pad_h - tile:
            ys.append(pad_h - tile)
        xs = list(range(0, max(1, pad_w - tile + 1), stride))
        if xs[-1] != pad_w - tile:
            xs.append(pad_w - tile)
        ramp2d = feather_weight(tile, overlap)

        with torch.no_grad():
            for y in ys:
                for x in xs:
                    crop = img_padded[y:y + tile, x:x + tile, :]
                    t = torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0).to(device)
                    a, ink = model(t)
                    a_np = a[0, 0].float().cpu().numpy()
                    ink_np = ink[0].float().permute(1, 2, 0).cpu().numpy()
                    alpha_acc[y:y + tile, x:x + tile] += a_np * ramp2d
                    ink_acc[y:y + tile, x:x + tile, :] += ink_np * ramp2d[..., None]
                    weight_acc[y:y + tile, x:x + tile] += ramp2d

        weight_safe = np.clip(weight_acc, 1e-6, None)
        alpha_full = (alpha_acc / weight_safe)[:h, :w]
        ink_full = (ink_acc / weight_safe[..., None])[:h, :w, :]

        obs_t = torch.from_numpy(img_f).permute(2, 0, 1).unsqueeze(0)
        alpha_t = torch.from_numpy(alpha_full).unsqueeze(0).unsqueeze(0)
        ink_t = torch.from_numpy(ink_full).permute(2, 0, 1).unsqueeze(0)
        recovered_t = recover(obs_t, alpha_t, ink_t, alpha_max)
        recovered = np.clip(np.round(recovered_t[0].permute(1, 2, 0).numpy() * 255.0), 0, 255).astype(np.uint8)
        return alpha_full.astype(np.float32), ink_full.astype(np.float32), recovered
    print('predict_page was not defined this session -- defined it locally.')
else:
    print('Reusing predict_page from an earlier section.')

assert SCRAPED_DIR is not None, 'Could not find images_scraped -- see 10.1.'
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
real_images = sorted(p for p in SCRAPED_DIR.rglob('*') if p.suffix.lower() in EXTS and p.is_file())
assert real_images, f'No images found under {SCRAPED_DIR}'

V3_REAL_OUT_DIR = pathlib.Path('/kaggle/working/alpha_net_v3_real')
V3_REAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

v3_panel_paths = []
for p in real_images[:6]:
    img = np.array(Image.open(p).convert('RGB'))
    t0 = time.time()
    alpha, ink, recovered = predict_page(v3_model, img, device)
    dt = time.time() - t0

    alpha_vis = (np.clip(alpha / max(float(ckpt['alpha_max']), 1e-6), 0, 1) * 255).astype(np.uint8)
    alpha_vis = np.stack([alpha_vis] * 3, axis=-1)
    panel = np.concatenate([img, alpha_vis, recovered], axis=1)
    panel_path = V3_REAL_OUT_DIR / f'{p.stem}_alpha_panel_v3.png'
    Image.fromarray(panel).save(panel_path)
    v3_panel_paths.append(panel_path)

    frac_marked = float((alpha > 0.05).mean()) * 100
    print(f'{p.name}: alpha_max={alpha.max():.3f} marked={frac_marked:.2f}% time={dt:.2f}s')

for path in v3_panel_paths[:3]:
    display(IPyImage(filename=str(path), width=1200))

# =============================================================================
# Coloured-mark isolation: composite ariatender_wide's REAL calibrated ink
# (not colour-augmented -- this is the stamp exactly as shipped) onto a held-
# out background and zoom into the pink SHIELD glyph specifically, which sits
# in roughly the left ~22% of the stamp's width (the "AriaTender.neT"
# wordmark, grey, fills the rest) -- this is the part v2's oracle ablation
# showed was only 70.4% removed (vs. 94.9% for the grey lettering) because of
# the collapsed ink head. Side-by-side with predicted ink, this panel shows
# at a glance whether v3 actually reads the pink colour off the observation.
# =============================================================================
_wide_stamp = next((s for s in calibrated_stamps if s['name'] == 'ariatender_wide'), None)
if _wide_stamp is None:
    print('[skip] ariatender_wide not found in calibrated_stamps -- upload stamps/ (see 11.1 markdown) '
          'to see the coloured-mark zoom panel.')
else:
    import cv2
    _margin = 40
    _bg_path = VAL_BACKGROUNDS[0]
    _bg_full = cv2.cvtColor(cv2.imread(str(_bg_path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    _bh, _bw = _bg_full.shape[:2]

    _target_w = int(min(_bw - 2 * _margin, 1100))
    _scale = _target_w / _wide_stamp['alpha'].shape[1]
    _s_ink, _s_alpha = scale_stamp(_wide_stamp['ink'], _wide_stamp['alpha'], _scale)
    _th, _tw = _s_alpha.shape[:2]

    _y0 = max(0, (_bh - _th) // 2 - _margin)
    _x0 = max(0, (_bw - _tw) // 2 - _margin)
    _ch, _cw = _th + 2 * _margin, _tw + 2 * _margin
    _clean_shield = _bg_full[_y0:_y0 + _ch, _x0:_x0 + _cw, :].copy()
    if _clean_shield.shape[:2] != (_ch, _cw):
        _pad_h, _pad_w = _ch - _clean_shield.shape[0], _cw - _clean_shield.shape[1]
        _clean_shield = np.pad(_clean_shield, ((0, max(0, _pad_h)), (0, max(0, _pad_w)), (0, 0)), mode='reflect')
    _obs_shield = _clean_shield.copy()
    _alpha_shield = np.zeros((_ch, _cw), dtype=np.float32)
    paste_over_crop(_obs_shield, _alpha_shield, _s_ink, _s_alpha, _margin, _margin)

    _obs_t = torch.from_numpy(np.ascontiguousarray(_obs_shield.transpose(2, 0, 1)).astype(np.float32)).unsqueeze(0).to(device)
    with torch.no_grad():
        _pred_alpha, _pred_ink = v3_model(_obs_t)
        _rec = recover(_obs_t, _pred_alpha, _pred_ink, float(ckpt['alpha_max']))
    _pred_alpha_np = _pred_alpha[0, 0].cpu().numpy()
    _pred_ink_np = np.clip(_pred_ink[0].permute(1, 2, 0).cpu().numpy(), 0, 1)
    _rec_np = np.clip(_rec[0].permute(1, 2, 0).cpu().numpy(), 0, 1)

    # The shield glyph occupies roughly the left 22% of ariatender_wide's
    # width (see assets/stamps/ariatender_wide.png); zoom to the left ~30%
    # of the placed tile (plus a small margin) so it's clearly visible.
    _shield_x1 = _margin + int(0.30 * _tw)
    _zoom = np.s_[max(0, _margin - 15):min(_ch, _th + _margin + 15), max(0, _margin - 15):_shield_x1]

    fig, axes = plt.subplots(1, 5, figsize=(20, 4.4))
    axes[0].imshow(np.clip(_obs_shield[_zoom], 0, 1)); axes[0].set_title('observed (pink shield, zoomed)')
    axes[1].imshow(_pred_alpha_np[_zoom], cmap='inferno', vmin=0, vmax=float(ckpt['alpha_max'])); axes[1].set_title('predicted alpha')
    axes[2].imshow(_pred_ink_np[_zoom]); axes[2].set_title('predicted ink')
    axes[3].imshow(_rec_np[_zoom]); axes[3].set_title('recovered')
    axes[4].imshow(np.clip(_clean_shield[_zoom], 0, 1)); axes[4].set_title('clean')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('/kaggle/working/alpha_net_v3_shield_zoom.png', dpi=130)
    plt.show()
    print('Saved coloured-mark (pink shield) zoom panel to /kaggle/working/alpha_net_v3_shield_zoom.png')

### 12.4 Download the v3 model

To use this model in the app: place it at `weights/alpha_net_best_final.pt` (keeping a copy of the previous one, e.g. `weights/alpha_net_v2_best_final.pt` if you haven't already), then run the repo benchmark, `python scripts/alpha_net/benchmark.py`, to compare. This file is a slim, model-only checkpoint (`save_checkpoint`/10.5's format, same as every prior section's final download) -- unaffected by 12.2's larger full-state periodic snapshots, which exist only to make cross-session resume exact.

In [ ]:
import pathlib
import shutil
import torch
from IPython.display import FileLink, display

assert 'V3_RUN_DIR' in globals(), 'V3_RUN_DIR is not defined -- run 12.2 first.'

v3_best_src = V3_RUN_DIR / 'best.pt'
v3_last_src = V3_RUN_DIR / 'last.pt'
v3_final_path = pathlib.Path('/kaggle/working/alpha_net_v3_best_final.pt')

if v3_best_src.exists():
    # best.pt is slim and holds the EMA weights of the best-scoring epoch -- what the app should load.
    shutil.copy2(v3_best_src, v3_final_path)
    ckpt = torch.load(v3_final_path, map_location='cpu', weights_only=True)
    print('BEST epoch:', ckpt['epoch'])
    print('crisp val metrics:', ckpt['val_metrics']['crisp'])
    print('kaggle val metrics:', ckpt['val_metrics']['kaggle'])
    print(f'size: {v3_final_path.stat().st_size / 1e6:.2f} MB')
    display(FileLink(v3_final_path.name))
elif v3_last_src.exists():
    # No epoch was ever selected as best (e.g. an interrupted first epoch). Export the LAST epoch's
    # EMA weights as a slim file -- clearly labelled, never the raw weights.
    ck = torch.load(v3_last_src, map_location='cpu', weights_only=True)
    ema_sd = ck['ema_state'] if 'ema_state' in ck else ck['model_state']
    slim = {k: ck[k] for k in ('format_version', 'arch', 'widths', 'alpha_max', 'epoch', 'val_metrics', 'train_config')}
    slim['model_state'] = strip_module_prefix(ema_sd)
    last_path = pathlib.Path('/kaggle/working/alpha_net_v3_LAST_epoch_ema.pt')
    torch.save(slim, last_path)
    print(f'WARNING: no best.pt -- exported the LAST epoch ({ck["epoch"]}) EMA weights instead, as {last_path.name}. '
          'This was not selected by validation.')
    display(FileLink(last_path.name))
else:
    print(f'No best.pt or last.pt in {V3_RUN_DIR} -- run the training cell (12.2) first.')
